# NBA Injury / Return-to-Play Multi-Season Extractor — FINAL 2021–22 to 2024–25

Clean reproducible primary-cohort notebook incorporating the validated v5.2 core, historical ESPN source fixes, Stat Surge historical injury CSV source, schedule-team identity correction, canonical team aliases, and the corrected historical age merge and authoritative two-player age fallback.

**Primary validated cohort:** 2021–22, 2022–23, 2023–24, 2024–25. Experimental 2019–21 / 2025–26 extension work is intentionally excluded from this production notebook so it cannot contaminate the primary analysis.


## 1. Install dependencies

Run this cell in a fresh Colab/Jupyter environment. If you already have the packages installed, it is safe to skip it.


In [ ]:
%pip install -q pandas numpy requests pyarrow openpyxl xlsxwriter nba_api tqdm pdfplumber


## 2. Multi-season configuration

The primary historical window is now **2021–22 through 2024–25**. Official NBA injury-report archives are treated as research-grade historical inputs beginning in 2021–22. The default validation run is 2023–24; after it passes QC, switch `RUN_MODE` to `"full"`.


In [ ]:
from pathlib import Path

# ------------------------------ RUN SELECTION ------------------------------
RUN_MODE = "validation"   # safe default; switch to "full" for a from-scratch four-season rebuild
VALIDATION_SEASONS = ["2023-24"]
FULL_SEASONS = ["2021-22", "2022-23", "2023-24", "2024-25"]
SEASONS = VALIDATION_SEASONS if RUN_MODE == "validation" else FULL_SEASONS

# Root folder. Each season gets its own isolated subfolder.
MULTI_ROOT = Path("nba_injury_rtp_multiseason")
MULTI_ROOT.mkdir(parents=True, exist_ok=True)

# Resume / output behavior.
SKIP_COMPLETED_PASS_SEASONS = True
FORCE_RERUN_SEASONS = set()   # e.g. {"2022-23"}
WRITE_SEASON_CSV = True
WRITE_SEASON_EXCEL = False    # Parquet is much faster/smaller; combined QC Excel is written later.
WRITE_COMBINED_EXCEL = True

# Core cohort settings inherited from the validated v5.2 pilot.
MIN_GAMES_MISSED = 2
MEANINGFUL_RTP_MINUTES = 5.0
PRIMARY_BASELINE_MINUTES = 10.0
STRICT_BASELINE_MINUTES = 15.0
MIN_PRE_GAMES_PRIMARY = 5
MIN_HEALTHY_PLAYED_GAMES_FOR_SEPARATED_EVENT = 3

# Historical source policy (v2).
# 2021-22 through 2023-24 use ESPN scoreboards + ESPN player_box data because
# the per-season NBA playerlog parquet files are incomplete for these years.
HISTORICAL_ESPN_SEASONS = {"2021-22", "2022-23", "2023-24"}
OFFICIAL_IR_SUPPORTED_SEASONS = {"2021-22", "2022-23", "2023-24", "2024-25"}
ESPN_SCOREBOARD_WORKERS = 10
ESPN_SCOREBOARD_START_INTERVAL = 0.06
HISTORICAL_MIN_OBSERVED_GAME_COVERAGE = 0.95
INCLUDE_REVIEW_IN_COMBINED = False

# Live CDN is OFF by default for multi-season extraction because the current
# Colab/cloud runtime may receive 403 Access Denied. The core RTP study does not
# require it; exact rest-hours / live-DNP enrichment can be added later.
FETCH_LIVE = False

# Deep stats archive stays OFF until the core multi-season cohort passes QC.
DEEP_MODE = "off"
DEEP_SCOPE = "episode_windows"

# ----------------------- OFFICIAL INJURY REPORT ARCHIVE -----------------------
# 2024-25 uses the already-parsed public season_rows.csv from the validated pilot.
# Older seasons are fetched directly from the NBA official PDF archive, cache-first.
USE_PREPARSED_2425 = True
IR_ARCHIVE_BASE = "https://ak-static.cms.nba.com/referee/injury/Injury-Report_{date}_{slot}.pdf"
IR_SLOTS = ["01PM", "03PM", "05PM", "08PM", "11PM"]
IR_POLITE_DELAY = 1.5          # sequential; raise toward 2.0 if the CDN starts blocking
IR_TIMEOUT = (6, 25)
IR_RECHECK_MISSING = False     # True only if you intentionally want to retry prior misses
IR_PROGRESS_EVERY = 100

# Known-good archive probe used only to detect a CDN block before a long scrape.
IR_PROBE_DATE = "2023-03-29"
IR_PROBE_SLOT = "05PM"

# End dates used only to cover postseason report dates after the regular-season
# team log ends. The regular-season game dates themselves are derived from data.
SEASON_ARCHIVE_END = {
    "2019-20": "2020-10-12",  # Orlando bubble / Finals
    "2020-21": "2021-07-21",
    "2021-22": "2022-06-17",
    "2022-23": "2023-06-13",
    "2023-24": "2024-06-18",
    "2024-25": "2025-06-23",
}

# --------------------------- EXISTING NETWORK KNOBS ---------------------------
# Kept because the validated engine references them even when FETCH_LIVE=False.
LIVE_WORKERS = 4
LIVE_CONNECT_TIMEOUT = 8
LIVE_READ_TIMEOUT = 30
LIVE_RETRIES = 3
LIVE_MIN_START_INTERVAL = 0.20
LIVE_BATCH_SIZE = 60
LIVE_ABORT_FAILURE_RATE = 0.40
DEEP_WORKERS = 2
NBA_STATS_MIN_START_INTERVAL = 0.55
NBA_STATS_TIMEOUT = 45
NBA_STATS_RETRIES = 2

print("Run mode:", RUN_MODE)
print("Target seasons:", SEASONS)


### Speed / resume notes

- Historical official injury-report PDFs use the fast v1.2 cache-first downloader.
- Historical player-game data use one shared ESPN `player_box.parquet` file plus cached daily ESPN scoreboards.
- Scoreboards are fetched concurrently and cached; reruns reuse them.
- 2019–20 and 2020–21 are intentionally excluded from the primary batch because the official historical injury-report archive is not treated as consistently available before 2021–22.
- Completed `PASS` seasons are skipped on rerun unless explicitly forced.


## 3. Validated v5.2 core extraction engine

The cells below are the same core player-game / schedule / RTP engine validated on 2024–25. The multi-season layer later in the notebook replaces only the season-specific injury-report loader and output orchestration.


In [ ]:
#!/usr/bin/env python3
"""
NBA injury / return-to-play feasibility extractor
================================================

Purpose
-------
Build a research-ready player-game table and episode-level return-to-play (RTP)
features for studying minutes ramp-up after injury, subsequent injury-related
absence, performance recovery, back-to-backs, short rest, and travel.

The code deliberately separates four source layers:
  1) player game logs (minutes + performance),
  2) team game logs (schedule/home-away),
  3) official injury-report designations (medical reason/body part), and
  4) NBA live box scores (actual participation reason + exact UTC game time/arena).

The first pilot is designed around 2024-25 because a public research repository
already contains a parsed full-season official NBA injury-report table. For a
publication, re-fetch/validate the official NBA PDFs and audit source coverage.

This file mirrors the caching/fallback philosophy of the user's prior NBA
extraction notebook: download once, normalize once, export reusable raw tables.
"""

from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
from typing import Iterable, Optional
import argparse
import json
import math
import re
import time
import unicodedata
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter

ROOT = Path("nba_injury_rtp_exports")
CACHE = ROOT / "cache"
RAW = ROOT / "raw"
for _d in (ROOT, CACHE, RAW):
    _d.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "Referer": "https://www.nba.com/",
}

# Public, reproducible mirrors used for the pilot.
LLIMLLIB_RAW = "https://raw.githubusercontent.com/llimllib/nba_data/main/data"
PETE_2425_IR = (
    "https://raw.githubusercontent.com/Professor-Pete/"
    "2024-2025-injury_analysis/main/data/season_rows.csv"
)
NBA_LIVE_BOXSCORE = (
    "https://cdn.nba.com/static/json/liveData/boxscore/boxscore_{game_id}.json"
)

TEAM_NAME_TO_ABBR = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Clippers": "LAC", "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM", "Miami Heat": "MIA", "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN", "New Orleans Pelicans": "NOP", "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC", "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHX", "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS", "Toronto Raptors": "TOR", "Utah Jazz": "UTA",
    "Washington Wizards": "WAS",
}

# Approximate city-center coordinates; good enough for a reproducible travel-distance
# proxy. Live boxscore arena city/state are retained separately so neutral-site games
# can be audited/re-coded later.
CITY_COORDS = {
    "atlanta": (33.7490, -84.3880), "boston": (42.3601, -71.0589),
    "brooklyn": (40.6782, -73.9442), "charlotte": (35.2271, -80.8431),
    "chicago": (41.8781, -87.6298), "cleveland": (41.4993, -81.6944),
    "dallas": (32.7767, -96.7970), "denver": (39.7392, -104.9903),
    "detroit": (42.3314, -83.0458), "san francisco": (37.7749, -122.4194),
    "houston": (29.7604, -95.3698), "indianapolis": (39.7684, -86.1581),
    "los angeles": (34.0522, -118.2437), "memphis": (35.1495, -90.0490),
    "miami": (25.7617, -80.1918), "milwaukee": (43.0389, -87.9065),
    "minneapolis": (44.9778, -93.2650), "new orleans": (29.9511, -90.0715),
    "new york": (40.7128, -74.0060), "oklahoma city": (35.4676, -97.5164),
    "orlando": (28.5383, -81.3792), "philadelphia": (39.9526, -75.1652),
    "phoenix": (33.4484, -112.0740), "portland": (45.5152, -122.6784),
    "sacramento": (38.5816, -121.4944), "san antonio": (29.4241, -98.4936),
    "toronto": (43.6532, -79.3832), "salt lake city": (40.7608, -111.8910),
    "washington": (38.9072, -77.0369), "mexico city": (19.4326, -99.1332),
    "paris": (48.8566, 2.3522), "abu dhabi": (24.4539, 54.3773),
    "las vegas": (36.1699, -115.1398),
}


# --------------------------- fast HTTP session pool --------------------------

_HTTP_LOCAL = threading.local()

def _http_session():
    """One requests.Session per worker thread with a reusable connection pool."""
    sess = getattr(_HTTP_LOCAL, "session", None)
    if sess is None:
        sess = requests.Session()
        sess.headers.update(HEADERS)
        pool = max(32, int(globals().get("LIVE_WORKERS", 16)) * 2)
        adapter = HTTPAdapter(pool_connections=pool, pool_maxsize=pool, max_retries=0)
        sess.mount("https://", adapter)
        sess.mount("http://", adapter)
        _HTTP_LOCAL.session = sess
    return sess

def _atomic_write_json(path: Path, obj: dict):
    """Prevent an interrupted write from leaving a corrupt cache file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".{threading.get_ident()}.tmp")
    tmp.write_text(json.dumps(obj))
    tmp.replace(path)

# ------------------------------- IO helpers ---------------------------------

def _download(url: str, path: Path, retries: int = 4, delay: float = 1.5) -> Path:
    if path.exists() and path.stat().st_size > 0:
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    last = None
    for attempt in range(retries):
        try:
            r = _http_session().get(url, timeout=(10, 90))
            r.raise_for_status()
            path.write_bytes(r.content)
            return path
        except Exception as e:
            last = e
            time.sleep(delay * (2 ** attempt))
    raise RuntimeError(f"download failed: {url}: {last}")


def _season_end_year(season: str) -> int:
    start = int(str(season)[:4])
    return start + 1


def get_public_playerlog(season: str) -> pd.DataFrame:
    y = _season_end_year(season)
    p = RAW / f"playerlog_{y}.parquet"
    _download(f"{LLIMLLIB_RAW}/playerlog_{y}.parquet", p)
    return pd.read_parquet(p)


def get_public_teamlog(season: str) -> pd.DataFrame:
    y = _season_end_year(season)
    p = RAW / f"gamelog_{y}.parquet"
    _download(f"{LLIMLLIB_RAW}/gamelog_{y}.parquet", p)
    return pd.read_parquet(p)


def get_public_player_season_stats(season: str) -> pd.DataFrame:
    y = _season_end_year(season)
    p = RAW / f"players_{y}.parquet"
    _download(f"{LLIMLLIB_RAW}/players_{y}.parquet", p)
    return pd.read_parquet(p)


def get_2425_official_injury_designations() -> pd.DataFrame:
    p = RAW / "official_injury_designations_2024_25.csv"
    _download(PETE_2425_IR, p)
    return pd.read_csv(p, low_memory=False)


# -------------------------- normalization helpers ---------------------------

def _first_col(df: pd.DataFrame, names: Iterable[str]) -> Optional[str]:
    lookup = {str(c).lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lookup:
            return lookup[n.lower()]
    return None


def parse_minutes(x) -> float:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return 0.0
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    if not s:
        return 0.0
    # NBA ISO-8601 duration, e.g. PT26M29.00S
    m = re.fullmatch(r"PT(?:(\d+)M)?([\d.]+)S", s)
    if m:
        return float(m.group(1) or 0) + float(m.group(2)) / 60.0
    # Normal MM:SS
    m = re.fullmatch(r"(\d+):(\d+(?:\.\d+)?)", s)
    if m:
        return float(m.group(1)) + float(m.group(2)) / 60.0
    try:
        return float(s)
    except Exception:
        return 0.0


def norm_name(s: str) -> str:
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    # Convert "Williams III, Robert" -> "Robert Williams III" before removing suffix.
    if "," in s:
        left, right = s.split(",", 1)
        s = f"{right.strip()} {left.strip()}"
    s = re.sub(r"\b(jr|sr|ii|iii|iv|v)\.?\b", " ", s, flags=re.I)
    s = re.sub(r"[^A-Za-z0-9]+", " ", s).lower()
    return re.sub(r"\s+", " ", s).strip()


def reason_family(*parts) -> str:
    text = " ".join(str(x or "") for x in parts).lower()
    if any(k in text for k in ["injury", "injury/illness", "dnd_injury", "dnp_injury", "inactive_injury", "nwt_injury"]):
        # Keep illness separate when it is clearly illness without a musculoskeletal reason.
        if "illness" in text and not any(k in text for k in [
            "ankle", "knee", "hamstring", "calf", "foot", "achilles", "groin", "adductor",
            "quad", "hip", "back", "shoulder", "wrist", "hand", "finger", "toe", "neck",
            "concussion", "elbow", "thigh", "leg", "rib", "abdomen", "core", "surgery",
        ]):
            return "illness"
        return "injury"
    if "illness" in text or "health_and_safety" in text or "health and safety" in text:
        return "illness"
    if "rest" in text or "load management" in text:
        return "rest"
    if "susp" in text:
        return "suspension"
    if "personal" in text:
        return "personal"
    if "g league" in text or "gleague" in text or "two-way" in text or "assignment" in text:
        return "g_league"
    if "trade" in text or "not with team" in text:
        return "roster_transaction"
    if "coach" in text:
        return "coach"
    return "unknown"


BODY_PATTERNS = [
    ("achilles", r"achilles"), ("hamstring", r"hamstring"), ("calf", r"\bcalf\b"),
    ("adductor_groin", r"adductor|groin"), ("quadriceps", r"quad(?:riceps)?|quadricep"),
    ("ankle", r"ankle"), ("knee", r"knee|patella|menisc|acl|mcl|pcl|lcl"),
    ("foot", r"foot|metatars|plantar"), ("toe", r"\btoe\b"), ("hip", r"\bhip\b"),
    ("back", r"back|lumbar|thoracic"), ("core_abdomen", r"abdomen|abdominal|core"),
    ("shoulder", r"shoulder|rotator|labrum"), ("elbow", r"elbow"),
    ("wrist", r"wrist|radius"), ("hand_finger", r"hand|finger|thumb"),
    ("neck", r"neck|cervical"), ("head_concussion", r"concussion|head"),
    ("rib_chest", r"rib|chest|sternum"), ("leg_unspecified", r"\bleg\b|tibia|fibula"),
]

DX_PATTERNS = [
    ("tear", r"tear|torn|rupture"), ("strain", r"strain"), ("sprain", r"sprain"),
    ("fracture", r"fracture|broken"), ("soreness", r"soreness|sore"),
    ("inflammation", r"inflamm"), ("contusion", r"contusion|bruise"),
    ("tightness", r"tightness"), ("tendinopathy", r"tendinitis|tendonitis|tendinopathy"),
    ("surgery_recovery", r"surgery|repair|rehab|recovery"),
    ("management", r"injury management|management"), ("dislocation", r"disloc|sublux"),
]


def parse_injury_text(text: str) -> dict:
    s = str(text or "").lower()
    laterality = "left" if re.search(r"\bleft\b", s) else ("right" if re.search(r"\bright\b", s) else "unknown")
    body = next((label for label, pat in BODY_PATTERNS if re.search(pat, s)), "unknown")
    dx = next((label for label, pat in DX_PATTERNS if re.search(pat, s)), "unknown")
    lower = body in {"achilles", "hamstring", "calf", "adductor_groin", "quadriceps", "ankle", "knee", "foot", "toe", "hip", "leg_unspecified"}
    upper = body in {"shoulder", "elbow", "wrist", "hand_finger"}
    region = "lower_extremity" if lower else ("upper_extremity" if upper else ("head_neck" if body in {"head_concussion", "neck"} else ("trunk" if body in {"back", "core_abdomen", "rib_chest"} else "unknown")))
    return {"injury_body_part": body, "injury_laterality": laterality, "injury_diagnosis_class": dx, "injury_region": region}


def normalize_playerlog(df: pd.DataFrame, season: str) -> pd.DataFrame:
    d = df.copy()
    mapping = {
        "gameId": "game_id", "GAME_ID": "game_id",
        "teamId": "team_id", "TEAM_ID": "team_id",
        "teamTricode": "team", "TEAM_ABBREVIATION": "team",
        "personId": "player_id", "PLAYER_ID": "player_id",
        "firstName": "first_name", "familyName": "last_name",
        "PLAYER_NAME": "player_name", "minutes": "minutes_raw", "MIN": "minutes_raw",
        "comment": "comment", "GAME_DATE": "game_date",
    }
    for src, dst in mapping.items():
        if src in d.columns and dst not in d.columns:
            d = d.rename(columns={src: dst})
    if "player_name" not in d.columns:
        d["player_name"] = (d.get("first_name", "").astype(str) + " " + d.get("last_name", "").astype(str)).str.strip()
    for c in ["game_id", "team", "player_id", "player_name", "minutes_raw"]:
        if c not in d.columns:
            raise ValueError(f"player log missing required column {c}; columns={list(d.columns)[:30]}")
    if "comment" not in d.columns:
        d["comment"] = ""
    d["game_id"] = d["game_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(10)
    d["player_id"] = pd.to_numeric(d["player_id"], errors="coerce").astype("Int64")
    d["minutes"] = d["minutes_raw"].map(parse_minutes)
    d["played"] = d["minutes"] > 0
    d["player_key"] = d["player_name"].map(norm_name)
    d["season"] = season
    return d


def normalize_team_schedule(df: pd.DataFrame, season: str) -> pd.DataFrame:
    d = df.copy()
    ren = {}
    for c in d.columns:
        lc = str(c).lower()
        if lc == "game_id": ren[c] = "game_id"
        elif lc == "game_date": ren[c] = "game_date"
        elif lc == "team_abbreviation": ren[c] = "team"
        elif lc == "matchup": ren[c] = "matchup"
    d = d.rename(columns=ren)
    needed = {"game_id", "game_date", "team", "matchup"}
    if not needed.issubset(d.columns):
        raise ValueError(f"team log lacks {needed - set(d.columns)}")
    d["game_id"] = d["game_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(10)
    d["game_date"] = pd.to_datetime(d["game_date"], errors="coerce").dt.normalize()
    d["is_home"] = d["matchup"].astype(str).str.contains(r"\bvs\.?\b", case=False, regex=True)
    d["season"] = season

    rows = []
    for gid, g in d.groupby("game_id", sort=False):
        home = g[g["is_home"]]
        away = g[~g["is_home"]]
        if home.empty or away.empty:
            continue
        rows.append({
            "season": season, "game_id": gid,
            "game_date": g["game_date"].dropna().iloc[0] if g["game_date"].notna().any() else pd.NaT,
            "home_team": str(home.iloc[0]["team"]), "away_team": str(away.iloc[0]["team"]),
        })
    return pd.DataFrame(rows)


def normalize_official_injury_reports(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    col = lambda *xs: _first_col(d, xs)
    c_date = col("Game Date", "game_date")
    c_match = col("Matchup", "matchup")
    c_team = col("Team", "team")
    c_player = col("Player Name", "player_name", "player")
    c_status = col("Current Status", "status", "current_status")
    c_reason = col("Reason", "reason")
    if not all([c_date, c_team, c_player, c_reason]):
        raise ValueError(f"could not identify official injury report columns: {list(d.columns)}")

    out = pd.DataFrame({
        "game_date": pd.to_datetime(d[c_date], errors="coerce").dt.normalize(),
        "matchup": d[c_match].astype(str) if c_match else "",
        "team_full": d[c_team].astype(str),
        "player_name_report": d[c_player].astype(str),
        "report_status": d[c_status].astype(str) if c_status else "",
        "report_reason": d[c_reason].astype(str),
    })
    out["team"] = out["team_full"].map(TEAM_NAME_TO_ABBR)
    out["player_key"] = out["player_name_report"].map(norm_name)

    # Preserve report timestamp when source data expose it. If no timestamp is
    # available, row order is used only as a pragmatic pilot fallback.
    ts_col = _first_col(d, ["report_ts", "report_timestamp", "report_datetime", "timestamp", "Report Time"])
    if ts_col:
        out["report_ts"] = pd.to_datetime(d[ts_col], errors="coerce")
    else:
        # Some parsed archives preserve the source PDF filename.
        src_col = _first_col(d, ["source", "source_file", "filename", "pdf"])
        if src_col:
            def _ts_from_file(x):
                s = str(x)
                m = re.search(r"(20\d{2}-\d{2}-\d{2})_(\d{2})(AM|PM)", s, re.I)
                if not m: return pd.NaT
                h = int(m.group(2)) % 12 + (12 if m.group(3).upper() == "PM" else 0)
                return pd.Timestamp(f"{m.group(1)} {h:02d}:30:00")
            out["report_ts"] = d[src_col].map(_ts_from_file)
        else:
            out["report_ts"] = pd.NaT

    fam = [reason_family(r) for r in out["report_reason"]]
    out["report_reason_family"] = fam
    parsed = pd.DataFrame([parse_injury_text(x) for x in out["report_reason"]])
    out = pd.concat([out.reset_index(drop=True), parsed], axis=1)
    return out.dropna(subset=["game_date", "player_key"]).reset_index(drop=True)


def collapse_reports_to_player_game(reports: pd.DataFrame) -> pd.DataFrame:
    """Latest known designation per player/team/game.

    For a causal/predictive study, replace this with 'latest report strictly before
    tip-off'. This function is adequate for a feasibility pilot because actual
    participation comes from the box score, not the report status.
    """
    d = reports.copy()
    keys = ["game_date", "team", "player_key"]
    if d["report_ts"].notna().any():
        d = d.sort_values(keys + ["report_ts"])
    else:
        d["_row_order"] = np.arange(len(d))
        d = d.sort_values(keys + ["_row_order"])
    return d.groupby(keys, as_index=False).tail(1).reset_index(drop=True)


# ---------------------------- NBA live boxscore -----------------------------

def fetch_live_boxscore(game_id: str, use_cache: bool = True, retries: int | None = None,
                       timeout: tuple | None = None) -> dict:
    """Fast, cache-first fetch of one static NBA live boxscore JSON.

    The CDN portion of the pipeline is safe to parallelize. Cache files are
    written atomically, so interrupting the notebook cannot leave half-written
    JSON that poisons a later resume.
    """
    gid = str(game_id).zfill(10)
    p = CACHE / "live_boxscore" / f"{gid}.json"
    p.parent.mkdir(parents=True, exist_ok=True)

    if use_cache and p.exists() and p.stat().st_size > 20:
        try:
            return json.loads(p.read_text())
        except Exception:
            # Corrupt/interrupted cache: remove and refetch exactly this game.
            try: p.unlink()
            except Exception: pass

    retries = int(globals().get("LIVE_RETRIES", 2) if retries is None else retries)
    timeout = timeout or (
        float(globals().get("LIVE_CONNECT_TIMEOUT", 5)),
        float(globals().get("LIVE_READ_TIMEOUT", 12)),
    )
    url = NBA_LIVE_BOXSCORE.format(game_id=gid)
    last = None
    for attempt in range(max(1, retries)):
        try:
            r = _http_session().get(url, timeout=timeout)
            r.raise_for_status()
            js = r.json()
            _atomic_write_json(p, js)
            return js
        except Exception as e:
            last = e
            if attempt + 1 < retries:
                time.sleep(0.5 * (2 ** attempt))
    raise RuntimeError(f"live boxscore failed {gid}: {last}")


def flatten_live_boxscore(js: dict) -> tuple[pd.DataFrame, dict]:
    game = js.get("game", js.get("boxScore", {}).get("game", {})) or {}
    arena = game.get("arena", {}) or {}
    meta = {
        "game_id": str(game.get("gameId", "")),
        "game_time_utc": pd.to_datetime(game.get("gameTimeUTC"), errors="coerce", utc=True),
        "game_time_local": game.get("gameTimeLocal"),
        "game_et": game.get("gameEt"),
        "game_duration": game.get("duration"),
        "arena_name": arena.get("arenaName"),
        "arena_city": arena.get("arenaCity"),
        "arena_state": arena.get("arenaState"),
        "arena_country": arena.get("arenaCountry"),
        "arena_timezone": arena.get("arenaTimezone"),
    }
    rows = []
    for side in ["homeTeam", "awayTeam"]:
        team = game.get(side, {}) or {}
        for p in team.get("players", []) or []:
            st = p.get("statistics", {}) or {}
            rows.append({
                "game_id": str(game.get("gameId", "")),
                "team": team.get("teamTricode"),
                "player_id": p.get("personId"),
                "player_name_live": p.get("name"),
                "live_status": p.get("status"),
                "live_played": bool(p.get("played")) if p.get("played") is not None else parse_minutes(st.get("minutes")) > 0,
                "not_playing_reason": p.get("notPlayingReason"),
                "not_playing_description": p.get("notPlayingDescription"),
                "live_minutes": parse_minutes(st.get("minutes")),
            })
    return pd.DataFrame(rows), meta


def build_live_availability(game_ids: Iterable[str], use_cache: bool = True,
                            workers: int | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Parallel, resume-safe live-boxscore collector (core flattened schema)."""
    gids = list(dict.fromkeys(str(x).zfill(10) for x in game_ids if pd.notna(x)))
    workers = int(workers or globals().get("LIVE_WORKERS", 16))
    cache_dir = CACHE / "live_boxscore"
    cached_before = {g for g in gids if (cache_dir / f"{g}.json").exists()}
    players, games, failures = [], [], []
    done = downloaded = failed = 0

    def _one(gid):
        js = fetch_live_boxscore(gid, use_cache=use_cache)
        p, m = flatten_live_boxscore(js)
        return gid, p, m

    print(f"  live boxscores: {len(cached_before):,}/{len(gids):,} already cached; "
          f"processing with {workers} workers")
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(_one, gid): gid for gid in gids}
        for fut in as_completed(futs):
            gid = futs[fut]
            try:
                _, p, m = fut.result()
                players.append(p); games.append(m)
                if gid not in cached_before: downloaded += 1
            except Exception as e:
                failures.append({"game_id": gid, "error": str(e)})
                failed += 1
            done += 1
            if done % 50 == 0 or done == len(gids):
                print(f"  live boxscores: {done:,}/{len(gids):,} | "
                      f"cached={min(done, len(cached_before)):,}+ | "
                      f"new={downloaded:,} | failed={failed:,}")

    pd.DataFrame(failures).to_csv(ROOT / "live_boxscore_failures.csv", index=False)
    return (pd.concat(players, ignore_index=True, sort=False) if players else pd.DataFrame(),
            pd.DataFrame(games))


# ------------------------------ join + features -----------------------------

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1); dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def add_schedule_context(pg: pd.DataFrame) -> pd.DataFrame:
    d = pg.copy().sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    # Exact start times if live boxscore is available; otherwise date-only fallback.
    d["game_time_utc"] = pd.to_datetime(d.get("game_time_utc"), errors="coerce", utc=True)
    d["prev_game_time_utc"] = d.groupby("player_id")["game_time_utc"].shift(1)
    d["rest_hours"] = (d["game_time_utc"] - d["prev_game_time_utc"]).dt.total_seconds() / 3600
    prev_date = d.groupby("player_id")["game_date"].shift(1)
    d["calendar_days_since_prev"] = (d["game_date"] - prev_date).dt.days
    d["is_back_to_back_calendar"] = (d["calendar_days_since_prev"] == 1).astype("Int64")
    d["short_rest_lt36h"] = np.where(d["rest_hours"].notna(), d["rest_hours"] < 36, np.nan)

    city = d.get("arena_city", pd.Series(index=d.index, dtype=object)).fillna("").astype(str).str.lower()
    coords = city.map(CITY_COORDS)
    d["arena_lat"] = coords.map(lambda x: x[0] if isinstance(x, tuple) else np.nan)
    d["arena_lon"] = coords.map(lambda x: x[1] if isinstance(x, tuple) else np.nan)
    plat = d.groupby("player_id")["arena_lat"].shift(1)
    plon = d.groupby("player_id")["arena_lon"].shift(1)
    d["travel_km_since_prev"] = [
        haversine_km(a,b,c,e) if all(pd.notna(x) for x in [a,b,c,e]) else np.nan
        for a,b,c,e in zip(plat, plon, d["arena_lat"], d["arena_lon"])
    ]
    return d


def build_player_game_master(season: str, fetch_live: bool = True, official_reports: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    player = normalize_playerlog(get_public_playerlog(season), season)
    sched = normalize_team_schedule(get_public_teamlog(season), season)

    # Playerlog may already contain game_date; schedule is the authoritative join here.
    if "game_date" in player.columns:
        player = player.drop(columns=["game_date"])
    d = player.merge(sched, on=["season", "game_id"], how="left", validate="many_to_one")
    d["game_date"] = pd.to_datetime(d["game_date"], errors="coerce").dt.normalize()
    d["is_home"] = (d["team"] == d["home_team"]).astype("Int64")
    d["opponent"] = np.where(d["team"] == d["home_team"], d["away_team"], d["home_team"])

    if fetch_live:
        live_p, live_g = build_live_availability(d["game_id"].dropna().unique())
        if not live_p.empty:
            live_p["player_id"] = pd.to_numeric(live_p["player_id"], errors="coerce").astype("Int64")
            d = d.merge(live_p, on=["game_id", "team", "player_id"], how="left")
        if not live_g.empty:
            d = d.merge(live_g, on="game_id", how="left")

    # Official report diagnosis layer.
    if official_reports is not None and len(official_reports):
        rr = collapse_reports_to_player_game(normalize_official_injury_reports(official_reports))
        keep = ["game_date", "team", "player_key", "report_status", "report_reason", "report_reason_family",
                "injury_body_part", "injury_laterality", "injury_diagnosis_class", "injury_region", "report_ts"]
        d = d.merge(rr[keep], on=["game_date", "team", "player_key"], how="left")

    # Actual participation should be boxscore-driven when live data exist.
    if "live_played" in d.columns:
        d["played_actual"] = np.where(d["live_played"].notna(), d["live_played"], d["played"]).astype(bool)
    else:
        d["played_actual"] = d["played"].astype(bool)

    live_reason = d.get("not_playing_reason", pd.Series("", index=d.index))
    live_desc = d.get("not_playing_description", pd.Series("", index=d.index))
    report_reason = d.get("report_reason", pd.Series("", index=d.index))
    comment = d.get("comment", pd.Series("", index=d.index))
    d["absence_reason_family"] = [reason_family(a,b,c,e) for a,b,c,e in zip(live_reason, live_desc, report_reason, comment)]
    d.loc[d["played_actual"], "absence_reason_family"] = "played"

    # Use report diagnosis first; otherwise parse available live/comment text.
    if "injury_body_part" not in d.columns:
        d["injury_body_part"] = "unknown"; d["injury_laterality"] = "unknown"
        d["injury_diagnosis_class"] = "unknown"; d["injury_region"] = "unknown"
    fallback_text = (live_desc.fillna("").astype(str) + " " + comment.fillna("").astype(str))
    fb = pd.DataFrame([parse_injury_text(x) for x in fallback_text])
    for c in ["injury_body_part", "injury_laterality", "injury_diagnosis_class", "injury_region"]:
        missing = d[c].isna() | d[c].astype(str).str.lower().isin(["", "nan", "unknown"])
        d.loc[missing, c] = fb.loc[missing, c].values

    d = add_schedule_context(d)
    return d.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)


def _safe_mean(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna()
    return float(x.mean()) if len(x) else np.nan


def _safe_slope(y):
    y = pd.to_numeric(pd.Series(y), errors="coerce").dropna().to_numpy(float)
    if len(y) < 2: return np.nan
    return float(np.polyfit(np.arange(1, len(y)+1), y, 1)[0])


def build_injury_episodes(pg: pd.DataFrame, min_games_missed: int = 2, pre_games: int = 10, post_games: int = 10) -> pd.DataFrame:
    """Create one row per injury-related time-loss episode.

    Primary episode definition: >= min_games_missed consecutive player-team games
    not played with medical reason family == injury. Illness is excluded from the
    primary cohort but retained in player_game_master.csv for sensitivity analyses.
    """
    rows = []
    d = pg.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    for pid, g0 in d.groupby("player_id", sort=False):
        g = g0.reset_index(drop=True)
        med = (~g["played_actual"]) & (g["absence_reason_family"] == "injury")
        idx = np.flatnonzero(med.to_numpy())
        if len(idx) == 0: continue

        # contiguous indices among this player's team-game sequence
        starts = [idx[0]]; ends = []
        for a,b in zip(idx[:-1], idx[1:]):
            if b != a + 1:
                ends.append(a); starts.append(b)
        ends.append(idx[-1])

        for start, end in zip(starts, ends):
            nmiss = end - start + 1
            if nmiss < min_games_missed: continue
            # return = first actually played game after the missed run
            ret_candidates = g.index[(g.index > end) & g["played_actual"]].tolist()
            if not ret_candidates: continue
            ret = ret_candidates[0]
            prev_played = g.iloc[:start]
            prev_played = prev_played[prev_played["played_actual"]].tail(pre_games)
            if len(prev_played) < max(3, min(5, pre_games)):
                continue

            # Stop post window at the next injury-related missed game.
            tail = g.iloc[ret:].copy()
            future_med = tail.index[(~tail["played_actual"]) & (tail["absence_reason_family"] == "injury")].tolist()
            next_inj_idx = future_med[0] if future_med else None
            if next_inj_idx is not None:
                post_pool = g.loc[ret:next_inj_idx-1]
            else:
                post_pool = g.loc[ret:]
            post = post_pool[post_pool["played_actual"]].head(post_games)
            if post.empty: continue

            baseline = float(prev_played["minutes"].median())
            post_minutes = post["minutes"].astype(float).to_numpy()
            ratios = post_minutes / baseline if baseline > 0 else np.full(len(post_minutes), np.nan)
            return_row = g.loc[ret]
            ep = g.loc[start:end]

            # Injury label: prefer the modal known report body part/region in episode.
            def mode_known(col):
                s = ep[col].dropna().astype(str)
                s = s[~s.str.lower().isin(["", "nan", "unknown"])]
                return s.mode().iloc[0] if len(s) else "unknown"

            first90 = np.flatnonzero(ratios >= 0.90)
            episode_id = f"{int(pid)}_{pd.Timestamp(g.loc[start,'game_date']).date()}_{start}"

            row = {
                "episode_id": episode_id, "season": g.loc[start, "season"],
                "player_id": pid, "player_name": g.loc[start, "player_name"],
                "team": g.loc[start, "team"],
                "injury_start_game_date": g.loc[start, "game_date"],
                "last_missed_game_date": g.loc[end, "game_date"],
                "return_game_date": return_row["game_date"],
                "games_missed": nmiss,
                "calendar_days_from_first_miss_to_return": int((return_row["game_date"] - g.loc[start, "game_date"]).days),
                "injury_region": mode_known("injury_region"),
                "injury_body_part": mode_known("injury_body_part"),
                "injury_laterality": mode_known("injury_laterality"),
                "injury_diagnosis_class": mode_known("injury_diagnosis_class"),
                "pre_n_games": len(prev_played),
                "pre_median_minutes": baseline,
                "pre_mean_minutes": _safe_mean(prev_played["minutes"]),
                "return_game_minutes": float(post_minutes[0]),
                "return_minutes_ratio": float(ratios[0]) if len(ratios) else np.nan,
                "post3_mean_minutes_ratio": float(np.nanmean(ratios[:3])) if len(ratios) else np.nan,
                "post5_mean_minutes_ratio": float(np.nanmean(ratios[:5])) if len(ratios) else np.nan,
                "ramp_slope_minutes_per_game": _safe_slope(post_minutes[:5]),
                "max_positive_minute_step_first5": float(np.max(np.diff(post_minutes[:5]))) if len(post_minutes[:5]) >= 2 else np.nan,
                "ramp_deficit_auc_first5": float(np.nansum(np.maximum(0, baseline - post_minutes[:5]))),
                "games_to_90pct_baseline": int(first90[0] + 1) if len(first90) else np.nan,
                "return_is_b2b": return_row.get("is_back_to_back_calendar", np.nan),
                "return_rest_hours": return_row.get("rest_hours", np.nan),
                "return_travel_km": return_row.get("travel_km_since_prev", np.nan),
                "b2b_count_first5_post": int(pd.to_numeric(post.head(5)["is_back_to_back_calendar"], errors="coerce").fillna(0).sum()),
                "min_rest_hours_first5_post": pd.to_numeric(post.head(5)["rest_hours"], errors="coerce").min(),
                "travel_km_first5_post": pd.to_numeric(post.head(5)["travel_km_since_prev"], errors="coerce").sum(min_count=1),
            }

            # Performance deltas when these public playerlog advanced fields exist.
            perf_candidates = [
                ("trueShootingPercentage", "ts_pct"), ("usagePercentage", "usage_pct"),
                ("netRating", "net_rating"), ("PIE", "pie"), ("plusMinusPoints", "plus_minus"),
                ("points", "points"),
            ]
            for src, label in perf_candidates:
                if src in g.columns:
                    row[f"pre_{label}"] = _safe_mean(prev_played[src])
                    row[f"post5_{label}"] = _safe_mean(post.head(5)[src])
                    row[f"post5_minus_pre_{label}"] = row[f"post5_{label}"] - row[f"pre_{label}"] if pd.notna(row[f"pre_{label}"]) and pd.notna(row[f"post5_{label}"]) else np.nan

            # Subsequent injury outcomes.
            after_ret = g.loc[ret+1:].copy()
            injury_after = after_ret[(~after_ret["played_actual"]) & (after_ret["absence_reason_family"] == "injury")]
            if len(injury_after):
                nxt = injury_after.iloc[0]
                row["next_injury_absence_date"] = nxt["game_date"]
                row["days_to_next_injury_absence"] = int((nxt["game_date"] - return_row["game_date"]).days)
                row["next_injury_body_part"] = nxt.get("injury_body_part", "unknown")
                row["next_injury_region"] = nxt.get("injury_region", "unknown")
                row["any_injury_absence_within_14d"] = int(row["days_to_next_injury_absence"] <= 14)
                row["any_injury_absence_within_30d"] = int(row["days_to_next_injury_absence"] <= 30)
                row["any_injury_absence_within_60d"] = int(row["days_to_next_injury_absence"] <= 60)
                same_region = (row["injury_region"] != "unknown" and str(nxt.get("injury_region")) == row["injury_region"])
                same_body = (row["injury_body_part"] != "unknown" and str(nxt.get("injury_body_part")) == row["injury_body_part"])
                row["same_region_recurrence_within_30d"] = int(same_region and row["days_to_next_injury_absence"] <= 30)
                row["same_body_part_recurrence_within_30d"] = int(same_body and row["days_to_next_injury_absence"] <= 30)
            else:
                row.update({
                    "next_injury_absence_date": pd.NaT, "days_to_next_injury_absence": np.nan,
                    "next_injury_body_part": "", "next_injury_region": "",
                    "any_injury_absence_within_14d": 0, "any_injury_absence_within_30d": 0,
                    "any_injury_absence_within_60d": 0, "same_region_recurrence_within_30d": 0,
                    "same_body_part_recurrence_within_30d": 0,
                })
            rows.append(row)
    return pd.DataFrame(rows)


def add_player_age(episodes: pd.DataFrame, season: str) -> pd.DataFrame:
    try:
        ps = get_public_player_season_stats(season)
    except Exception as e:
        print("age merge skipped:", e)
        return episodes
    idc = _first_col(ps, ["player_id", "PLAYER_ID", "personId"])
    agec = _first_col(ps, ["age", "AGE"])
    if not idc or not agec: return episodes
    x = ps[[idc, agec]].copy().rename(columns={idc:"player_id", agec:"season_age"})
    x["player_id"] = pd.to_numeric(x["player_id"], errors="coerce").astype("Int64")
    x = x.drop_duplicates("player_id")
    return episodes.merge(x, on="player_id", how="left")


def run_pilot_2024_25(fetch_live: bool = True, min_games_missed: int = 2):
    season = "2024-25"
    print("Downloading/reading player and team game logs...")
    reports = get_2425_official_injury_designations()
    print(f"Official injury designation rows: {len(reports):,}")
    pg = build_player_game_master(season, fetch_live=fetch_live, official_reports=reports)
    pg.to_csv(ROOT / "player_game_master_2024_25.csv", index=False)
    print(f"Player-game rows: {len(pg):,}")

    ep = build_injury_episodes(pg, min_games_missed=min_games_missed)
    ep = add_player_age(ep, season)
    ep.to_csv(ROOT / "rtp_injury_episodes_2024_25.csv", index=False)
    print(f"RTP episodes: {len(ep):,}")

    audit = {
        "season": season,
        "player_game_rows": int(len(pg)),
        "rtp_episode_rows": int(len(ep)),
        "live_boxscore_requested": bool(fetch_live),
        "played_rows": int(pg["played_actual"].sum()) if len(pg) else 0,
        "injury_absence_rows": int((pg["absence_reason_family"] == "injury").sum()) if len(pg) else 0,
        "unknown_absence_rows": int(((~pg["played_actual"]) & (pg["absence_reason_family"] == "unknown")).sum()) if len(pg) else 0,
        "known_body_part_injury_absences": int(((pg["absence_reason_family"] == "injury") & (pg["injury_body_part"] != "unknown")).sum()) if len(pg) else 0,
    }
    (ROOT / "pilot_audit_2024_25.json").write_text(json.dumps(audit, indent=2))
    print(json.dumps(audit, indent=2))
    return pg, ep


# ------------------------------- toy self-test ------------------------------

def self_test():
    assert abs(parse_minutes("26:30") - 26.5) < 1e-9
    assert abs(parse_minutes("PT26M30.00S") - 26.5) < 1e-9
    assert norm_name("Williams III, Robert") == "robert williams"
    assert reason_family("INACTIVE_INJURY", "Left knee soreness") == "injury"
    assert reason_family("DNP_REST") == "rest"
    p = parse_injury_text("Injury/Illness - Left Hamstring; Strain")
    assert p["injury_body_part"] == "hamstring" and p["injury_laterality"] == "left" and p["injury_diagnosis_class"] == "strain"
    print("self-test passed")


## 4. Maximal archive extensions

The code below expands the first prototype in several important ways:

1. **Preserves every scalar field from the NBA live player/box-score JSON**, rather than selecting only a few columns.
2. Computes B2B/rest from the **team's schedule**, distinct from time since the player's previous appearance.
3. Precomputes rolling 3/5/10/15/20-game and 3/7/14/28-day workload histories.
4. Builds **all absence episodes**, not only injury episodes, allowing suspension/rest/personal/G-League negative-control analyses.
5. Builds a long-form **episode game window** around RTP (`-10 ... 0 ... +20`).
6. Adds optional cache-first deep NBA Stats endpoint collection for tracking, hustle, rotations, advanced box scores, etc.


In [ ]:
# ========================= MAXIMAL ARCHIVE ADD-ONS =========================
# These functions extend the core feasibility extractor without requiring them
# for the first pilot. They are intentionally cache-first and failure-tolerant.

from pathlib import Path
import importlib
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import time
import math
import re
import numpy as np
import pandas as pd

DEEP_RAW = ROOT / "deep_raw"
DEEP_CACHE = CACHE / "deep_endpoints"
DEEP_RAW.mkdir(parents=True, exist_ok=True)
DEEP_CACHE.mkdir(parents=True, exist_ok=True)


def _flatten_scalar_dict(prefix, obj, out):
    """Recursively retain scalar JSON fields; lists/dicts are JSON-serialized.

    Useful for live box score feeds whose schemas occasionally gain new fields.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}_{k}" if prefix else str(k)
            if isinstance(v, dict):
                _flatten_scalar_dict(key, v, out)
            elif isinstance(v, list):
                # Preserve list payload rather than silently discarding it.
                out[key] = json.dumps(v, default=str)
            else:
                out[key] = v
    else:
        out[prefix] = obj


def flatten_live_boxscore_maximal(js: dict):
    """Maximal version of flatten_live_boxscore.

    Keeps every scalar player field and every scalar statistic field currently
    returned by cdn.nba.com, while retaining the standard canonical columns.
    """
    game = js.get("game", js.get("boxScore", {}).get("game", {})) or {}
    arena = game.get("arena", {}) or {}

    game_meta = {}
    for k, v in game.items():
        if k in {"homeTeam", "awayTeam", "arena", "officials"}:
            continue
        if isinstance(v, (str, int, float, bool)) or v is None:
            game_meta[f"live_game_{k}"] = v
    for k, v in arena.items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            game_meta[f"arena_{k}"] = v

    game_meta.update({
        "game_id": str(game.get("gameId", "")),
        "game_time_utc": pd.to_datetime(game.get("gameTimeUTC"), errors="coerce", utc=True),
        "game_time_local": game.get("gameTimeLocal"),
        "game_et": game.get("gameEt"),
        "game_duration": game.get("duration"),
        "arena_name": arena.get("arenaName"),
        "arena_city": arena.get("arenaCity"),
        "arena_state": arena.get("arenaState"),
        "arena_country": arena.get("arenaCountry"),
        "arena_timezone": arena.get("arenaTimezone"),
    })

    rows = []
    for side in ["homeTeam", "awayTeam"]:
        team = game.get(side, {}) or {}
        team_tri = team.get("teamTricode")
        for p in team.get("players", []) or []:
            row = {
                "game_id": str(game.get("gameId", "")),
                "team": team_tri,
                "player_id": p.get("personId"),
                "player_name_live": p.get("name"),
                "live_status": p.get("status"),
                "not_playing_reason": p.get("notPlayingReason"),
                "not_playing_description": p.get("notPlayingDescription"),
            }
            # Preserve every scalar player-level field.
            for k, v in p.items():
                if k == "statistics":
                    continue
                if isinstance(v, (str, int, float, bool)) or v is None:
                    row[f"live_player_{k}"] = v
            # Preserve every statistics field with a namespaced prefix.
            stats = p.get("statistics", {}) or {}
            for k, v in stats.items():
                if isinstance(v, (str, int, float, bool)) or v is None:
                    row[f"live_stat_{k}"] = v
                else:
                    row[f"live_stat_{k}"] = json.dumps(v, default=str)

            row["live_minutes"] = parse_minutes(stats.get("minutes"))
            played_value = p.get("played")
            row["live_played"] = bool(played_value) if played_value is not None else row["live_minutes"] > 0
            rows.append(row)
    return pd.DataFrame(rows), game_meta


def build_live_availability_maximal(game_ids, use_cache=True, workers=None):
    """High-throughput CDN collector preserving the maximal live schema.

    Static cdn.nba.com boxscores are fetched concurrently. Every successful JSON
    is cached immediately, so stopping/restarting only processes missing games.
    """
    gids = list(dict.fromkeys(str(x).zfill(10) for x in game_ids if pd.notna(x)))
    workers = int(workers or globals().get("LIVE_WORKERS", 16))
    cache_dir = CACHE / "live_boxscore"
    cached_before = {g for g in gids if (cache_dir / f"{g}.json").exists()}
    players, games, failures = [], [], []
    done = downloaded = failed = cache_hits_done = 0

    def _one(gid):
        was_cached = gid in cached_before
        js = fetch_live_boxscore(gid, use_cache=use_cache)
        p, g = flatten_live_boxscore_maximal(js)
        return gid, was_cached, p, g

    print(f"  maximal live boxscores: {len(cached_before):,}/{len(gids):,} already cached; "
          f"{workers} parallel workers")
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(_one, gid): gid for gid in gids}
        for fut in as_completed(futures):
            gid = futures[fut]
            try:
                _, was_cached, p, g = fut.result()
                players.append(p); games.append(g)
                if was_cached: cache_hits_done += 1
                else: downloaded += 1
            except Exception as e:
                failures.append({"game_id": gid, "error": str(e)})
                failed += 1
            done += 1
            if done % 50 == 0 or done == len(gids):
                print(f"  maximal live boxscores: {done:,}/{len(gids):,} | "
                      f"cache_hits={cache_hits_done:,} | new={downloaded:,} | failed={failed:,}")

    pd.DataFrame(failures).to_csv(ROOT / "live_boxscore_failures_maximal.csv", index=False)
    print(f"  live stage complete: cache_hits={cache_hits_done:,}, "
          f"downloaded={downloaded:,}, failed={failed:,}")
    return (
        pd.concat(players, ignore_index=True, sort=False) if players else pd.DataFrame(),
        pd.DataFrame(games),
    )


def add_team_schedule_context(player_games: pd.DataFrame) -> pd.DataFrame:
    """Calculate B2B/density from TEAM schedule, not player appearances.

    This is distinct from time since the player's previous appearance, which is
    separately useful for layoff duration.
    """
    d = player_games.copy()
    # Build one row per team-game.
    base_cols = [c for c in ["season", "game_id", "game_date", "home_team", "away_team", "game_time_utc"] if c in d.columns]
    games = d[base_cols].drop_duplicates(["game_id"]).copy()
    rows = []
    for r in games.itertuples(index=False):
        dd = r._asdict()
        for side in ("home_team", "away_team"):
            rows.append({
                "season": dd.get("season"),
                "game_id": str(dd.get("game_id")),
                "team": dd.get(side),
                "game_date": pd.to_datetime(dd.get("game_date"), errors="coerce").normalize(),
                "team_game_time_utc": pd.to_datetime(dd.get("game_time_utc"), errors="coerce", utc=True),
                "team_is_home": int(side == "home_team"),
            })
    ts = pd.DataFrame(rows)
    if ts.empty:
        return d
    ts = ts.sort_values(["team", "game_date", "game_id"]).reset_index(drop=True)
    ts["team_prev_game_date"] = ts.groupby("team")["game_date"].shift(1)
    ts["team_calendar_rest_days"] = (ts["game_date"] - ts["team_prev_game_date"]).dt.days
    ts["team_is_b2b"] = (ts["team_calendar_rest_days"] == 1).astype("Int64")
    ts["team_prev_game_time_utc"] = ts.groupby("team")["team_game_time_utc"].shift(1)
    ts["team_rest_hours"] = (ts["team_game_time_utc"] - ts["team_prev_game_time_utc"]).dt.total_seconds()/3600

    # Schedule density based on number of team games in prior N calendar days.
    for window in [3, 4, 6, 7, 14, 28]:
        vals = []
        for _, g in ts.groupby("team", sort=False):
            dates = g["game_date"].to_numpy(dtype="datetime64[ns]")
            for j, dt in enumerate(dates):
                lo = dt - np.timedelta64(window - 1, "D")
                vals.append(int(np.sum((dates[:j+1] >= lo) & (dates[:j+1] <= dt))))
        # group order matches ts because groupby(sort=False) traverses current order blocks
        ts[f"team_games_last_{window}d"] = vals

    ts["team_3in4"] = (ts["team_games_last_4d"] >= 3).astype("Int64")
    ts["team_4in6"] = (ts["team_games_last_6d"] >= 4).astype("Int64")
    ts["team_5in7"] = (ts["team_games_last_7d"] >= 5).astype("Int64")

    keep = [c for c in ts.columns if c not in {"game_date"}]
    d["game_id"] = d["game_id"].astype(str)
    d = d.merge(ts[keep], on=["season", "game_id", "team"], how="left")

    # Player-appearance rest is a separate construct.
    d = d.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    played = d.get("played_actual", pd.Series(False, index=d.index)).fillna(False).astype(bool)
    last_played_time = []
    state = {}
    for i, r in d.iterrows():
        pid = r.get("player_id")
        current = pd.to_datetime(r.get("game_time_utc"), errors="coerce", utc=True)
        prev = state.get(pid, pd.NaT)
        last_played_time.append(prev)
        if played.iloc[i] and pd.notna(current):
            state[pid] = current
    d["previous_player_appearance_time_utc"] = last_played_time
    d["hours_since_last_player_appearance"] = (
        pd.to_datetime(d.get("game_time_utc"), errors="coerce", utc=True) -
        pd.to_datetime(d["previous_player_appearance_time_utc"], errors="coerce", utc=True)
    ).dt.total_seconds()/3600
    return d


def add_player_workload_rolling_features(pg: pd.DataFrame) -> pd.DataFrame:
    """Precompute workload history without future leakage.

    Rolling values are shifted so each row only sees workload before that game.
    """
    d = pg.copy().sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    mins = pd.to_numeric(d["minutes"], errors="coerce").fillna(0.0)
    d["minutes_numeric"] = mins
    for n in [3, 5, 10, 15, 20]:
        d[f"prev{n}_team_games_minutes"] = d.groupby("player_id")["minutes_numeric"].transform(
            lambda s: s.shift(1).rolling(n, min_periods=1).sum()
        )
        d[f"prev{n}_team_games_mean_minutes"] = d.groupby("player_id")["minutes_numeric"].transform(
            lambda s: s.shift(1).rolling(n, min_periods=1).mean()
        )

    # Calendar-day acute/chronic windows.
    out_parts = []
    for pid, g in d.groupby("player_id", sort=False):
        g = g.sort_values(["game_date", "game_id"]).copy()
        dates = pd.to_datetime(g["game_date"], errors="coerce")
        m = pd.to_numeric(g["minutes_numeric"], errors="coerce").fillna(0).to_numpy()
        for days in [3, 7, 14, 28]:
            arr = []
            for i, dt in enumerate(dates):
                if pd.isna(dt):
                    arr.append(np.nan); continue
                lo = dt - pd.Timedelta(days=days)
                mask = (dates.iloc[:i] > lo) & (dates.iloc[:i] < dt)
                arr.append(float(m[:i][mask.to_numpy()].sum()))
            g[f"minutes_prev_{days}d"] = arr
        out_parts.append(g)
    d = pd.concat(out_parts, ignore_index=True, sort=False) if out_parts else d
    d["acute_chronic_minutes_7d_28d"] = d["minutes_prev_7d"] / (d["minutes_prev_28d"] / 4.0).replace(0, np.nan)
    return d


def build_all_absence_episodes(pg: pd.DataFrame, min_team_games_missed: int = 1) -> pd.DataFrame:
    """All contiguous missed-game episodes, medical and nonmedical."""
    d = pg.copy().sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    rows = []
    for pid, g in d.groupby("player_id", sort=False):
        g = g.reset_index(drop=True)
        missed = ~g["played_actual"].fillna(False).astype(bool)
        i = 0
        while i < len(g):
            if not missed.iloc[i]:
                i += 1; continue
            j = i
            while j + 1 < len(g) and missed.iloc[j+1]:
                j += 1
            n = j-i+1
            if n >= min_team_games_missed:
                ep = g.loc[i:j]
                fam = ep["absence_reason_family"].fillna("unknown").astype(str)
                primary = fam.mode().iloc[0] if len(fam) else "unknown"
                ret = j+1 if j+1 < len(g) else None
                rows.append({
                    "absence_episode_id": f"{pid}_{pd.Timestamp(g.loc[i,'game_date']).date()}_{i}",
                    "season": g.loc[i,"season"], "player_id": pid,
                    "player_name": g.loc[i,"player_name"], "team": g.loc[i,"team"],
                    "absence_reason_family": primary,
                    "first_missed_game_date": g.loc[i,"game_date"],
                    "last_missed_game_date": g.loc[j,"game_date"],
                    "team_games_missed": n,
                    "return_game_date": g.loc[ret,"game_date"] if ret is not None else pd.NaT,
                    "returned_same_season": int(ret is not None),
                    "injury_region": ep.get("injury_region", pd.Series(dtype=object)).mode().iloc[0] if "injury_region" in ep and len(ep["injury_region"].dropna()) else "unknown",
                    "injury_body_part": ep.get("injury_body_part", pd.Series(dtype=object)).mode().iloc[0] if "injury_body_part" in ep and len(ep["injury_body_part"].dropna()) else "unknown",
                })
            i = j+1
    return pd.DataFrame(rows)


def build_episode_game_windows(pg: pd.DataFrame, injury_episodes: pd.DataFrame,
                               pre_played_games=10, post_team_games=20) -> pd.DataFrame:
    """Long-format episode trajectory: -10...-1, return=0, +1...+20."""
    if injury_episodes is None or injury_episodes.empty:
        return pd.DataFrame()
    d = pg.copy().sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    d["game_date"] = pd.to_datetime(d["game_date"], errors="coerce").dt.normalize()
    rows = []
    for e in injury_episodes.itertuples(index=False):
        g = d[d["player_id"] == e.player_id].sort_values(["game_date", "game_id"]).reset_index(drop=True)
        ret_date = pd.to_datetime(e.return_game_date, errors="coerce").normalize()
        idx = g.index[g["game_date"] == ret_date].tolist()
        if not idx:
            continue
        r = idx[0]
        pre = g.loc[:r-1]
        pre = pre[pre["played_actual"].fillna(False).astype(bool)].tail(pre_played_games)
        pre_positions = list(range(-len(pre), 0))
        for rel, (_, row) in zip(pre_positions, pre.iterrows()):
            x = row.to_dict(); x.update({"episode_id": e.episode_id, "relative_game": rel, "window_phase": "preinjury_played"}); rows.append(x)
        post = g.loc[r:r+post_team_games]
        for rel, (_, row) in enumerate(post.iterrows()):
            x = row.to_dict(); x.update({"episode_id": e.episode_id, "relative_game": rel, "window_phase": "return_or_post"}); rows.append(x)
    return pd.DataFrame(rows)


# ----------------------------- DEEP NBA API --------------------------------
# Endpoint names evolve. We dynamically import each one and skip gracefully if
# a user's installed nba_api version does not contain it.

DEEP_ENDPOINT_SPECS = {
    "traditional": ("nba_api.stats.endpoints.boxscoretraditionalv3", "BoxScoreTraditionalV3"),
    "advanced": ("nba_api.stats.endpoints.boxscoreadvancedv3", "BoxScoreAdvancedV3"),
    "misc": ("nba_api.stats.endpoints.boxscoremiscv3", "BoxScoreMiscV3"),
    "scoring": ("nba_api.stats.endpoints.boxscorescoringv3", "BoxScoreScoringV3"),
    "usage": ("nba_api.stats.endpoints.boxscoreusagev3", "BoxScoreUsageV3"),
    "four_factors": ("nba_api.stats.endpoints.boxscorefourfactorsv3", "BoxScoreFourFactorsV3"),
    "player_tracking": ("nba_api.stats.endpoints.boxscoreplayertrackv3", "BoxScorePlayerTrackV3"),
    "hustle": ("nba_api.stats.endpoints.boxscorehustlev2", "BoxScoreHustleV2"),
    "defensive": ("nba_api.stats.endpoints.boxscoredefensivev2", "BoxScoreDefensiveV2"),
    "matchups": ("nba_api.stats.endpoints.boxscorematchupsv3", "BoxScoreMatchupsV3"),
    "rotation": ("nba_api.stats.endpoints.gamerotation", "GameRotation"),
    "summary": ("nba_api.stats.endpoints.boxscoresummaryv3", "BoxScoreSummaryV3"),
}

FOCUSED_DEEP_ENDPOINTS = ["advanced", "player_tracking", "hustle", "rotation"]
MAXIMAL_DEEP_ENDPOINTS = list(DEEP_ENDPOINT_SPECS)


def _load_endpoint(name):
    module_name, class_name = DEEP_ENDPOINT_SPECS[name]
    try:
        module = importlib.import_module(module_name)
        return getattr(module, class_name)
    except Exception as e:
        print(f"  endpoint unavailable in this nba_api version: {name}: {e}")
        return None


def _extract_endpoint_frames(obj):
    """Return all DataFrames from an nba_api endpoint response."""
    frames = []
    try:
        for i, df in enumerate(obj.get_data_frames()):
            frames.append((i, df.copy()))
        return frames
    except Exception:
        pass
    # Fallback through raw normalized dict if supported.
    try:
        data = obj.nba_response.get_normalized_dict()
        for i, (k, rows) in enumerate(data.items()):
            if isinstance(rows, list):
                frames.append((k, pd.DataFrame(rows)))
    except Exception:
        pass
    return frames


_NBA_STATS_RATE_LOCK = threading.Lock()
_NBA_STATS_LAST_START = 0.0

def _nba_stats_rate_limit():
    """Globally space stats.nba.com request starts even when using two workers."""
    global _NBA_STATS_LAST_START
    interval = float(globals().get("NBA_STATS_MIN_START_INTERVAL", 0.55))
    if interval <= 0: return
    with _NBA_STATS_RATE_LOCK:
        now = time.monotonic()
        wait = interval - (now - _NBA_STATS_LAST_START)
        if wait > 0:
            time.sleep(wait)
        _NBA_STATS_LAST_START = time.monotonic()

def _deep_manifest_table_count(game_id, endpoint_name):
    gid = str(game_id).zfill(10)
    manifest = DEEP_CACHE / endpoint_name / gid / "manifest.json"
    if not manifest.exists():
        return None
    try:
        return len(json.loads(manifest.read_text()).get("tables", []))
    except Exception:
        return None

def fetch_deep_endpoint_game(game_id, endpoint_name, use_cache=True, delay=0.0,
                             timeout=None, return_frames=True):
    """Fetch/cache one stats.nba.com endpoint x game.

    On archive-only runs, return_frames=False avoids rereading cached CSV tables
    merely to count them, which makes resumptions dramatically faster.
    """
    gid = str(game_id).zfill(10)
    out_dir = DEEP_CACHE / endpoint_name / gid
    manifest = out_dir / "manifest.json"
    if use_cache and manifest.exists():
        try:
            info = json.loads(manifest.read_text())
            if not return_frames:
                return int(len(info.get("tables", [])))
            frames = []
            for item in info.get("tables", []):
                p = out_dir / item["file"]
                if p.exists():
                    frames.append((item["table"], pd.read_csv(p, low_memory=False)))
            return frames
        except Exception:
            pass

    Cls = _load_endpoint(endpoint_name)
    if Cls is None:
        return 0 if not return_frames else []
    out_dir.mkdir(parents=True, exist_ok=True)
    timeout = int(timeout or globals().get("NBA_STATS_TIMEOUT", 45))
    retries = int(globals().get("NBA_STATS_RETRIES", 2))
    last = None
    for attempt in range(max(1, retries)):
        try:
            _nba_stats_rate_limit()
            try:
                obj = Cls(game_id=gid, timeout=timeout)
            except TypeError:
                obj = Cls(game_id=gid)
            frames = _extract_endpoint_frames(obj)
            tables = []
            for table_name, df in frames:
                fname = f"table_{str(table_name).replace('/','_')}.csv"
                df.to_csv(out_dir / fname, index=False)
                tables.append({"table": str(table_name), "file": fname,
                               "rows": int(len(df)), "cols": int(df.shape[1])})
            manifest.write_text(json.dumps({"game_id": gid, "endpoint": endpoint_name,
                                            "tables": tables}, indent=2))
            try:
                (out_dir / "FAILED.txt").unlink()
            except Exception:
                pass
            if delay: time.sleep(delay)
            return frames if return_frames else len(tables)
        except Exception as e:
            last = e
            if attempt + 1 < retries:
                time.sleep(1.0 * (2 ** attempt))
    (out_dir / "FAILED.txt").write_text(str(last))
    return 0 if not return_frames else []


def archive_deep_endpoints(game_ids, endpoint_names, use_cache=True, delay=0.0, workers=None):
    """Faster resume-safe deep archive with conservative stats.nba.com concurrency."""
    gids = list(dict.fromkeys(str(x).zfill(10) for x in game_ids if pd.notna(x)))
    workers = int(workers or globals().get("DEEP_WORKERS", 2))
    audit = []

    for e_i, name in enumerate(endpoint_names, 1):
        cached = {g: _deep_manifest_table_count(g, name) for g in gids}
        pending = [g for g in gids if cached[g] is None]
        for g in gids:
            if cached[g] is not None:
                n = int(cached[g])
                audit.append({"endpoint": name, "game_id": g, "tables_returned": n,
                              "ok": int(n > 0), "source": "cache"})

        print(f"\n=== deep endpoint {e_i}/{len(endpoint_names)}: {name} ===")
        print(f"  cached={len(gids)-len(pending):,}; pending={len(pending):,}; workers={workers}")
        if not pending:
            continue

        done = ok = failed = 0
        def _one(gid):
            n = fetch_deep_endpoint_game(gid, name, use_cache=use_cache,
                                         delay=delay, return_frames=False)
            return gid, int(n)

        with ThreadPoolExecutor(max_workers=max(1, workers)) as ex:
            futures = {ex.submit(_one, gid): gid for gid in pending}
            for fut in as_completed(futures):
                gid = futures[fut]
                try:
                    _, n = fut.result()
                    good = int(n > 0)
                    ok += good; failed += (1-good)
                    audit.append({"endpoint": name, "game_id": gid,
                                  "tables_returned": n, "ok": good, "source": "download"})
                except Exception as e:
                    failed += 1
                    audit.append({"endpoint": name, "game_id": gid,
                                  "tables_returned": 0, "ok": 0,
                                  "source": "exception", "error": str(e)})
                done += 1
                if done % 25 == 0 or done == len(pending):
                    print(f"  pending progress {done:,}/{len(pending):,} | ok={ok:,} | failed={failed:,}")

        # Save after every endpoint so an interrupted later endpoint loses nothing.
        pd.DataFrame(audit).to_csv(ROOT / "deep_archive_audit.csv", index=False)

    audit_df = pd.DataFrame(audit)
    audit_df.to_csv(ROOT / "deep_archive_audit.csv", index=False)
    return audit_df


def episode_window_game_ids(windows: pd.DataFrame):
    if windows is None or windows.empty:
        return []
    return windows["game_id"].dropna().astype(str).drop_duplicates().tolist()


def player_profile_archive(player_ids, season="2024-25", use_cache=True, delay=0.0, workers=None):
    """Cache-first CommonPlayerInfo archive with conservative parallel fetching."""
    try:
        mod = importlib.import_module("nba_api.stats.endpoints.commonplayerinfo")
        Cls = getattr(mod, "CommonPlayerInfo")
    except Exception as e:
        print("CommonPlayerInfo unavailable:", e)
        return pd.DataFrame()

    ids = pd.Series(player_ids).dropna().astype(int).unique().tolist()
    pdir = DEEP_CACHE / "common_player_info"; pdir.mkdir(parents=True, exist_ok=True)
    workers = int(workers or globals().get("DEEP_WORKERS", 2))
    rows = []
    pending = []
    for pid in ids:
        p = pdir / f"{pid}.csv"
        if use_cache and p.exists():
            try:
                df = pd.read_csv(p, low_memory=False)
                if not df.empty: rows.append(df.assign(_requested_player_id=pid))
                continue
            except Exception:
                pass
        pending.append(pid)

    print(f"  player profiles: cached={len(ids)-len(pending):,}; pending={len(pending):,}; workers={workers}")

    def _one(pid):
        p = pdir / f"{pid}.csv"
        try:
            _nba_stats_rate_limit()
            try:
                obj = Cls(player_id=pid, timeout=int(globals().get("NBA_STATS_TIMEOUT",45)))
            except TypeError:
                obj = Cls(player_id=pid)
            dfs = obj.get_data_frames()
            df = dfs[0] if dfs else pd.DataFrame()
            df.to_csv(p, index=False)
            return pid, df, None
        except Exception as e:
            (pdir / f"{pid}.FAILED.txt").write_text(str(e))
            return pid, pd.DataFrame(), str(e)

    if pending:
        with ThreadPoolExecutor(max_workers=max(1, workers)) as ex:
            futures = {ex.submit(_one, pid): pid for pid in pending}
            for i, fut in enumerate(as_completed(futures), 1):
                pid, df, err = fut.result()
                if not df.empty:
                    rows.append(df.assign(_requested_player_id=pid))
                if i % 50 == 0 or i == len(pending):
                    print(f"  player profiles pending: {i:,}/{len(pending):,}")

    out = pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()
    out.to_csv(ROOT / f"player_profiles_{season.replace('-','_')}.csv", index=False)
    return out


def export_excel_bundle(tables: dict, filename="NBA_Injury_RTP_Pilot_2024_25.xlsx"):
    """Convenience workbook. Large raw endpoint archives remain as separate files."""
    path = ROOT / filename
    try:
        import xlsxwriter  # noqa: F401
        engine = "xlsxwriter"
    except Exception:
        engine = "openpyxl"
    print(f"Writing Excel bundle with {engine}...")
    with pd.ExcelWriter(path, engine=engine) as writer:
        for name, df in tables.items():
            if df is None or not isinstance(df, pd.DataFrame):
                continue
            # Excel hard row limit. Keep giant tables in CSV but avoid crashing export.
            if len(df) > 1_048_000:
                print(f"Skipping Excel sheet {name}: {len(df):,} rows; retained as CSV.")
                continue
            df.to_excel(writer, sheet_name=re.sub(r"[\\/*?:\[\]]", "_", name)[:31], index=False)
    print("Excel bundle:", path.resolve())
    return path


def build_player_game_master_maximal(season: str, fetch_live: bool = True,
                                     official_reports: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """Core player-game master that preserves ALL public playerlog/live columns."""
    player = normalize_playerlog(get_public_playerlog(season), season)
    sched = normalize_team_schedule(get_public_teamlog(season), season)
    if "game_date" in player.columns:
        player = player.drop(columns=["game_date"])
    d = player.merge(sched, on=["season", "game_id"], how="left", validate="many_to_one")
    d["game_date"] = pd.to_datetime(d["game_date"], errors="coerce").dt.normalize()
    d["is_home"] = (d["team"] == d["home_team"]).astype("Int64")
    d["opponent"] = np.where(d["team"] == d["home_team"], d["away_team"], d["home_team"])

    if fetch_live:
        live_p, live_g = build_live_availability_maximal(d["game_id"].dropna().unique())
        if not live_p.empty:
            live_p["player_id"] = pd.to_numeric(live_p["player_id"], errors="coerce").astype("Int64")
            d = d.merge(live_p, on=["game_id", "team", "player_id"], how="left")
        if not live_g.empty:
            d = d.merge(live_g, on="game_id", how="left")

    if official_reports is not None and len(official_reports):
        rr = collapse_reports_to_player_game(normalize_official_injury_reports(official_reports))
        keep = [c for c in [
            "game_date", "team", "player_key", "report_status", "report_reason",
            "report_reason_family", "injury_body_part", "injury_laterality",
            "injury_diagnosis_class", "injury_region", "report_ts"
        ] if c in rr.columns]
        d = d.merge(rr[keep], on=["game_date", "team", "player_key"], how="left")

    if "live_played" in d.columns:
        d["played_actual"] = np.where(d["live_played"].notna(), d["live_played"], d["played"]).astype(bool)
    else:
        d["played_actual"] = d["played"].astype(bool)

    live_reason = d.get("not_playing_reason", pd.Series("", index=d.index))
    live_desc = d.get("not_playing_description", pd.Series("", index=d.index))
    report_reason = d.get("report_reason", pd.Series("", index=d.index))
    comment = d.get("comment", pd.Series("", index=d.index))
    d["absence_reason_family"] = [reason_family(a,b,c,e) for a,b,c,e in zip(live_reason, live_desc, report_reason, comment)]
    d.loc[d["played_actual"], "absence_reason_family"] = "played"

    for c in ["injury_body_part", "injury_laterality", "injury_diagnosis_class", "injury_region"]:
        if c not in d.columns:
            d[c] = "unknown"
    fallback_text = live_desc.fillna("").astype(str) + " " + comment.fillna("").astype(str)
    fb = pd.DataFrame([parse_injury_text(x) for x in fallback_text])
    for c in ["injury_body_part", "injury_laterality", "injury_diagnosis_class", "injury_region"]:
        missing = d[c].isna() | d[c].astype(str).str.lower().isin(["", "nan", "unknown"])
        d.loc[missing, c] = fb.loc[missing, c].values

    # Keep player-appearance context from the original helper, then add TEAM schedule context.
    d = add_schedule_context(d)
    d = add_team_schedule_context(d)
    d = add_player_workload_rolling_features(d)
    return d.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)


def run_maximal_pilot_2024_25(fetch_live=True, min_games_missed=2,
                              make_excel=True, deep_mode="off",
                              deep_scope="episode_windows"):
    """Recommended one-season pilot entry point.

    deep_mode: off | focused | maximal
    deep_scope: episode_windows | full_season
    """
    season = "2024-25"
    reports = get_2425_official_injury_designations()
    print(f"Official injury designation rows: {len(reports):,}")

    pg = build_player_game_master_maximal(season, fetch_live=fetch_live, official_reports=reports)
    pg_path = ROOT / "player_game_master_2024_25_MAXIMAL.csv"
    pg.to_csv(pg_path, index=False)
    print(f"Player-game rows: {len(pg):,} x {pg.shape[1]:,} columns -> {pg_path}")

    injury_ep = build_injury_episodes(pg, min_games_missed=min_games_missed, pre_games=10, post_games=20)
    injury_ep = add_player_age(injury_ep, season)
    injury_ep.to_csv(ROOT / "rtp_injury_episodes_2024_25.csv", index=False)

    all_abs = build_all_absence_episodes(pg, min_team_games_missed=1)
    all_abs.to_csv(ROOT / "all_absence_episodes_2024_25.csv", index=False)

    windows = build_episode_game_windows(pg, injury_ep, pre_played_games=10, post_team_games=20)
    windows.to_csv(ROOT / "rtp_episode_game_windows_2024_25.csv", index=False)

    profiles = pd.DataFrame()
    if deep_mode in {"focused", "maximal"}:
        try:
            profiles = player_profile_archive(pg["player_id"].dropna().unique(), season=season)
        except Exception as e:
            print("Player profile archive skipped:", e)

        endpoint_names = FOCUSED_DEEP_ENDPOINTS if deep_mode == "focused" else MAXIMAL_DEEP_ENDPOINTS
        gids = episode_window_game_ids(windows) if deep_scope == "episode_windows" else pg["game_id"].dropna().astype(str).unique().tolist()
        print(f"Deep archive target: {len(gids):,} games x {len(endpoint_names)} endpoint families")
        deep_audit = archive_deep_endpoints(gids, endpoint_names)
    else:
        deep_audit = pd.DataFrame()

    if make_excel:
        export_excel_bundle({
            "player_game_master": pg,
            "injury_episodes": injury_ep,
            "all_absence_episodes": all_abs,
            "rtp_game_windows": windows,
            "player_profiles": profiles,
            "deep_archive_audit": deep_audit,
        })

    audit = {
        "season": season,
        "player_game_rows": int(len(pg)),
        "player_game_columns": int(pg.shape[1]),
        "injury_episode_rows": int(len(injury_ep)),
        "all_absence_episode_rows": int(len(all_abs)),
        "episode_window_rows": int(len(windows)),
        "deep_mode": deep_mode,
        "deep_scope": deep_scope,
        "injury_absence_rows": int((pg["absence_reason_family"] == "injury").sum()),
        "unknown_absence_rows": int(((~pg["played_actual"]) & (pg["absence_reason_family"] == "unknown")).sum()),
    }
    (ROOT / "maximal_pilot_audit_2024_25.json").write_text(json.dumps(audit, indent=2))
    print(json.dumps(audit, indent=2))
    return pg, injury_ep, all_abs, windows, profiles, deep_audit


## 4B. Live-CDN reliability patch (v2)

This override is intentionally placed after the core/maximal function definitions. It probes the NBA live endpoint before bulk requests, uses browser-like headers and conservative parallelism when available, stops early rather than issuing hundreds of doomed requests when the CDN is blocked, and makes downstream datetime/B2B/travel calculations safe when live data are missing.


In [ ]:
"""
NBA RTP live-data reliability patch v2
-------------------------------------
Drop-in overrides for NBA_Injury_RTP_Pilot_FAST.ipynb.

Goals:
1) Do NOT blast cdn.nba.com if the runtime is receiving 403/Access Denied.
2) Use browser-like headers + conservative parallelism when live access works.
3) Preserve cache-first/resume behavior.
4) Make all downstream datetime calculations safe when live data are empty/partial.
5) Fall back to team-home-city travel context when arena metadata are unavailable.

Run this AFTER the notebook's core/maximal function-definition cells and BEFORE
calling run_maximal_pilot_2024_25().
"""

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import requests

# Conservative defaults. The prior 16-worker burst can trigger NBA CDN WAF rules.
LIVE_WORKERS = 4
LIVE_MIN_START_INTERVAL = 0.20   # max ~5 request starts/sec across all workers
LIVE_CONNECT_TIMEOUT = 8
LIVE_READ_TIMEOUT = 30
LIVE_RETRIES = 3
LIVE_BATCH_SIZE = 60
LIVE_ABORT_FAILURE_RATE = 0.40

LIVE_BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Origin": "https://www.nba.com",
    "Referer": "https://www.nba.com/",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-site",
    "Connection": "keep-alive",
}

_LIVE_RATE_LOCK = threading.Lock()
_LIVE_NEXT_START = 0.0


def _wait_for_live_slot():
    """Global request-start limiter shared by all worker threads."""
    global _LIVE_NEXT_START
    interval = float(globals().get("LIVE_MIN_START_INTERVAL", 0.20))
    with _LIVE_RATE_LOCK:
        now = time.monotonic()
        slot = max(now, _LIVE_NEXT_START)
        _LIVE_NEXT_START = slot + interval
        wait = slot - now
    if wait > 0:
        time.sleep(wait)


def _live_cache_path(game_id):
    gid = str(game_id).replace(".0", "").zfill(10)
    return CACHE / "live_boxscore" / f"{gid}.json"


def _load_cached_live(game_id):
    p = _live_cache_path(game_id)
    if not p.exists() or p.stat().st_size <= 20:
        return None
    try:
        return json.loads(p.read_text())
    except Exception:
        try:
            p.unlink()
        except Exception:
            pass
        return None


def _atomic_live_write(game_id, obj):
    p = _live_cache_path(game_id)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + f".{threading.get_ident()}.tmp")
    tmp.write_text(json.dumps(obj))
    tmp.replace(p)


def _http_error_message(resp, gid):
    ctype = resp.headers.get("content-type", "")
    body = (resp.text or "")[:240].replace("\n", " ").replace("\r", " ")
    return f"HTTP {resp.status_code} game={gid} content-type={ctype!r} body={body!r}"


def fetch_live_boxscore(game_id: str, use_cache: bool = True, retries=None, timeout=None) -> dict:
    """Reliable cache-first live boxscore fetch with visible HTTP errors.

    Uses direct requests.get() rather than a shared pooled session. This is a bit
    less theoretically efficient, but more robust across Colab/Jupyter runtimes.
    """
    gid = str(game_id).replace(".0", "").zfill(10)
    if use_cache:
        cached = _load_cached_live(gid)
        if cached is not None:
            return cached

    retries = int(globals().get("LIVE_RETRIES", 3) if retries is None else retries)
    timeout = timeout or (
        float(globals().get("LIVE_CONNECT_TIMEOUT", 8)),
        float(globals().get("LIVE_READ_TIMEOUT", 30)),
    )
    url = NBA_LIVE_BOXSCORE.format(game_id=gid)
    last = None

    for attempt in range(max(1, retries)):
        _wait_for_live_slot()
        try:
            resp = requests.get(url, headers=LIVE_BROWSER_HEADERS, timeout=timeout)
            if resp.status_code != 200:
                msg = _http_error_message(resp, gid)
                last = RuntimeError(msg)
                # 403/429 are WAF/rate-limit signals: give the server real breathing room.
                if resp.status_code in (403, 429):
                    if attempt + 1 < retries:
                        time.sleep(2.5 * (attempt + 1))
                    continue
                resp.raise_for_status()
            ctype = (resp.headers.get("content-type") or "").lower()
            if "json" not in ctype and not resp.text.lstrip().startswith("{"):
                raise RuntimeError(_http_error_message(resp, gid))
            js = resp.json()
            if not isinstance(js, dict) or "game" not in js:
                raise RuntimeError(f"Unexpected live JSON structure for {gid}")
            _atomic_live_write(gid, js)
            return js
        except Exception as exc:
            last = exc
            if attempt + 1 < retries and "HTTP 403" not in str(exc):
                time.sleep(0.8 * (2 ** attempt))

    raise RuntimeError(f"live boxscore failed {gid}: {last}")


def diagnose_live_endpoint(game_id="0022400001"):
    """One-request health check that exposes HTTP status/body instead of hiding it."""
    gid = str(game_id).zfill(10)
    url = NBA_LIVE_BOXSCORE.format(game_id=gid)
    try:
        _wait_for_live_slot()
        r = requests.get(
            url,
            headers=LIVE_BROWSER_HEADERS,
            timeout=(LIVE_CONNECT_TIMEOUT, LIVE_READ_TIMEOUT),
        )
        print(f"NBA live probe {gid}: HTTP {r.status_code}; content-type={r.headers.get('content-type')}")
        if r.status_code != 200:
            print("Response preview:", (r.text or "")[:300].replace("\n", " "))
        return r.status_code
    except Exception as e:
        print(f"NBA live probe {gid}: request exception: {e}")
        return None


def _flatten_one_live(gid, js):
    p, g = flatten_live_boxscore_maximal(js)
    return p, g


def build_live_availability_maximal(game_ids, use_cache=True, workers=None):
    """Adaptive live collector.

    Behavior:
    - Parse cached games first.
    - Probe at most two uncached games sequentially.
    - If the CDN is blocked, STOP immediately and return cached/empty live data.
    - If probe succeeds, fetch in modest parallel batches.
    - Abort live enrichment if a batch suddenly has a high failure rate.
    - Never prevents the rest of the RTP pilot from running.
    """
    gids = list(dict.fromkeys(str(x).replace(".0", "").zfill(10) for x in game_ids if pd.notna(x)))
    workers = int(workers or globals().get("LIVE_WORKERS", 4))
    workers = max(1, min(workers, 6))  # hard safety ceiling

    players, games, failures = [], [], []
    cache_hits = 0
    uncached = []

    # Load existing cache without network.
    for gid in gids:
        js = _load_cached_live(gid) if use_cache else None
        if js is None:
            uncached.append(gid)
            continue
        try:
            p, g = _flatten_one_live(gid, js)
            players.append(p); games.append(g); cache_hits += 1
        except Exception as e:
            failures.append({"game_id": gid, "stage": "cached_parse", "error": str(e)})

    print(f"  maximal live boxscores: {cache_hits:,}/{len(gids):,} usable from cache; "
          f"{len(uncached):,} need network")

    if not uncached:
        pd.DataFrame(failures).to_csv(ROOT / "live_boxscore_failures_maximal.csv", index=False)
        return (
            pd.concat(players, ignore_index=True, sort=False) if players else pd.DataFrame(),
            pd.DataFrame(games),
        )

    # Probe one or two distinct games. A 404 may be game-specific; a 403 generally is not.
    probe_success = False
    probe_consumed = []
    first_errors = []
    for gid in uncached[:2]:
        try:
            js = fetch_live_boxscore(gid, use_cache=use_cache)
            p, g = _flatten_one_live(gid, js)
            players.append(p); games.append(g)
            probe_consumed.append(gid)
            probe_success = True
            print(f"  live probe succeeded on {gid}; proceeding with {workers} workers")
            break
        except Exception as e:
            err = str(e)
            first_errors.append((gid, err))
            failures.append({"game_id": gid, "stage": "probe", "error": err})
            probe_consumed.append(gid)
            print(f"  live probe failed on {gid}: {err[:300]}")
            if "HTTP 403" in err or "Access Denied" in err:
                break

    remaining = [g for g in uncached if g not in set(probe_consumed)]

    if not probe_success:
        print("\n  WARNING: NBA live CDN is not usable from this runtime right now.")
        print("  Live enrichment will be SKIPPED for this run; core player logs and official")
        print("  injury-report data will still be processed. This prevents 1,300 doomed calls.")
        for gid in remaining:
            failures.append({"game_id": gid, "stage": "skipped_after_probe_failure",
                             "error": "live CDN unavailable from runtime"})
        pd.DataFrame(failures).to_csv(ROOT / "live_boxscore_failures_maximal.csv", index=False)
        return (
            pd.concat(players, ignore_index=True, sort=False) if players else pd.DataFrame(),
            pd.DataFrame(games),
        )

    downloaded = 1
    completed_network = 1
    # Batches allow us to stop adapting if the CDN starts rejecting requests.
    batch_size = int(globals().get("LIVE_BATCH_SIZE", 60))
    abort_rate = float(globals().get("LIVE_ABORT_FAILURE_RATE", 0.40))

    for start in range(0, len(remaining), batch_size):
        batch = remaining[start:start + batch_size]
        batch_fail = 0
        batch_success = 0

        def _one(gid):
            js = fetch_live_boxscore(gid, use_cache=use_cache)
            p, g = _flatten_one_live(gid, js)
            return gid, p, g

        with ThreadPoolExecutor(max_workers=workers) as ex:
            futs = {ex.submit(_one, gid): gid for gid in batch}
            for fut in as_completed(futs):
                gid = futs[fut]
                try:
                    _, p, g = fut.result()
                    players.append(p); games.append(g)
                    downloaded += 1; batch_success += 1
                except Exception as e:
                    failures.append({"game_id": gid, "stage": "parallel", "error": str(e)})
                    batch_fail += 1
                completed_network += 1

        total_done = cache_hits + completed_network
        print(f"  maximal live boxscores: {min(total_done, len(gids)):,}/{len(gids):,} | "
              f"cache={cache_hits:,} | downloaded={downloaded:,} | failed={len([x for x in failures if x['stage']!='skipped_after_probe_failure']):,}")

        batch_n = batch_success + batch_fail
        if batch_n >= 10 and batch_fail / batch_n >= abort_rate:
            examples = [x["error"] for x in failures if x["stage"] == "parallel"][-3:]
            print("\n  WARNING: live failure rate spiked; stopping live network requests early.")
            for exmsg in examples:
                print("   ", exmsg[:260])
            not_attempted = remaining[start + len(batch):]
            for gid in not_attempted:
                failures.append({"game_id": gid, "stage": "skipped_after_high_failure_rate",
                                 "error": "adaptive live downloader stopped early"})
            break

    pd.DataFrame(failures).to_csv(ROOT / "live_boxscore_failures_maximal.csv", index=False)
    print(f"  live stage complete: cache={cache_hits:,}, downloaded={downloaded:,}, "
          f"failure/skipped rows={len(failures):,}")
    return (
        pd.concat(players, ignore_index=True, sort=False) if players else pd.DataFrame(),
        pd.DataFrame(games),
    )


# Fallback venue city mapping lets travel proxies survive even without live arena metadata.
TEAM_HOME_CITY = {
    "ATL": "Atlanta", "BOS": "Boston", "BKN": "Brooklyn", "CHA": "Charlotte",
    "CHI": "Chicago", "CLE": "Cleveland", "DAL": "Dallas", "DEN": "Denver",
    "DET": "Detroit", "GSW": "San Francisco", "HOU": "Houston", "IND": "Indianapolis",
    "LAC": "Los Angeles", "LAL": "Los Angeles", "MEM": "Memphis", "MIA": "Miami",
    "MIL": "Milwaukee", "MIN": "Minneapolis", "NOP": "New Orleans", "NYK": "New York",
    "OKC": "Oklahoma City", "ORL": "Orlando", "PHI": "Philadelphia", "PHX": "Phoenix",
    "POR": "Portland", "SAC": "Sacramento", "SAS": "San Antonio", "TOR": "Toronto",
    "UTA": "Salt Lake City", "WAS": "Washington",
}


def _safe_utc_series(df, col):
    if col in df.columns:
        return pd.to_datetime(df[col], errors="coerce", utc=True)
    return pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")


def _safe_date_series(df, col):
    if col in df.columns:
        return pd.to_datetime(df[col], errors="coerce").dt.normalize()
    return pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")


def add_schedule_context(pg: pd.DataFrame) -> pd.DataFrame:
    """Datetime-safe version; works even when every live request failed."""
    d = pg.copy()
    d["game_date"] = _safe_date_series(d, "game_date")
    d = d.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)

    d["game_time_utc"] = _safe_utc_series(d, "game_time_utc")
    d["live_data_available"] = d["game_time_utc"].notna()
    d["prev_game_time_utc"] = d.groupby("player_id")["game_time_utc"].shift(1)
    delta = d["game_time_utc"] - d["prev_game_time_utc"]
    d["rest_hours"] = delta.dt.total_seconds() / 3600.0

    prev_date = d.groupby("player_id")["game_date"].shift(1)
    date_delta = d["game_date"] - prev_date
    d["calendar_days_since_prev"] = date_delta.dt.days
    d["is_back_to_back_calendar"] = (d["calendar_days_since_prev"] == 1).astype("Int64")
    d["short_rest_lt36h"] = pd.Series(pd.NA, index=d.index, dtype="boolean")
    mask = d["rest_hours"].notna()
    d.loc[mask, "short_rest_lt36h"] = d.loc[mask, "rest_hours"] < 36

    if "arena_city" not in d.columns:
        d["arena_city"] = pd.NA
    inferred_city = d.get("home_team", pd.Series(index=d.index, dtype=object)).map(TEAM_HOME_CITY)
    d["arena_city"] = d["arena_city"].where(d["arena_city"].notna() & (d["arena_city"].astype(str).str.len() > 0), inferred_city)
    d["arena_city_source"] = np.where(d["live_data_available"], "live_boxscore", "home_team_inference")

    city = d["arena_city"].fillna("").astype(str).str.lower()
    coords = city.map(CITY_COORDS)
    d["arena_lat"] = coords.map(lambda x: x[0] if isinstance(x, tuple) else np.nan)
    d["arena_lon"] = coords.map(lambda x: x[1] if isinstance(x, tuple) else np.nan)
    plat = d.groupby("player_id")["arena_lat"].shift(1)
    plon = d.groupby("player_id")["arena_lon"].shift(1)
    d["travel_km_since_prev"] = [
        haversine_km(a, b, c, e) if all(pd.notna(x) for x in [a, b, c, e]) else np.nan
        for a, b, c, e in zip(plat, plon, d["arena_lat"], d["arena_lon"])
    ]
    return d


def add_team_schedule_context(player_games: pd.DataFrame) -> pd.DataFrame:
    """Datetime-safe team schedule context independent of NBA live availability."""
    d = player_games.copy()
    d["game_date"] = _safe_date_series(d, "game_date")
    d["game_time_utc"] = _safe_utc_series(d, "game_time_utc")

    base_cols = [c for c in ["season", "game_id", "game_date", "home_team", "away_team", "game_time_utc"] if c in d.columns]
    games0 = d[base_cols].drop_duplicates(["game_id"]).copy()
    rows = []
    for r in games0.itertuples(index=False):
        dd = r._asdict()
        for side in ("home_team", "away_team"):
            rows.append({
                "season": dd.get("season"),
                "game_id": str(dd.get("game_id")),
                "team": dd.get(side),
                "game_date": dd.get("game_date"),
                "team_game_time_utc": dd.get("game_time_utc"),
                "team_is_home": int(side == "home_team"),
            })
    ts = pd.DataFrame(rows)
    if ts.empty:
        return d

    ts["game_date"] = pd.to_datetime(ts["game_date"], errors="coerce").dt.normalize()
    ts["team_game_time_utc"] = pd.to_datetime(ts["team_game_time_utc"], errors="coerce", utc=True)
    ts = ts.sort_values(["team", "game_date", "game_id"]).reset_index(drop=True)

    ts["team_prev_game_date"] = ts.groupby("team")["game_date"].shift(1)
    ts["team_calendar_rest_days"] = (ts["game_date"] - ts["team_prev_game_date"]).dt.days
    ts["team_is_b2b"] = (ts["team_calendar_rest_days"] == 1).astype("Int64")
    ts["team_prev_game_time_utc"] = ts.groupby("team")["team_game_time_utc"].shift(1)
    ts["team_rest_hours"] = (
        ts["team_game_time_utc"] - ts["team_prev_game_time_utc"]
    ).dt.total_seconds() / 3600.0

    # Correct schedule-density calculation, group-local and index-aligned.
    for window in [3, 4, 6, 7, 14, 28]:
        ts[f"team_games_last_{window}d"] = 0
        for team, idx in ts.groupby("team", sort=False).groups.items():
            idx = list(idx)
            dates = ts.loc[idx, "game_date"].to_numpy(dtype="datetime64[ns]")
            vals = []
            for j, dt in enumerate(dates):
                if np.isnat(dt):
                    vals.append(0); continue
                lo = dt - np.timedelta64(window - 1, "D")
                vals.append(int(np.sum((dates[:j+1] >= lo) & (dates[:j+1] <= dt))))
            ts.loc[idx, f"team_games_last_{window}d"] = vals

    ts["team_3in4"] = (ts["team_games_last_4d"] >= 3).astype("Int64")
    ts["team_4in6"] = (ts["team_games_last_6d"] >= 4).astype("Int64")
    ts["team_5in7"] = (ts["team_games_last_7d"] >= 5).astype("Int64")

    keep = [c for c in ts.columns if c != "game_date"]
    d["game_id"] = d["game_id"].astype(str)
    d = d.merge(ts[keep], on=["season", "game_id", "team"], how="left")

    d = d.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    played = d.get("played_actual", pd.Series(False, index=d.index)).fillna(False).astype(bool)
    last_played_time = []
    state = {}
    for i, r in d.iterrows():
        pid = r.get("player_id")
        current = r.get("game_time_utc", pd.NaT)
        current = pd.to_datetime(current, errors="coerce", utc=True)
        prev = state.get(pid, pd.NaT)
        last_played_time.append(prev)
        if played.iloc[i] and pd.notna(current):
            state[pid] = current

    d["previous_player_appearance_time_utc"] = pd.to_datetime(
        pd.Series(last_played_time, index=d.index), errors="coerce", utc=True
    )
    d["hours_since_last_player_appearance"] = (
        d["game_time_utc"] - d["previous_player_appearance_time_utc"]
    ).dt.total_seconds() / 3600.0
    return d


print("NBA RTP live-data fix v2 loaded.")
print(f"  LIVE_WORKERS={LIVE_WORKERS}; start interval={LIVE_MIN_START_INTERVAL}s")
print("  The pipeline will now stop early if the NBA live CDN is blocked and continue without live enrichment.")


## 4C. Schedule/date reliability patch (v3)

This patch makes schedule parsing more tolerant and quarantines residual rows with invalid game dates from chronology-dependent episode analyses while retaining them in the raw player-game master.


In [ ]:
"""
NBA RTP date/schedule reliability patch v3
------------------------------------------
Run AFTER the notebook's function-definition cells (including LIVE_FIX v2) and
BEFORE calling run_maximal_pilot_2024_25().

Fixes:
1) More robust team-schedule normalization, including one-sided/ambiguous rows.
2) Explicit schedule coverage audit.
3) Episode builders quarantine residual rows with missing/invalid game dates.
4) Prevents NaT/NaN -> int crashes in injury and all-absence episodes.

This patch is intentionally compatible with NBA_Injury_RTP_Pilot_FAST_v2.ipynb.
"""

import re
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1) Resilient team schedule normalization
# ---------------------------------------------------------------------------

def _parse_matchup_home_away(team, matchup):
    """Infer home/away abbreviations from strings like 'BOS vs. NYK' or 'BOS @ NYK'."""
    team = "" if pd.isna(team) else str(team).strip().upper()
    s = "" if pd.isna(matchup) else str(matchup).strip().upper()
    s = re.sub(r"\s+", " ", s)

    # Most NBA logs use 'AAA vs. BBB' or 'AAA @ BBB'.
    m = re.search(r"\b([A-Z]{2,3})\s+(VS\.?|@)\s+([A-Z]{2,3})\b", s)
    if m:
        left, sep, right = m.group(1), m.group(2), m.group(3)
        if sep.startswith("VS"):
            return left, right
        return right, left

    # Fallback when matchup formatting is odd but the row's own team is known.
    toks = re.findall(r"\b[A-Z]{2,3}\b", s)
    toks = [x for x in toks if x not in {"VS"}]
    if team and len(toks) >= 2:
        other = next((x for x in toks if x != team), None)
        if other:
            if "@" in s:
                return other, team
            if "VS" in s:
                return team, other
    return None, None


def normalize_team_schedule(df: pd.DataFrame, season: str) -> pd.DataFrame:
    """Robust one-row-per-game schedule builder.

    Unlike the original version, this does NOT discard a game simply because one
    of the two team rows is missing or matchup parsing is imperfect. It preserves
    any valid game date and infers home/away from the matchup string when needed.
    """
    d = df.copy()
    ren = {}
    for c in d.columns:
        lc = str(c).lower()
        if lc == "game_id": ren[c] = "game_id"
        elif lc == "game_date": ren[c] = "game_date"
        elif lc == "team_abbreviation": ren[c] = "team"
        elif lc == "matchup": ren[c] = "matchup"
    d = d.rename(columns=ren)
    needed = {"game_id", "game_date", "team", "matchup"}
    if not needed.issubset(d.columns):
        raise ValueError(f"team log lacks {needed - set(d.columns)}")

    d["game_id"] = d["game_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(10)
    d["game_date"] = pd.to_datetime(d["game_date"], errors="coerce").dt.normalize()
    d["team"] = d["team"].astype(str).str.upper().str.strip()
    d["season"] = season

    rows = []
    inferred = 0
    unresolved_matchup = 0
    for gid, g in d.groupby("game_id", sort=False):
        dates = g["game_date"].dropna()
        game_date = dates.iloc[0] if len(dates) else pd.NaT

        home_team = away_team = None

        # First use explicit vs/@ row evidence.
        home_rows = g[g["matchup"].astype(str).str.contains(r"\bvs\.?\b", case=False, regex=True)]
        away_rows = g[g["matchup"].astype(str).str.contains("@", regex=False)]
        if len(home_rows):
            home_team = str(home_rows.iloc[0]["team"]).upper()
        if len(away_rows):
            away_team = str(away_rows.iloc[0]["team"]).upper()

        # If either side is missing, parse each matchup string directly.
        if not home_team or not away_team:
            for rr in g.itertuples(index=False):
                h, a = _parse_matchup_home_away(getattr(rr, "team", None), getattr(rr, "matchup", None))
                if not home_team and h: home_team = h
                if not away_team and a: away_team = a
                if home_team and away_team:
                    inferred += 1
                    break

        # If exactly two distinct teams exist, fill whichever side is still missing.
        teams = [x for x in g["team"].dropna().astype(str).str.upper().unique().tolist() if x and x != "NAN"]
        if len(teams) == 2:
            if home_team and not away_team:
                away_team = next((x for x in teams if x != home_team), None)
            elif away_team and not home_team:
                home_team = next((x for x in teams if x != away_team), None)

        if not home_team or not away_team:
            unresolved_matchup += 1

        rows.append({
            "season": season,
            "game_id": gid,
            "game_date": game_date,
            "home_team": home_team,
            "away_team": away_team,
            "schedule_matchup_resolved": int(bool(home_team and away_team)),
        })

    out = pd.DataFrame(rows)
    missing_dates = int(out["game_date"].isna().sum()) if len(out) else 0
    print(
        f"  schedule normalization: {len(out):,} games; "
        f"missing_dates={missing_dates:,}; unresolved_matchups={unresolved_matchup:,}"
    )
    return out


# ---------------------------------------------------------------------------
# 2) Date sanitation helpers
# ---------------------------------------------------------------------------

def _sanitize_player_game_dates(pg: pd.DataFrame, label="episode builder"):
    d = pg.copy()
    if "game_date" not in d.columns:
        raise ValueError("player-game table has no game_date column")
    d["game_date"] = pd.to_datetime(d["game_date"], errors="coerce").dt.normalize()
    bad = d[d["game_date"].isna()].copy()
    if len(bad):
        audit_cols = [c for c in [
            "season", "game_id", "player_id", "player_name", "team", "home_team",
            "away_team", "minutes", "played_actual", "absence_reason_family", "comment"
        ] if c in bad.columns]
        audit_path = ROOT / "rows_with_missing_game_date.csv"
        bad[audit_cols].to_csv(audit_path, index=False)
        print(
            f"  WARNING [{label}]: quarantining {len(bad):,} player-game rows "
            f"({bad['game_id'].nunique() if 'game_id' in bad.columns else 0:,} games) with invalid game_date."
        )
        print(f"  Missing-date audit -> {audit_path}")
        d = d[d["game_date"].notna()].copy()
    return d


# Capture the currently-loaded, feature-complete builders before overriding them.
_build_injury_episodes_before_date_fix = build_injury_episodes
_build_all_absence_episodes_before_date_fix = build_all_absence_episodes
_build_episode_game_windows_before_date_fix = build_episode_game_windows
_build_player_game_master_maximal_before_date_fix = build_player_game_master_maximal


def build_injury_episodes(pg: pd.DataFrame, min_games_missed: int = 2,
                          pre_games: int = 10, post_games: int = 10) -> pd.DataFrame:
    """Date-safe wrapper around the existing injury-episode implementation."""
    d = _sanitize_player_game_dates(pg, "injury episodes")
    return _build_injury_episodes_before_date_fix(
        d, min_games_missed=min_games_missed, pre_games=pre_games, post_games=post_games
    )


def build_all_absence_episodes(pg: pd.DataFrame, min_team_games_missed: int = 1) -> pd.DataFrame:
    """Date-safe wrapper around the existing all-absence episode implementation."""
    d = _sanitize_player_game_dates(pg, "all absence episodes")
    return _build_all_absence_episodes_before_date_fix(
        d, min_team_games_missed=min_team_games_missed
    )


def build_episode_game_windows(pg: pd.DataFrame, injury_episodes: pd.DataFrame,
                               pre_played_games=10, post_team_games=20) -> pd.DataFrame:
    """Date-safe wrapper around episode-relative longitudinal windows."""
    d = _sanitize_player_game_dates(pg, "episode windows")
    if injury_episodes is not None and len(injury_episodes):
        injury_episodes = injury_episodes.copy()
        injury_episodes["return_game_date"] = pd.to_datetime(
            injury_episodes["return_game_date"], errors="coerce"
        ).dt.normalize()
        injury_episodes = injury_episodes[injury_episodes["return_game_date"].notna()].copy()
    return _build_episode_game_windows_before_date_fix(
        d, injury_episodes,
        pre_played_games=pre_played_games, post_team_games=post_team_games
    )


def build_player_game_master_maximal(season: str, fetch_live: bool = True,
                                     official_reports=None) -> pd.DataFrame:
    """Wrapper that audits residual schedule/date coverage after the maximal master build."""
    d = _build_player_game_master_maximal_before_date_fix(
        season, fetch_live=fetch_live, official_reports=official_reports
    )
    d["game_date"] = pd.to_datetime(d["game_date"], errors="coerce").dt.normalize()
    bad = d[d["game_date"].isna()].copy()
    if len(bad):
        cols = [c for c in [
            "season", "game_id", "player_id", "player_name", "team", "home_team",
            "away_team", "minutes", "comment"
        ] if c in bad.columns]
        path = ROOT / "player_game_missing_schedule_dates.csv"
        bad[cols].to_csv(path, index=False)
        print(
            f"  NOTE: master contains {len(bad):,} rows across "
            f"{bad['game_id'].nunique():,} games without a usable date; "
            f"they are retained in the RAW master but excluded from episode analyses."
        )
        print(f"  Schedule-date audit -> {path}")
    return d


print("NBA RTP date/schedule fix v3 loaded.")
print("  - schedule parser is more tolerant")
print("  - residual missing-date rows are audited and quarantined from episode analyses")
print("  - NaT/NaN -> int episode crashes are prevented")


## 4D. Quality-control / cohort-definition patch (v4)

This patch fixes the official report join rather than merely auditing it. It forward-fills PDF block fields, preserves the last substantive reason even when a later snapshot becomes `Available` / `-`, supplements postseason schedule dates from official report matchups, records reason provenance, preserves raw first appearances while defining a ≥5-minute meaningful RTP anchor, and creates manuscript-oriented QC tables.


In [ ]:
# NBA RTP QUALITY / COHORT FIX v4
# Load AFTER the v3 notebook definitions/patches and BEFORE Section 6.
#
# Fixes:
#   1) official injury-report forward-fill + robust team normalization
#   2) preserve substantive injury reason even when latest snapshot is Available / '-'
#   3) build postseason schedule dates from official report matchups when teamlog is regular-season-only
#   4) primary absence classification no longer trusts generic "DNP - Coach's Decision" comments over official reports
#   5) meaningful RTP + rotation-player eligibility flags without deleting raw episodes
#   6) rename/rebuild outcomes as subsequent-injury-absence proxies, with stricter same-site definitions
#   7) QC/audit tables written to CSV and Excel

import re, json, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# Configuration — deliberately explicit so sensitivity analyses are easy later.
# -----------------------------------------------------------------------------
MEANINGFUL_RTP_MINUTES = 5.0
PRIMARY_BASELINE_MINUTES = 10.0
STRICT_BASELINE_MINUTES = 15.0
MIN_PRE_GAMES_PRIMARY = 5
MIN_HEALTHY_PLAYED_GAMES_FOR_SEPARATED_EVENT = 3

GAME_TYPE_LABELS = {
    "001": "preseason",
    "002": "regular_season",
    "003": "all_star",
    "004": "playoffs",
    "005": "play_in",
    "006": "nba_cup_final",
}

# -----------------------------------------------------------------------------
# 1) Robust team / placeholder normalization
# -----------------------------------------------------------------------------
def _clean_text_or_nan(s):
    if s is None:
        return np.nan
    x = str(s).strip()
    if x.lower() in {"", "nan", "none", "null"}:
        return np.nan
    return x


def _team_token(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    x = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "", x.lower())


_TEAM_TOKEN_TO_ABBR = {}
for _full, _abbr in TEAM_NAME_TO_ABBR.items():
    _TEAM_TOKEN_TO_ABBR[_team_token(_full)] = _abbr
    _TEAM_TOKEN_TO_ABBR[_team_token(_abbr)] = _abbr
# Common compact/alternate spellings seen in parsed tables.
_TEAM_TOKEN_TO_ABBR.update({
    "lac": "LAC", "losangelesclippers": "LAC", "laclippers": "LAC",
    "lal": "LAL", "losangeleslakers": "LAL",
    "nyk": "NYK", "newyorkknicks": "NYK",
    "bkn": "BKN", "brooklynnets": "BKN",
    "nop": "NOP", "neworleanspelicans": "NOP",
    "okc": "OKC", "oklahomacitythunder": "OKC",
    "phi": "PHI", "philadelphia76ers": "PHI", "philadelphiaseventysixers": "PHI",
    "por": "POR", "portlandtrailblazers": "POR",
    "sac": "SAC", "sacramentokings": "SAC",
    "sas": "SAS", "sanantoniospurs": "SAS",
    "gsw": "GSW", "goldenstatewarriors": "GSW",
})


def normalize_team_abbr(s):
    tok = _team_token(s)
    return _TEAM_TOKEN_TO_ABBR.get(tok, np.nan)


def _is_placeholder_reason(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return True
    x = re.sub(r"\s+", " ", str(s)).strip().lower()
    return x in {"", "-", "--", "n/a", "na", "none", "nan", "not yet submitted"}

# -----------------------------------------------------------------------------
# 2) Correct official-report normalization.
#    IMPORTANT: forward-fill block-level fields BEFORE dropping missing rows.
# -----------------------------------------------------------------------------
def normalize_official_injury_reports(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    col = lambda *xs: _first_col(d, xs)
    c_date = col("Game Date", "game_date")
    c_match = col("Matchup", "matchup", "Game")
    c_team = col("Team", "team")
    c_player = col("Player Name", "player_name", "player", "Player")
    c_status = col("Current Status", "status", "current_status", "Status")
    c_reason = col("Reason", "reason", "Injury/Condition")
    if not all([c_date, c_team, c_player, c_reason]):
        raise ValueError(f"could not identify official injury report columns: {list(d.columns)}")

    # The source is a parsed PDF table. Game Date / Matchup / Team can appear only
    # on the first row of a visual block; blank cells below belong to that block.
    date_raw = d[c_date].map(_clean_text_or_nan).ffill()
    match_raw = d[c_match].map(_clean_text_or_nan).ffill() if c_match else pd.Series(np.nan, index=d.index)
    team_raw = d[c_team].map(_clean_text_or_nan).ffill()
    player_raw = d[c_player].map(_clean_text_or_nan)
    status_raw = d[c_status].map(_clean_text_or_nan) if c_status else pd.Series(np.nan, index=d.index)
    reason_raw = d[c_reason].map(_clean_text_or_nan)

    out = pd.DataFrame({
        "game_date": pd.to_datetime(date_raw, errors="coerce").dt.normalize(),
        "matchup": match_raw.fillna("").astype(str).str.strip(),
        "team_full": team_raw.fillna("").astype(str).str.strip(),
        "player_name_report": player_raw.fillna("").astype(str).str.strip(),
        "report_status": status_raw.fillna("").astype(str).str.strip(),
        "report_reason": reason_raw.fillna("").astype(str).str.strip(),
        "_source_row": np.arange(len(d), dtype=int),
    })
    out["team"] = out["team_full"].map(normalize_team_abbr)
    out["player_key"] = out["player_name_report"].map(norm_name)

    # Preserve a true report timestamp when available. Source filename parsing is
    # deliberately permissive because archive schemas vary.
    ts_col = _first_col(d, [
        "report_ts", "report_timestamp", "report_datetime", "timestamp",
        "Report Timestamp", "Report DateTime", "Report Time"
    ])
    if ts_col:
        # A time-only column is not enough by itself; combine with game/report date.
        ts_try = pd.to_datetime(d[ts_col], errors="coerce")
        if ts_try.notna().sum() > 0:
            out["report_ts"] = ts_try
        else:
            out["report_ts"] = pd.NaT
    else:
        out["report_ts"] = pd.NaT

    src_col = _first_col(d, ["source", "source_file", "filename", "pdf", "file"])
    if src_col:
        def _ts_from_source(x):
            s = str(x)
            # Examples: Injury-Report_2025-01-10_05PM.pdf / ..._0530PM.pdf
            m = re.search(r"(20\d{2}-\d{2}-\d{2})[_-](\d{1,4})(AM|PM)", s, re.I)
            if not m:
                return pd.NaT
            ds, digits, ap = m.group(1), m.group(2), m.group(3).upper()
            if len(digits) <= 2:
                hh, mm = int(digits), 30
            else:
                digits = digits.zfill(4)
                hh, mm = int(digits[:-2]), int(digits[-2:])
            hh = hh % 12 + (12 if ap == "PM" else 0)
            try:
                return pd.Timestamp(f"{ds} {hh:02d}:{mm:02d}:00")
            except Exception:
                return pd.NaT
        src_ts = d[src_col].map(_ts_from_source)
        out["report_ts"] = out["report_ts"].where(out["report_ts"].notna(), src_ts)

    out["report_reason_substantive"] = ~out["report_reason"].map(_is_placeholder_reason)
    out["report_reason_family"] = out["report_reason"].map(reason_family)
    parsed = pd.DataFrame([parse_injury_text(x) for x in out["report_reason"]])
    out = pd.concat([out.reset_index(drop=True), parsed.reset_index(drop=True)], axis=1)

    # Rows with no player are structural PDF rows, not designations.
    out = out[(out["player_key"].astype(str).str.len() > 0) & out["game_date"].notna()].copy()
    return out.reset_index(drop=True)


# -----------------------------------------------------------------------------
# 3) Collapse snapshots correctly: latest STATUS, latest SUBSTANTIVE reason.
# -----------------------------------------------------------------------------
def collapse_reports_to_player_game(reports: pd.DataFrame) -> pd.DataFrame:
    d = reports.copy()
    keys = ["game_date", "team", "player_key"]
    d = d[d["game_date"].notna() & d["team"].notna() & (d["player_key"].astype(str).str.len() > 0)].copy()
    if d.empty:
        return pd.DataFrame(columns=keys)

    # Deterministic chronological sort. report_ts is preferred; source row breaks ties.
    d["_sort_ts"] = pd.to_datetime(d.get("report_ts"), errors="coerce")
    # NaT sorts first so rows with real timestamps dominate the "last" choice.
    d = d.sort_values(keys + ["_sort_ts", "_source_row"], na_position="first")

    rows = []
    for key, g in d.groupby(keys, sort=False, dropna=False):
        g = g.sort_values(["_sort_ts", "_source_row"], na_position="first")
        latest = g.iloc[-1]
        substantive = g[g["report_reason_substantive"].fillna(False)]
        reason_row = substantive.iloc[-1] if len(substantive) else latest

        rec = {
            "game_date": key[0], "team": key[1], "player_key": key[2],
            "report_status": latest.get("report_status", ""),
            "report_status_latest": latest.get("report_status", ""),
            "report_reason_latest": latest.get("report_reason", ""),
            "report_reason": reason_row.get("report_reason", ""),
            "report_reason_family": reason_row.get("report_reason_family", "unknown"),
            "report_reason_substantive": bool(reason_row.get("report_reason_substantive", False)),
            "report_ts": latest.get("report_ts", pd.NaT),
            "report_reason_ts": reason_row.get("report_ts", pd.NaT),
            "report_snapshot_count": int(len(g)),
            "report_had_substantive_reason": int(len(substantive) > 0),
            "report_matchup": reason_row.get("matchup", latest.get("matchup", "")),
            "report_team_full": reason_row.get("team_full", latest.get("team_full", "")),
        }
        for c in ["injury_body_part", "injury_laterality", "injury_diagnosis_class", "injury_region"]:
            rec[c] = reason_row.get(c, "unknown")
        rows.append(rec)
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 4) Full schedule: regular-season team log + postseason bridge from official IR.
# -----------------------------------------------------------------------------
def _report_matchup_to_home_away(matchup):
    s = re.sub(r"\s+", "", str(matchup or "").upper())
    m = re.search(r"([A-Z]{2,3})@([A-Z]{2,3})", s)
    if not m:
        return None, None
    away, home = m.group(1), m.group(2)
    # Normalize any unusual abbreviation tokens through the same alias layer.
    away = normalize_team_abbr(away) if pd.notna(normalize_team_abbr(away)) else away
    home = normalize_team_abbr(home) if pd.notna(normalize_team_abbr(home)) else home
    return home, away


def _game_type_from_id(gid):
    s = str(gid).zfill(10)
    return GAME_TYPE_LABELS.get(s[:3], "other")


def _postseason_gid_sort_key(gid):
    s = str(gid).zfill(10)
    # Play-in precedes playoffs chronologically even though 005 > 004.
    prefix_order = {"005": 0, "004": 1, "006": 2}.get(s[:3], 9)
    return (prefix_order, s)


def build_full_schedule_v4(player: pd.DataFrame, teamlog: pd.DataFrame,
                           normalized_reports: pd.DataFrame, season: str) -> pd.DataFrame:
    reg = normalize_team_schedule(teamlog, season).copy()
    if "schedule_matchup_resolved" not in reg.columns:
        reg["schedule_matchup_resolved"] = ((reg["home_team"].notna()) & (reg["away_team"].notna())).astype(int)
    reg["schedule_source"] = "teamlog"
    reg["game_type"] = reg["game_id"].map(_game_type_from_id)

    all_gids = player["game_id"].dropna().astype(str).unique().tolist()
    reg_gids = set(reg["game_id"].astype(str))
    missing = [g for g in all_gids if g not in reg_gids]
    if not missing:
        return reg.sort_values(["game_date", "game_id"]).reset_index(drop=True)

    # Derive each missing NBA game pair from the player-game rows themselves.
    pair_rows = []
    for gid, gg in player[player["game_id"].isin(missing)].groupby("game_id", sort=False):
        teams = [x for x in gg["team"].dropna().astype(str).str.upper().unique().tolist() if x]
        if len(teams) == 2:
            pair_rows.append({"game_id": str(gid), "pair_key": "|".join(sorted(teams)), "team_a": teams[0], "team_b": teams[1]})
        else:
            pair_rows.append({"game_id": str(gid), "pair_key": None, "team_a": teams[0] if teams else None, "team_b": None})
    missing_pairs = pd.DataFrame(pair_rows)

    # Distinct report games after the regular-season teamlog endpoint ends.
    rr = normalized_reports.copy()
    report_games = rr[["game_date", "matchup"]].dropna(subset=["game_date"]).drop_duplicates().copy()
    ha = report_games["matchup"].map(_report_matchup_to_home_away)
    report_games["home_team"] = [x[0] for x in ha]
    report_games["away_team"] = [x[1] for x in ha]
    report_games = report_games.dropna(subset=["home_team", "away_team"]).copy()
    report_games["pair_key"] = report_games.apply(lambda r: "|".join(sorted([str(r["home_team"]), str(r["away_team"])])), axis=1)
    reg_max = pd.to_datetime(reg["game_date"], errors="coerce").max()
    if pd.notna(reg_max):
        report_games = report_games[report_games["game_date"] > reg_max].copy()
    report_games = report_games.sort_values(["pair_key", "game_date", "matchup"]).drop_duplicates(["pair_key", "game_date"])

    inferred = []
    audit = []
    for pair_key, mg in missing_pairs.groupby("pair_key", dropna=False, sort=False):
        gids = sorted(mg["game_id"].astype(str).tolist(), key=_postseason_gid_sort_key)
        if pair_key is None or pd.isna(pair_key):
            for gid in gids:
                audit.append({"game_id": gid, "pair_key": pair_key, "status": "player_pair_unresolved"})
            continue
        rg = report_games[report_games["pair_key"] == pair_key].sort_values("game_date").reset_index(drop=True)
        if len(rg) != len(gids):
            audit.append({
                "game_id": ";".join(gids), "pair_key": pair_key,
                "status": f"count_mismatch_missing={len(gids)}_report_dates={len(rg)}"
            })
        n = min(len(gids), len(rg))
        for k in range(n):
            gid = gids[k]; r = rg.iloc[k]
            inferred.append({
                "season": season, "game_id": gid, "game_date": pd.Timestamp(r["game_date"]).normalize(),
                "home_team": r["home_team"], "away_team": r["away_team"],
                "schedule_matchup_resolved": 1, "schedule_source": "official_report_postseason_bridge",
                "game_type": _game_type_from_id(gid),
            })
        for gid in gids[n:]:
            audit.append({"game_id": gid, "pair_key": pair_key, "status": "no_report_date_assigned"})

    extra = pd.DataFrame(inferred)
    full = pd.concat([reg, extra], ignore_index=True, sort=False)
    full = full.drop_duplicates("game_id", keep="first")
    # Preserve unresolved game IDs as explicit rows rather than silently discarding them.
    unresolved_gids = [g for g in missing if g not in set(extra.get("game_id", pd.Series(dtype=str)).astype(str))]
    if unresolved_gids:
        unresolved_rows = []
        for gid in unresolved_gids:
            gg = missing_pairs[missing_pairs["game_id"] == gid]
            teams = [] if gg.empty else [gg.iloc[0].get("team_a"), gg.iloc[0].get("team_b")]
            teams = [t for t in teams if pd.notna(t)]
            unresolved_rows.append({
                "season": season, "game_id": gid, "game_date": pd.NaT,
                "home_team": np.nan, "away_team": np.nan,
                "schedule_matchup_resolved": 0, "schedule_source": "unresolved",
                "game_type": _game_type_from_id(gid),
            })
        full = pd.concat([full, pd.DataFrame(unresolved_rows)], ignore_index=True, sort=False)

    full["game_date"] = pd.to_datetime(full["game_date"], errors="coerce").dt.normalize()
    full = full.sort_values(["game_date", "game_id"], na_position="last").reset_index(drop=True)

    audit_df = pd.DataFrame(audit)
    audit_path = ROOT / "qc_postseason_schedule_bridge.csv"
    audit_df.to_csv(audit_path, index=False)
    print(
        f"  full schedule v4: {len(full):,}/{len(all_gids):,} player-log games; "
        f"dated={full['game_date'].notna().sum():,}; "
        f"postseason_bridge={int((full['schedule_source']=='official_report_postseason_bridge').sum()):,}; "
        f"unresolved={int(full['game_date'].isna().sum()):,}"
    )
    return full


# -----------------------------------------------------------------------------
# 5) Primary absence classification with evidence-source tracking.
# -----------------------------------------------------------------------------
def _explicit_comment_family(comment):
    txt = str(comment or "").lower()
    # Only use comments when they contain an explicit non-coach designation.
    if any(k in txt for k in ["injury", "injury/illness", "inactive - injury", "dnd - injury"]):
        return "injury"
    if "illness" in txt:
        return "illness"
    if "rest" in txt or "load management" in txt:
        return "rest"
    if "susp" in txt:
        return "suspension"
    if "g league" in txt or "two-way" in txt or "assignment" in txt:
        return "g_league"
    if "personal" in txt:
        return "personal"
    return "unknown"


def _choose_absence_reason(row):
    if bool(row.get("played_actual", False)):
        return "played", "played"

    live = reason_family(row.get("not_playing_reason", ""), row.get("not_playing_description", ""))
    if live != "unknown":
        return live, "live_boxscore"

    if bool(row.get("report_reason_substantive", False)):
        fam = str(row.get("report_reason_family", "unknown") or "unknown")
        if fam != "unknown":
            return fam, "official_injury_report"
        return "reported_other", "official_injury_report"

    # A matched report with no substantive reason (often latest Available / '-') is
    # distinct from no report. Keep the distinction for QC.
    if bool(row.get("official_report_matched", False)):
        return "report_no_reason", "official_injury_report"

    cf = _explicit_comment_family(row.get("comment", ""))
    if cf != "unknown":
        return cf, "boxscore_comment_explicit"

    # Do NOT let generic 'DNP - Coach\'s Decision' overwrite a missing injury report.
    return "not_on_report", "no_official_designation_matched"


# -----------------------------------------------------------------------------
# 6) Rebuild player-game master with corrected schedule + report join.
# -----------------------------------------------------------------------------
def build_player_game_master_maximal(season: str, fetch_live: bool = True,
                                     official_reports=None) -> pd.DataFrame:
    player = normalize_playerlog(get_public_playerlog(season), season)
    teamlog = get_public_teamlog(season)

    rr_norm = pd.DataFrame()
    rr_collapsed = pd.DataFrame()
    if official_reports is not None and len(official_reports):
        rr_norm = normalize_official_injury_reports(official_reports)
        rr_collapsed = collapse_reports_to_player_game(rr_norm)

    sched = build_full_schedule_v4(player, teamlog, rr_norm, season) if len(rr_norm) else normalize_team_schedule(teamlog, season)
    if "game_date" in player.columns:
        player = player.drop(columns=["game_date"])
    d = player.merge(sched, on=["season", "game_id"], how="left", validate="many_to_one")
    d["game_date"] = pd.to_datetime(d["game_date"], errors="coerce").dt.normalize()
    d["game_type"] = d["game_id"].map(_game_type_from_id)
    d["is_home"] = np.where(
        d["home_team"].notna(), (d["team"] == d["home_team"]).astype(int), np.nan
    )
    d["opponent"] = np.where(
        d["team"] == d["home_team"], d["away_team"],
        np.where(d["team"] == d["away_team"], d["home_team"], np.nan)
    )

    if fetch_live:
        live_p, live_g = build_live_availability_maximal(d["game_id"].dropna().unique())
        if not live_p.empty:
            live_p["player_id"] = pd.to_numeric(live_p["player_id"], errors="coerce").astype("Int64")
            d = d.merge(live_p, on=["game_id", "team", "player_id"], how="left")
        if not live_g.empty:
            d = d.merge(live_g, on="game_id", how="left")

    if not rr_collapsed.empty:
        keep = [c for c in [
            "game_date", "team", "player_key", "report_status", "report_status_latest",
            "report_reason_latest", "report_reason", "report_reason_family",
            "report_reason_substantive", "report_ts", "report_reason_ts",
            "report_snapshot_count", "report_had_substantive_reason", "report_matchup",
            "report_team_full", "injury_body_part", "injury_laterality",
            "injury_diagnosis_class", "injury_region"
        ] if c in rr_collapsed.columns]
        d = d.merge(rr_collapsed[keep], on=["game_date", "team", "player_key"], how="left")

    d["official_report_matched"] = (
        d.get("report_status", pd.Series(np.nan, index=d.index)).notna() |
        d.get("report_reason", pd.Series(np.nan, index=d.index)).notna()
    )
    d["report_reason_substantive"] = d.get(
        "report_reason_substantive", pd.Series(False, index=d.index)
    ).fillna(False).astype(bool)

    if "live_played" in d.columns:
        d["played_actual"] = np.where(d["live_played"].notna(), d["live_played"], d["played"]).astype(bool)
    else:
        d["played_actual"] = d["played"].astype(bool)

    chosen = d.apply(_choose_absence_reason, axis=1, result_type="expand")
    d["absence_reason_family"] = chosen[0]
    d["absence_reason_source"] = chosen[1]
    d["boxscore_comment_family"] = d.get("comment", pd.Series("", index=d.index)).map(_explicit_comment_family)

    for c in ["injury_body_part", "injury_laterality", "injury_diagnosis_class", "injury_region"]:
        if c not in d.columns:
            d[c] = "unknown"
    # Only use explicit live/comment text as fallback; never infer anatomy from a
    # generic coach-decision string.
    live_desc = d.get("not_playing_description", pd.Series("", index=d.index)).fillna("").astype(str)
    comment = d.get("comment", pd.Series("", index=d.index)).fillna("").astype(str)
    fallback_text = live_desc + " " + comment.where(comment.str.contains("injury|illness", case=False, regex=True), "")
    fb = pd.DataFrame([parse_injury_text(x) for x in fallback_text])
    for c in ["injury_body_part", "injury_laterality", "injury_diagnosis_class", "injury_region"]:
        missing = d[c].isna() | d[c].astype(str).str.lower().isin(["", "nan", "unknown"])
        d.loc[missing, c] = fb.loc[missing, c].values

    d = add_schedule_context(d)
    d = add_team_schedule_context(d)
    d = add_player_workload_rolling_features(d)
    d = d.sort_values(["player_id", "game_date", "game_id"], na_position="last").reset_index(drop=True)

    bad = d[d["game_date"].isna()].copy()
    if len(bad):
        bad.to_csv(ROOT / "player_game_missing_schedule_dates.csv", index=False)
        print(f"  WARNING v4: {len(bad):,} player-game rows remain without dates across {bad['game_id'].nunique():,} games.")
    return d


# -----------------------------------------------------------------------------
# 7) Episode utilities + clinically safer outcome naming.
# -----------------------------------------------------------------------------
def _mode_known_from_df(df, col):
    if col not in df.columns or df.empty:
        return "unknown"
    s = df[col].dropna().astype(str).str.strip()
    s = s[~s.str.lower().isin(["", "nan", "unknown", "none"])]
    return s.mode().iloc[0] if len(s) else "unknown"


def _safe_days(a, b):
    a = pd.to_datetime(a, errors="coerce")
    b = pd.to_datetime(b, errors="coerce")
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return int((a - b).days)


def _same_known(a, b):
    a = str(a or "").strip().lower(); b = str(b or "").strip().lower()
    return a not in {"", "unknown", "nan", "none"} and b not in {"", "unknown", "nan", "none"} and a == b


def build_injury_episodes(pg: pd.DataFrame, min_games_missed: int = 2,
                          pre_games: int = 10, post_games: int = 20,
                          meaningful_rtp_minutes: float = MEANINGFUL_RTP_MINUTES) -> pd.DataFrame:
    d = _sanitize_player_game_dates(pg, "injury episodes v4")
    d = d.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    rows = []

    for pid, g0 in d.groupby("player_id", sort=False):
        g = g0.reset_index(drop=True)
        med = (~g["played_actual"].fillna(False).astype(bool)) & (g["absence_reason_family"] == "injury")
        idx = np.flatnonzero(med.to_numpy())
        if len(idx) == 0:
            continue
        starts = [idx[0]]; ends = []
        for a, b in zip(idx[:-1], idx[1:]):
            if b != a + 1:
                ends.append(a); starts.append(b)
        ends.append(idx[-1])

        for start, end in zip(starts, ends):
            nmiss = end - start + 1
            if nmiss < min_games_missed:
                continue
            ep = g.loc[start:end].copy()

            first_app_cands = g.index[(g.index > end) & g["played_actual"].fillna(False).astype(bool)].tolist()
            first_app = first_app_cands[0] if first_app_cands else None
            meaningful_cands = g.index[
                (g.index > end) & g["played_actual"].fillna(False).astype(bool) &
                (pd.to_numeric(g["minutes"], errors="coerce") >= float(meaningful_rtp_minutes))
            ].tolist()
            meaningful = meaningful_cands[0] if meaningful_cands else None

            prev_played = g.iloc[:start]
            prev_played = prev_played[prev_played["played_actual"].fillna(False).astype(bool)].tail(pre_games)
            prev_meaningful = prev_played[pd.to_numeric(prev_played["minutes"], errors="coerce") >= meaningful_rtp_minutes]
            baseline = pd.to_numeric(prev_played["minutes"], errors="coerce").median() if len(prev_played) else np.nan
            baseline_meaningful = pd.to_numeric(prev_meaningful["minutes"], errors="coerce").median() if len(prev_meaningful) else np.nan

            anchor = meaningful if meaningful is not None else first_app
            anchor_row = g.loc[anchor] if anchor is not None else None
            first_row = g.loc[first_app] if first_app is not None else None

            injury_region = _mode_known_from_df(ep, "injury_region")
            injury_body = _mode_known_from_df(ep, "injury_body_part")
            injury_lat = _mode_known_from_df(ep, "injury_laterality")
            injury_dx = _mode_known_from_df(ep, "injury_diagnosis_class")

            episode_id = f"{int(pid)}_{pd.Timestamp(g.loc[start,'game_date']).date()}_{start}"
            row = {
                "episode_id": episode_id, "season": g.loc[start, "season"],
                "player_id": pid, "player_name": g.loc[start, "player_name"], "team": g.loc[start, "team"],
                "injury_start_game_date": g.loc[start, "game_date"],
                "last_missed_game_date": g.loc[end, "game_date"],
                "games_missed": int(nmiss),
                "injury_region": injury_region, "injury_body_part": injury_body,
                "injury_laterality": injury_lat, "injury_diagnosis_class": injury_dx,
                "returned_same_season_any_appearance": int(first_app is not None),
                "first_appearance_game_id": g.loc[first_app, "game_id"] if first_app is not None else np.nan,
                "first_appearance_date": g.loc[first_app, "game_date"] if first_app is not None else pd.NaT,
                "first_appearance_minutes": float(pd.to_numeric(g.loc[first_app, "minutes"], errors="coerce")) if first_app is not None else np.nan,
                "meaningful_rtp_threshold_minutes": float(meaningful_rtp_minutes),
                "meaningful_return_found": int(meaningful is not None),
                "meaningful_return_game_id": g.loc[meaningful, "game_id"] if meaningful is not None else np.nan,
                "meaningful_return_game_date": g.loc[meaningful, "game_date"] if meaningful is not None else pd.NaT,
                "meaningful_return_minutes": float(pd.to_numeric(g.loc[meaningful, "minutes"], errors="coerce")) if meaningful is not None else np.nan,
                # Backward-compatible names point to the meaningful return when possible.
                "return_game_date": g.loc[anchor, "game_date"] if anchor is not None else pd.NaT,
                "return_game_id": g.loc[anchor, "game_id"] if anchor is not None else np.nan,
                "return_game_minutes": float(pd.to_numeric(g.loc[anchor, "minutes"], errors="coerce")) if anchor is not None else np.nan,
                "games_between_first_appearance_and_meaningful_return": (int(meaningful-first_app) if meaningful is not None and first_app is not None else np.nan),
                "calendar_days_from_first_miss_to_first_appearance": _safe_days(g.loc[first_app,"game_date"], g.loc[start,"game_date"]) if first_app is not None else np.nan,
                "calendar_days_from_first_miss_to_return": _safe_days(g.loc[anchor,"game_date"], g.loc[start,"game_date"]) if anchor is not None else np.nan,
                "pre_n_games": int(len(prev_played)),
                "pre_n_meaningful_games": int(len(prev_meaningful)),
                "pre_median_minutes": float(baseline) if pd.notna(baseline) else np.nan,
                "pre_median_minutes_meaningful": float(baseline_meaningful) if pd.notna(baseline_meaningful) else np.nan,
                "pre_mean_minutes": _safe_mean(prev_played["minutes"]) if len(prev_played) else np.nan,
                "baseline_ge_10mpg_flag": int(pd.notna(baseline) and baseline >= PRIMARY_BASELINE_MINUTES),
                "baseline_ge_15mpg_flag": int(pd.notna(baseline) and baseline >= STRICT_BASELINE_MINUTES),
                "pre_games_ge5_flag": int(len(prev_played) >= MIN_PRE_GAMES_PRIMARY),
                "first_appearance_is_cameo_lt5min": int(first_app is not None and float(pd.to_numeric(g.loc[first_app,"minutes"],errors="coerce")) < meaningful_rtp_minutes),
            }
            row["primary_analysis_eligible"] = int(
                meaningful is not None and len(prev_played) >= MIN_PRE_GAMES_PRIMARY and
                pd.notna(baseline) and baseline >= PRIMARY_BASELINE_MINUTES
            )
            row["strict_rotation_analysis_eligible"] = int(
                meaningful is not None and len(prev_played) >= MIN_PRE_GAMES_PRIMARY and
                pd.notna(baseline) and baseline >= STRICT_BASELINE_MINUTES
            )

            # Ramp features are anchored on the meaningful return, not a 1-second cameo.
            if meaningful is not None and pd.notna(baseline) and baseline > 0:
                # Stop played-game ramp at the next injury-related absence.
                future_med_idx = g.index[(g.index > meaningful) & (~g["played_actual"].fillna(False).astype(bool)) & (g["absence_reason_family"] == "injury")].tolist()
                stop = future_med_idx[0] if future_med_idx else len(g)
                post_pool = g.loc[meaningful:stop-1] if stop > meaningful else g.loc[meaningful:meaningful]
                post_played = post_pool[post_pool["played_actual"].fillna(False).astype(bool)].head(post_games)
                post_minutes_arr = pd.to_numeric(post_played["minutes"], errors="coerce").to_numpy(float)
                ratios = post_minutes_arr / float(baseline)
                first90 = np.flatnonzero(ratios >= 0.90)
                row.update({
                    "return_minutes_ratio": float(ratios[0]) if len(ratios) else np.nan,
                    "post3_mean_minutes_ratio": float(np.nanmean(ratios[:3])) if len(ratios) else np.nan,
                    "post5_mean_minutes_ratio": float(np.nanmean(ratios[:5])) if len(ratios) else np.nan,
                    "ramp_slope_minutes_per_game": _safe_slope(post_minutes_arr[:5]),
                    "max_positive_minute_step_first5": float(np.nanmax(np.diff(post_minutes_arr[:5]))) if len(post_minutes_arr[:5]) >= 2 else np.nan,
                    "ramp_deficit_auc_first5": float(np.nansum(np.maximum(0, float(baseline) - post_minutes_arr[:5]))),
                    "games_to_90pct_baseline": int(first90[0]+1) if len(first90) else np.nan,
                    "post_played_games_available_before_next_injury": int(len(post_played)),
                })

                # Team schedule context: use team-game sequence, not player's prior appearance.
                team5 = g.loc[meaningful:min(len(g)-1, meaningful+4)].copy()
                row["return_team_is_b2b"] = anchor_row.get("team_is_b2b", np.nan)
                row["return_team_calendar_rest_days"] = anchor_row.get("team_calendar_rest_days", np.nan)
                row["return_team_rest_hours"] = anchor_row.get("team_rest_hours", np.nan)
                row["return_travel_km"] = anchor_row.get("travel_km_since_prev", np.nan)
                row["hours_since_last_player_appearance"] = anchor_row.get("hours_since_last_player_appearance", np.nan)
                row["b2b_count_first5_team_games_post"] = int(pd.to_numeric(team5.get("team_is_b2b"), errors="coerce").fillna(0).sum()) if "team_is_b2b" in team5 else 0
                row["travel_km_first5_team_games_post"] = pd.to_numeric(team5.get("travel_km_since_prev"), errors="coerce").sum(min_count=1) if "travel_km_since_prev" in team5 else np.nan

                perf_candidates = [
                    ("trueShootingPercentage", "ts_pct"), ("usagePercentage", "usage_pct"),
                    ("netRating", "net_rating"), ("PIE", "pie"), ("plusMinusPoints", "plus_minus"),
                    ("points", "points"),
                ]
                for src, label in perf_candidates:
                    if src in g.columns:
                        row[f"pre_{label}"] = _safe_mean(prev_played[src]) if len(prev_played) else np.nan
                        row[f"post5_{label}"] = _safe_mean(post_played.head(5)[src]) if len(post_played) else np.nan
                        a, b = row[f"pre_{label}"], row[f"post5_{label}"]
                        row[f"post5_minus_pre_{label}"] = b-a if pd.notna(a) and pd.notna(b) else np.nan
            else:
                for c in ["return_minutes_ratio","post3_mean_minutes_ratio","post5_mean_minutes_ratio",
                          "ramp_slope_minutes_per_game","max_positive_minute_step_first5",
                          "ramp_deficit_auc_first5","games_to_90pct_baseline","post_played_games_available_before_next_injury"]:
                    row[c] = np.nan

            # Subsequent injury-related absence outcomes. These are proxies, not diagnoses of reinjury.
            if anchor is not None:
                after = g.loc[anchor+1:].copy()
                future = after[(~after["played_actual"].fillna(False).astype(bool)) & (after["absence_reason_family"] == "injury")]
            else:
                future = pd.DataFrame()

            if len(future):
                nxt_idx = int(future.index[0]); nxt = g.loc[nxt_idx]
                days = _safe_days(nxt["game_date"], anchor_row["game_date"])
                between = g.loc[anchor+1:nxt_idx-1] if nxt_idx > anchor+1 else g.iloc[0:0]
                played_between = int(between["played_actual"].fillna(False).astype(bool).sum()) if len(between) else 0
                same_region = _same_known(injury_region, nxt.get("injury_region"))
                same_body = _same_known(injury_body, nxt.get("injury_body_part"))
                same_lat = _same_known(injury_lat, nxt.get("injury_laterality"))
                same_dx = _same_known(injury_dx, nxt.get("injury_diagnosis_class"))
                row.update({
                    "next_subsequent_injury_absence_date": nxt["game_date"],
                    "days_to_next_subsequent_injury_absence": days,
                    "played_games_before_next_subsequent_injury_absence": played_between,
                    "next_injury_body_part": nxt.get("injury_body_part", "unknown"),
                    "next_injury_region": nxt.get("injury_region", "unknown"),
                    "next_injury_laterality": nxt.get("injury_laterality", "unknown"),
                    "next_injury_diagnosis_class": nxt.get("injury_diagnosis_class", "unknown"),
                })
                for horizon in (14,30,60):
                    within = pd.notna(days) and days <= horizon
                    row[f"subsequent_any_injury_absence_within_{horizon}d"] = int(within)
                    row[f"subsequent_same_region_absence_within_{horizon}d"] = int(within and same_region)
                    row[f"subsequent_same_body_part_absence_within_{horizon}d"] = int(within and same_body)
                    row[f"subsequent_same_body_laterality_absence_within_{horizon}d"] = int(within and same_body and same_lat)
                    row[f"subsequent_same_body_laterality_dx_absence_within_{horizon}d"] = int(within and same_body and same_lat and same_dx)
                row["early_same_body_continuation_proxy_30d_lt3_played"] = int(
                    pd.notna(days) and days <= 30 and same_body and played_between < MIN_HEALTHY_PLAYED_GAMES_FOR_SEPARATED_EVENT
                )
                row["separated_same_body_event_proxy_30d_ge3_played"] = int(
                    pd.notna(days) and days <= 30 and same_body and played_between >= MIN_HEALTHY_PLAYED_GAMES_FOR_SEPARATED_EVENT
                )
                row["separated_same_body_laterality_dx_proxy_60d_ge3_played"] = int(
                    pd.notna(days) and days <= 60 and same_body and same_lat and same_dx and played_between >= MIN_HEALTHY_PLAYED_GAMES_FOR_SEPARATED_EVENT
                )
            else:
                row.update({
                    "next_subsequent_injury_absence_date": pd.NaT,
                    "days_to_next_subsequent_injury_absence": np.nan,
                    "played_games_before_next_subsequent_injury_absence": np.nan,
                    "next_injury_body_part": "", "next_injury_region": "",
                    "next_injury_laterality": "", "next_injury_diagnosis_class": "",
                    "early_same_body_continuation_proxy_30d_lt3_played": 0,
                    "separated_same_body_event_proxy_30d_ge3_played": 0,
                    "separated_same_body_laterality_dx_proxy_60d_ge3_played": 0,
                })
                for horizon in (14,30,60):
                    for stem in ["subsequent_any_injury_absence", "subsequent_same_region_absence",
                                 "subsequent_same_body_part_absence", "subsequent_same_body_laterality_absence",
                                 "subsequent_same_body_laterality_dx_absence"]:
                        row[f"{stem}_within_{horizon}d"] = 0
            rows.append(row)
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 8) All absence episodes + meaningful-return fields.
# -----------------------------------------------------------------------------
def build_all_absence_episodes(pg: pd.DataFrame, min_team_games_missed: int = 1,
                               meaningful_rtp_minutes: float = MEANINGFUL_RTP_MINUTES) -> pd.DataFrame:
    d = _sanitize_player_game_dates(pg, "all absence episodes v4")
    d = d.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    rows=[]
    for pid,g0 in d.groupby("player_id",sort=False):
        g=g0.reset_index(drop=True)
        missed=~g["played_actual"].fillna(False).astype(bool)
        i=0
        while i<len(g):
            if not missed.iloc[i]: i+=1; continue
            j=i
            while j+1<len(g) and missed.iloc[j+1]: j+=1
            n=j-i+1
            if n>=min_team_games_missed:
                ep=g.loc[i:j]
                fam=ep["absence_reason_family"].fillna("unknown").astype(str)
                primary=fam.mode().iloc[0] if len(fam) else "unknown"
                first_app = next((k for k in range(j+1,len(g)) if bool(g.loc[k,"played_actual"])), None)
                meaningful = next((k for k in range(j+1,len(g)) if bool(g.loc[k,"played_actual"]) and pd.to_numeric(g.loc[k,"minutes"],errors="coerce")>=meaningful_rtp_minutes), None)
                rows.append({
                    "absence_episode_id":f"{pid}_{pd.Timestamp(g.loc[i,'game_date']).date()}_{i}",
                    "season":g.loc[i,"season"],"player_id":pid,"player_name":g.loc[i,"player_name"],"team":g.loc[i,"team"],
                    "absence_reason_family":primary,"first_missed_game_date":g.loc[i,"game_date"],"last_missed_game_date":g.loc[j,"game_date"],
                    "team_games_missed":n,"first_appearance_date":g.loc[first_app,"game_date"] if first_app is not None else pd.NaT,
                    "first_appearance_minutes":pd.to_numeric(g.loc[first_app,"minutes"],errors="coerce") if first_app is not None else np.nan,
                    "meaningful_return_game_date":g.loc[meaningful,"game_date"] if meaningful is not None else pd.NaT,
                    "meaningful_return_minutes":pd.to_numeric(g.loc[meaningful,"minutes"],errors="coerce") if meaningful is not None else np.nan,
                    "returned_same_season":int(first_app is not None),"meaningful_return_found":int(meaningful is not None),
                    "injury_region":_mode_known_from_df(ep,"injury_region"),"injury_body_part":_mode_known_from_df(ep,"injury_body_part"),
                    "absence_reason_source_mode":_mode_known_from_df(ep,"absence_reason_source"),
                })
            i=j+1
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 9) RTP windows anchored on meaningful return and true pre-injury games.
# -----------------------------------------------------------------------------
def build_episode_game_windows(pg: pd.DataFrame, injury_episodes: pd.DataFrame,
                               pre_played_games=10, post_team_games=20) -> pd.DataFrame:
    d = _sanitize_player_game_dates(pg, "episode windows v4")
    if injury_episodes is None or injury_episodes.empty:
        return pd.DataFrame()
    d = d.sort_values(["player_id","game_date","game_id"]).reset_index(drop=True)
    rows=[]
    for e in injury_episodes.itertuples(index=False):
        if not int(getattr(e,"meaningful_return_found",0)):
            continue
        g=d[d["player_id"]==e.player_id].sort_values(["game_date","game_id"]).reset_index(drop=True)
        ret_gid=str(getattr(e,"meaningful_return_game_id",""))
        ret_idx=g.index[g["game_id"].astype(str)==ret_gid].tolist()
        if not ret_idx:
            continue
        r=ret_idx[0]
        inj_date=pd.to_datetime(getattr(e,"injury_start_game_date",pd.NaT),errors="coerce")
        pre=g[(g["game_date"]<inj_date)&g["played_actual"].fillna(False).astype(bool)].tail(pre_played_games)
        for rel,(_,row) in zip(range(-len(pre),0),pre.iterrows()):
            x=row.to_dict();x.update({"episode_id":e.episode_id,"relative_game":rel,"window_phase":"preinjury_played","analysis_anchor":"meaningful_rtp"});rows.append(x)
        post=g.loc[r:min(len(g)-1,r+post_team_games)]
        for rel,(_,row) in enumerate(post.iterrows()):
            x=row.to_dict();x.update({"episode_id":e.episode_id,"relative_game":rel,"window_phase":"meaningful_return_or_post","analysis_anchor":"meaningful_rtp"});rows.append(x)
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 10) QC tables and exports.
# -----------------------------------------------------------------------------
def build_v4_qc_tables(pg: pd.DataFrame, injury_ep: pd.DataFrame,
                       official_reports_raw=None) -> dict:
    tables={}
    x=pg.copy()
    x["nonplayed"]=(~x["played_actual"].fillna(False).astype(bool)).astype(int)
    x["report_matched_nonplayed"]=(x["nonplayed"].astype(bool)&x["official_report_matched"].fillna(False).astype(bool)).astype(int)
    x["report_substantive_nonplayed"]=(x["nonplayed"].astype(bool)&x["report_reason_substantive"].fillna(False).astype(bool)).astype(int)
    x["injury_absence"]=(x["absence_reason_family"]=="injury").astype(int)
    x["not_on_report_absence"]=(x["absence_reason_family"]=="not_on_report").astype(int)
    x["report_no_reason_absence"]=(x["absence_reason_family"]=="report_no_reason").astype(int)

    team=x.groupby("team",as_index=False).agg(
        player_game_rows=("game_id","size"),games=("game_id","nunique"),nonplayed_rows=("nonplayed","sum"),
        report_matched_nonplayed=("report_matched_nonplayed","sum"),
        substantive_report_nonplayed=("report_substantive_nonplayed","sum"),
        injury_absence_rows=("injury_absence","sum"),not_on_report_rows=("not_on_report_absence","sum"),
        report_no_reason_rows=("report_no_reason_absence","sum"),
    )
    team["report_match_rate_nonplayed"]=team["report_matched_nonplayed"]/team["nonplayed_rows"].replace(0,np.nan)
    team["substantive_reason_rate_nonplayed"]=team["substantive_report_nonplayed"]/team["nonplayed_rows"].replace(0,np.nan)
    team["qc_warning"]=""
    team.loc[team["injury_absence_rows"]==0,"qc_warning"]="ZERO injury absences — inspect join"
    team.loc[(team["report_match_rate_nonplayed"]<0.25)&(team["qc_warning"]==""),"qc_warning"]="low official-report match rate"
    tables["qc_team_report_coverage"]=team.sort_values(["qc_warning","team"],ascending=[False,True])

    sched=x[[c for c in ["game_id","game_date","game_type","schedule_source","home_team","away_team"] if c in x.columns]].drop_duplicates("game_id")
    if len(sched):
        sq=sched.groupby(["game_type","schedule_source"],dropna=False,as_index=False).agg(games=("game_id","nunique"),dated_games=("game_date",lambda s:int(pd.to_datetime(s,errors="coerce").notna().sum())))
        tables["qc_schedule_coverage"]=sq
        tables["qc_schedule_missing_games"]=sched[pd.to_datetime(sched["game_date"],errors="coerce").isna()].copy()

    if injury_ep is not None and len(injury_ep):
        summary=[]
        def add(label,mask): summary.append({"metric":label,"n":int(np.asarray(mask).sum()),"pct":float(np.asarray(mask).mean()) if len(mask) else np.nan})
        n=len(injury_ep)
        add("all injury time-loss episodes",np.ones(n,dtype=bool))
        add("meaningful return found",injury_ep["meaningful_return_found"].fillna(0).astype(bool))
        add("primary analysis eligible (>=5 pre games, baseline >=10 MPG, meaningful RTP)",injury_ep["primary_analysis_eligible"].fillna(0).astype(bool))
        add("strict rotation eligible (baseline >=15 MPG)",injury_ep["strict_rotation_analysis_eligible"].fillna(0).astype(bool))
        add("first appearance cameo <5 min",injury_ep["first_appearance_is_cameo_lt5min"].fillna(0).astype(bool))
        add("known body part",~injury_ep["injury_body_part"].astype(str).str.lower().isin(["","unknown","nan"]))
        add("subsequent injury absence within 30d",injury_ep["subsequent_any_injury_absence_within_30d"].fillna(0).astype(bool))
        add("same body-part subsequent absence within 30d",injury_ep["subsequent_same_body_part_absence_within_30d"].fillna(0).astype(bool))
        add("separated same-body event proxy within 30d after >=3 played games",injury_ep["separated_same_body_event_proxy_30d_ge3_played"].fillna(0).astype(bool))
        tables["qc_episode_summary"]=pd.DataFrame(summary)
        tables["qc_injury_body_part_counts"]=injury_ep.groupby(["injury_region","injury_body_part"],dropna=False).size().reset_index(name="episodes").sort_values("episodes",ascending=False)

    unclassified=x[x["nonplayed"].astype(bool)&x["absence_reason_family"].isin(["not_on_report","report_no_reason","reported_other"])].copy()
    cols=[c for c in ["season","game_id","game_date","team","player_id","player_name","comment","report_status","report_reason_latest","report_reason","absence_reason_family","absence_reason_source"] if c in unclassified.columns]
    tables["qc_unclassified_absences"]=unclassified[cols]

    if official_reports_raw is not None and len(official_reports_raw):
        rr=collapse_reports_to_player_game(normalize_official_injury_reports(official_reports_raw))
        keys=["game_date","team","player_key"]
        playerkeys=x[keys].drop_duplicates().assign(_playerlog_key=1)
        ur=rr.merge(playerkeys,on=keys,how="left")
        tables["qc_unmatched_report_keys"]=ur[ur["_playerlog_key"].isna()].drop(columns=["_playerlog_key"])

    for name,df in tables.items():
        if isinstance(df,pd.DataFrame):
            df.to_csv(ROOT/f"{name}.csv",index=False)
    return tables


# -----------------------------------------------------------------------------
# 11) Updated runner. Same six-object return signature for compatibility.
# -----------------------------------------------------------------------------
def run_maximal_pilot_2024_25(fetch_live=True, min_games_missed=2,
                              make_excel=True, deep_mode="off",
                              deep_scope="episode_windows"):
    season="2024-25"
    reports=get_2425_official_injury_designations()
    print(f"Official injury designation rows: {len(reports):,}")

    pg=build_player_game_master_maximal(season,fetch_live=fetch_live,official_reports=reports)
    pg_path=ROOT/"player_game_master_2024_25_MAXIMAL_v4.csv"
    pg.to_csv(pg_path,index=False)
    print(f"Player-game rows: {len(pg):,} x {pg.shape[1]:,} columns -> {pg_path}")

    injury_ep=build_injury_episodes(pg,min_games_missed=min_games_missed,pre_games=10,post_games=20)
    injury_ep=add_player_age(injury_ep,season)
    injury_ep.to_csv(ROOT/"rtp_injury_episodes_2024_25_v4.csv",index=False)

    all_abs=build_all_absence_episodes(pg,min_team_games_missed=1)
    all_abs.to_csv(ROOT/"all_absence_episodes_2024_25_v4.csv",index=False)

    windows=build_episode_game_windows(pg,injury_ep,pre_played_games=10,post_team_games=20)
    windows.to_csv(ROOT/"rtp_episode_game_windows_2024_25_v4.csv",index=False)

    qc=build_v4_qc_tables(pg,injury_ep,reports)

    profiles=pd.DataFrame();deep_audit=pd.DataFrame()
    if deep_mode in {"focused","maximal"}:
        try: profiles=player_profile_archive(pg["player_id"].dropna().unique(),season=season)
        except Exception as e: print("Player profile archive skipped:",e)
        endpoint_names=FOCUSED_DEEP_ENDPOINTS if deep_mode=="focused" else MAXIMAL_DEEP_ENDPOINTS
        gids=episode_window_game_ids(windows) if deep_scope=="episode_windows" else pg["game_id"].dropna().astype(str).unique().tolist()
        print(f"Deep archive target: {len(gids):,} games x {len(endpoint_names)} endpoint families")
        deep_audit=archive_deep_endpoints(gids,endpoint_names)

    if make_excel:
        tables={
            "player_game_master":pg,"injury_episodes":injury_ep,"all_absence_episodes":all_abs,
            "rtp_game_windows":windows,"player_profiles":profiles,"deep_archive_audit":deep_audit,
            **qc,
        }
        export_excel_bundle(tables,filename="NBA_Injury_RTP_Pilot_2024_25_v4.xlsx")

    dated_games=pg.loc[pg["game_date"].notna(),"game_id"].nunique()
    all_games=pg["game_id"].nunique()
    audit={
        "season":season,"player_game_rows":int(len(pg)),"player_game_columns":int(pg.shape[1]),
        "unique_games":int(all_games),"dated_games":int(dated_games),"undated_games":int(all_games-dated_games),
        "injury_episode_rows":int(len(injury_ep)),"primary_analysis_eligible_episodes":int(injury_ep.get("primary_analysis_eligible",pd.Series(dtype=int)).fillna(0).sum()) if len(injury_ep) else 0,
        "all_absence_episode_rows":int(len(all_abs)),"episode_window_rows":int(len(windows)),
        "deep_mode":deep_mode,"deep_scope":deep_scope,
        "injury_absence_rows":int((pg["absence_reason_family"]=="injury").sum()),
        "not_on_report_absence_rows":int((pg["absence_reason_family"]=="not_on_report").sum()),
        "report_no_reason_absence_rows":int((pg["absence_reason_family"]=="report_no_reason").sum()),
        "official_report_matches":int(pg["official_report_matched"].fillna(False).sum()),
        "substantive_official_report_matches":int(pg["report_reason_substantive"].fillna(False).sum()),
    }
    (ROOT/"maximal_pilot_audit_2024_25_v4.json").write_text(json.dumps(audit,indent=2,default=str))
    print(json.dumps(audit,indent=2,default=str))
    return pg,injury_ep,all_abs,windows,profiles,deep_audit


# -----------------------------------------------------------------------------
# 12) Lightweight synthetic regression tests for the two major join bugs.
# -----------------------------------------------------------------------------
def self_test_v4():
    # Test A: block-level date/team fields must forward-fill.
    fake=pd.DataFrame({
        "Game Date":["01/10/2025",None,None],"Matchup":["SAC@BOS",None,None],
        "Team":["Sacramento Kings",None,None],"Player Name":["Fox, De'Aaron","Murray, Keegan","Jones, Mason"],
        "Current Status":["Questionable","Out","Out"],
        "Reason":["Injury/Illness - Right Glute; Contusion","Injury/Illness - Left Ankle; Inflammation","G League - Two-Way"],
    })
    n=normalize_official_injury_reports(fake)
    assert len(n)==3 and n["game_date"].notna().all() and (n["team"]=="SAC").all(), n

    # Test B: later Available/'-' must not erase earlier injury reason.
    fake2=pd.DataFrame({
        "game_date":[pd.Timestamp("2025-01-10")]*2,"matchup":["SAC@BOS"]*2,
        "team_full":["Sacramento Kings"]*2,"team":["SAC"]*2,
        "player_name_report":["Fox, De'Aaron"]*2,"player_key":[norm_name("Fox, De'Aaron")]*2,
        "report_status":["Out","Available"],"report_reason":["Injury/Illness - Right Glute; Contusion","-"],
        "report_reason_substantive":[True,False],"report_reason_family":["injury","unknown"],
        "injury_body_part":["unknown","unknown"],"injury_laterality":["right","unknown"],
        "injury_diagnosis_class":["contusion","unknown"],"injury_region":["lower_extremity","unknown"],
        "report_ts":[pd.Timestamp("2025-01-10 17:30"),pd.Timestamp("2025-01-10 19:30")],"_source_row":[0,1],
    })
    c=collapse_reports_to_player_game(fake2)
    assert c.iloc[0]["report_status"]=="Available"
    assert c.iloc[0]["report_reason_family"]=="injury" and "Glute" in c.iloc[0]["report_reason"], c
    print("v4 self-test passed: report forward-fill + substantive-reason preservation are working.")
    return True

self_test_v4()
print("NBA RTP QC/cohort fix v4 loaded.")
print(f"  meaningful RTP threshold = {MEANINGFUL_RTP_MINUTES:g} min (raw first appearance still preserved)")
print(f"  primary rotation baseline flag = >= {PRIMARY_BASELINE_MINUTES:g} MPG")
print("  postseason schedule bridge + team/report QC + safer subsequent-injury outcomes enabled")


## 4E. Structural-completeness patch (v5)

This fixes the remaining major pilot issue: public player logs omit many inactive rostered players. Official-report-only player-game rows are appended as zero-minute observations, and raw player-log dates are retained as a schedule fallback.


In [ ]:
"""
NBA RTP structural completeness fix v5

Load AFTER the v4 patch. This fixes the remaining major pilot problem:
public player-game logs do not contain a row for every inactive rostered player.
Official NBA injury reports therefore contain thousands of legitimate player-game
records that cannot join to the box-score/player-log master.

v5:
  * preserves the original player-log game date instead of replacing it;
  * uses player-log dates to fill any residual schedule-date hole;
  * maps collapsed official-report rows to NBA game IDs by team + date;
  * appends synthetic zero-minute player-game rows for report-listed players who
    are absent from the public player log (they did not record a box-score row);
  * maps report-only players back to NBA player IDs whenever the player appears
    elsewhere in the season; otherwise assigns a deterministic negative surrogate;
  * adds explicit row provenance and completeness QC;
  * keeps the v4 clinically safer RTP and subsequent-injury definitions;
  * adds a 2024-25 benchmark QC against the source project's published 6,396
    injury-related missed games. The benchmark is QC only, never used to alter data.
"""

import hashlib
import json
import re
from pathlib import Path

V5_INJURY_MISSED_BENCHMARK_2425 = 6396


def _stable_surrogate_player_id(player_key: str) -> int:
    """Deterministic negative ID for a report-only player with no known NBA ID."""
    s = str(player_key or "unknown").encode("utf-8")
    return -int(hashlib.md5(s).hexdigest()[:12], 16)


def build_full_schedule_v5(player: pd.DataFrame, teamlog: pd.DataFrame,
                           normalized_reports: pd.DataFrame, season: str) -> pd.DataFrame:
    """v4 schedule + original player-log date fallback.

    Schedule is preferred for home/away metadata, but it must never erase a date
    already present in the raw player log. This resolves residual postseason date
    holes such as 0042400216 without an extra network dependency.
    """
    p = player.copy()
    if "game_date" in p.columns:
        p["game_date"] = pd.to_datetime(p["game_date"], errors="coerce").dt.normalize()
    sched = build_full_schedule_v4(p, teamlog, normalized_reports, season).copy()

    raw_dates = (
        p[["game_id", "game_date"]]
        .dropna(subset=["game_id"])
        .assign(game_id=lambda x: x["game_id"].astype(str))
        .groupby("game_id", as_index=False)["game_date"]
        .agg(lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NaT)
        .rename(columns={"game_date": "playerlog_game_date"})
    )
    sched["game_id"] = sched["game_id"].astype(str)
    sched = sched.merge(raw_dates, on="game_id", how="left")
    sched["schedule_date_original"] = pd.to_datetime(sched["game_date"], errors="coerce").dt.normalize()
    sched["game_date"] = sched["schedule_date_original"].where(
        sched["schedule_date_original"].notna(),
        pd.to_datetime(sched["playerlog_game_date"], errors="coerce").dt.normalize(),
    )
    filled = sched["schedule_date_original"].isna() & sched["game_date"].notna()
    sched.loc[filled, "schedule_source"] = sched.loc[filled, "schedule_source"].astype(str) + "+playerlog_date"
    sched["schedule_date_filled_from_playerlog"] = filled.astype(int)
    sched = sched.drop(columns=["playerlog_game_date"], errors="ignore")

    print(
        f"  full schedule v5: {len(sched):,} games; "
        f"dated={sched['game_date'].notna().sum():,}; "
        f"date_fallbacks={int(filled.sum()):,}; "
        f"unresolved_home_away={int((sched['home_team'].isna() | sched['away_team'].isna()).sum()):,}"
    )
    return sched


def _map_reports_to_game_ids_v5(rr_collapsed: pd.DataFrame, player: pd.DataFrame,
                                sched: pd.DataFrame) -> pd.DataFrame:
    """Attach game_id to each official player/team/date designation."""
    rr = rr_collapsed.copy()
    rr["game_date"] = pd.to_datetime(rr["game_date"], errors="coerce").dt.normalize()

    # Strongest map: observed player-log team/date -> game ID. Every real game has
    # multiple participants even when the injured player himself has no row.
    pmap = player[["game_id", "team", "game_date"]].copy()
    pmap["game_date"] = pd.to_datetime(pmap["game_date"], errors="coerce").dt.normalize()
    pmap["team"] = pmap["team"].astype(str).str.upper()
    pmap = pmap.dropna(subset=["game_id", "game_date", "team"]).drop_duplicates()
    counts = pmap.groupby(["game_date", "team"])["game_id"].nunique().rename("_n_games").reset_index()
    pmap = pmap.merge(counts, on=["game_date", "team"], how="left")
    pmap = pmap[pmap["_n_games"] == 1].drop(columns="_n_games").drop_duplicates(["game_date", "team"])
    rr = rr.merge(pmap.rename(columns={"game_id": "game_id_from_playerlog"}),
                  on=["game_date", "team"], how="left")

    # Secondary map from normalized schedule, useful if source-player logs change.
    home = sched[["game_id", "game_date", "home_team"]].rename(columns={"home_team": "team"})
    away = sched[["game_id", "game_date", "away_team"]].rename(columns={"away_team": "team"})
    smap = pd.concat([home, away], ignore_index=True).dropna(subset=["game_date", "team"]).copy()
    smap["game_date"] = pd.to_datetime(smap["game_date"], errors="coerce").dt.normalize()
    smap["team"] = smap["team"].astype(str).str.upper()
    smap = smap.drop_duplicates(["game_date", "team"])
    rr = rr.merge(smap.rename(columns={"game_id": "game_id_from_schedule"}),
                  on=["game_date", "team"], how="left")
    rr["game_id"] = rr["game_id_from_playerlog"].where(
        rr["game_id_from_playerlog"].notna(), rr["game_id_from_schedule"]
    )
    rr["game_id"] = rr["game_id"].astype("string")
    rr["report_game_id_resolved"] = rr["game_id"].notna().astype(int)
    rr = rr.drop(columns=["game_id_from_playerlog", "game_id_from_schedule"], errors="ignore")
    return rr


def _augment_playerlog_with_report_only_rows_v5(player: pd.DataFrame,
                                                rr_with_gid: pd.DataFrame,
                                                season: str):
    """Append report-listed player/game records absent from the public player log."""
    p = player.copy()
    p["game_date"] = pd.to_datetime(p.get("game_date"), errors="coerce").dt.normalize()
    p["team"] = p["team"].astype(str).str.upper()
    p["player_key"] = p["player_key"].astype(str)
    p["game_id"] = p["game_id"].astype(str)
    p["synthetic_report_only_row"] = 0
    p["playerlog_row_present"] = 1

    # Known identity lookup from any observed game in the season.
    identity = (
        p[["player_key", "player_id", "player_name"]]
        .dropna(subset=["player_key"])
        .sort_values(["player_key", "player_id"], na_position="last")
        .drop_duplicates("player_key")
        .set_index("player_key")
    )
    team_meta_cols = [c for c in ["team_id", "teamCity", "teamName", "teamSlug"] if c in p.columns]
    team_meta = p[["team"] + team_meta_cols].drop_duplicates("team").set_index("team") if team_meta_cols else None

    existing = set(zip(p["game_id"], p["team"], p["player_key"]))
    rr = rr_with_gid[rr_with_gid["game_id"].notna()].copy()
    rr["game_id"] = rr["game_id"].astype(str)
    rr["team"] = rr["team"].astype(str).str.upper()
    missing = rr[["game_id", "team", "player_key"]].apply(tuple, axis=1).map(lambda k: k not in existing)
    add = rr[missing].copy()

    if add.empty:
        return p, add

    rows = []
    for _, r in add.iterrows():
        key = str(r["player_key"])
        team = str(r["team"])
        rec = {c: np.nan for c in p.columns}
        rec.update({
            "season": season,
            "game_id": str(r["game_id"]),
            "game_date": pd.to_datetime(r["game_date"], errors="coerce"),
            "team": team,
            "player_key": key,
            "minutes_raw": "0:00",
            "minutes": 0.0,
            "played": False,
            "comment": "",
            "synthetic_report_only_row": 1,
            "playerlog_row_present": 0,
        })
        if key in identity.index:
            ident = identity.loc[key]
            rec["player_id"] = ident.get("player_id")
            rec["player_name"] = ident.get("player_name")
        else:
            rec["player_id"] = _stable_surrogate_player_id(key)
            rec["player_name"] = key.title()
        if team_meta is not None and team in team_meta.index:
            tm = team_meta.loc[team]
            for c in team_meta_cols:
                rec[c] = tm.get(c)
        rows.append(rec)

    synthetic = pd.DataFrame(rows, columns=p.columns)
    out = pd.concat([p, synthetic], ignore_index=True, sort=False)
    # Restore common dtypes after concat.
    out["player_id"] = pd.to_numeric(out["player_id"], errors="coerce").astype("Int64")
    out["played"] = out["played"].fillna(False).astype(bool)
    out["synthetic_report_only_row"] = out["synthetic_report_only_row"].fillna(0).astype(int)
    out["playerlog_row_present"] = out["playerlog_row_present"].fillna(0).astype(int)
    return out, add


def build_player_game_master_maximal(season: str, fetch_live: bool = True,
                                     official_reports=None) -> pd.DataFrame:
    """v5 master: observed player logs + official-report-only inactive rows."""
    player = normalize_playerlog(get_public_playerlog(season), season)
    teamlog = get_public_teamlog(season)
    player["game_date"] = pd.to_datetime(player.get("game_date"), errors="coerce").dt.normalize()
    player["playerlog_game_date_raw"] = player["game_date"]

    rr_norm = pd.DataFrame(); rr_collapsed = pd.DataFrame(); rr_gid = pd.DataFrame()
    if official_reports is not None and len(official_reports):
        rr_norm = normalize_official_injury_reports(official_reports)
        rr_collapsed = collapse_reports_to_player_game(rr_norm)

    sched = build_full_schedule_v5(player, teamlog, rr_norm, season) if len(rr_norm) else normalize_team_schedule(teamlog, season)

    if not rr_collapsed.empty:
        rr_gid = _map_reports_to_game_ids_v5(rr_collapsed, player, sched)
        player, report_only = _augment_playerlog_with_report_only_rows_v5(player, rr_gid, season)
        print(
            f"  v5 report-only augmentation: added {len(report_only):,} player-game rows; "
            f"injury={int((report_only.get('report_reason_family') == 'injury').sum()):,}; "
            f"illness={int((report_only.get('report_reason_family') == 'illness').sum()):,}; "
            f"g_league={int((report_only.get('report_reason_family') == 'g_league').sum()):,}"
        )
    else:
        report_only = pd.DataFrame()
        player["synthetic_report_only_row"] = 0
        player["playerlog_row_present"] = 1

    # Preserve the original date. Schedule enriches but does not overwrite it.
    player = player.rename(columns={"game_date": "_player_game_date_for_merge"})
    d = player.merge(sched, on=["season", "game_id"], how="left", validate="many_to_one", suffixes=("", "_sched"))
    schedule_date = pd.to_datetime(d.get("game_date"), errors="coerce")
    raw_date = pd.to_datetime(d.get("_player_game_date_for_merge"), errors="coerce")
    d["game_date"] = schedule_date.where(schedule_date.notna(), raw_date).dt.normalize()
    d["playerlog_game_date_raw"] = pd.to_datetime(d.get("playerlog_game_date_raw"), errors="coerce").dt.normalize()
    d = d.drop(columns=["_player_game_date_for_merge"], errors="ignore")
    d["game_type"] = d["game_id"].map(_game_type_from_id)
    d["is_home"] = np.where(d["home_team"].notna(), (d["team"] == d["home_team"]).astype(int), np.nan)
    d["opponent"] = np.where(
        d["team"] == d["home_team"], d["away_team"],
        np.where(d["team"] == d["away_team"], d["home_team"], np.nan)
    )

    if fetch_live:
        live_p, live_g = build_live_availability_maximal(d["game_id"].dropna().unique())
        if not live_p.empty:
            live_p["player_id"] = pd.to_numeric(live_p["player_id"], errors="coerce").astype("Int64")
            d = d.merge(live_p, on=["game_id", "team", "player_id"], how="left")
        if not live_g.empty:
            d = d.merge(live_g, on="game_id", how="left")

    if not rr_gid.empty:
        keep = [c for c in [
            "game_id", "game_date", "team", "player_key", "report_status", "report_status_latest",
            "report_reason_latest", "report_reason", "report_reason_family",
            "report_reason_substantive", "report_ts", "report_reason_ts",
            "report_snapshot_count", "report_had_substantive_reason", "report_matchup",
            "report_team_full", "injury_body_part", "injury_laterality",
            "injury_diagnosis_class", "injury_region", "report_game_id_resolved"
        ] if c in rr_gid.columns]
        # game_id+team+player_key is stronger than date and works for synthetic rows.
        rrj = rr_gid[keep].drop_duplicates(["game_id", "team", "player_key"])
        d = d.drop(columns=[c for c in keep if c in d.columns and c not in ["game_id", "team", "player_key", "game_date"]], errors="ignore")
        # Avoid a duplicate report-side game_date column.
        rrj = rrj.drop(columns=["game_date"], errors="ignore")
        d = d.merge(rrj, on=["game_id", "team", "player_key"], how="left", validate="many_to_one")

    d["official_report_matched"] = (
        d.get("report_status", pd.Series(np.nan, index=d.index)).notna() |
        d.get("report_reason", pd.Series(np.nan, index=d.index)).notna()
    )
    d["report_reason_substantive"] = d.get(
        "report_reason_substantive", pd.Series(False, index=d.index)
    ).fillna(False).astype(bool)

    if "live_played" in d.columns:
        d["played_actual"] = np.where(d["live_played"].notna(), d["live_played"], d["played"]).astype(bool)
    else:
        d["played_actual"] = d["played"].fillna(False).astype(bool)

    chosen = d.apply(_choose_absence_reason, axis=1, result_type="expand")
    d["absence_reason_family"] = chosen[0]
    d["absence_reason_source"] = chosen[1]
    d["boxscore_comment_family"] = d.get("comment", pd.Series("", index=d.index)).map(_explicit_comment_family)

    for c in ["injury_body_part", "injury_laterality", "injury_diagnosis_class", "injury_region"]:
        if c not in d.columns:
            d[c] = "unknown"
    live_desc = d.get("not_playing_description", pd.Series("", index=d.index)).fillna("").astype(str)
    comment = d.get("comment", pd.Series("", index=d.index)).fillna("").astype(str)
    fallback_text = live_desc + " " + comment.where(comment.str.contains("injury|illness", case=False, regex=True), "")
    fb = pd.DataFrame([parse_injury_text(x) for x in fallback_text])
    for c in ["injury_body_part", "injury_laterality", "injury_diagnosis_class", "injury_region"]:
        missing = d[c].isna() | d[c].astype(str).str.lower().isin(["", "nan", "unknown"])
        d.loc[missing, c] = fb.loc[missing, c].values

    d = add_schedule_context(d)
    d = add_team_schedule_context(d)
    d = add_player_workload_rolling_features(d)
    d = d.sort_values(["player_id", "game_date", "game_id"], na_position="last").reset_index(drop=True)

    bad = d[d["game_date"].isna()].copy()
    if len(bad):
        bad.to_csv(ROOT / "player_game_missing_schedule_dates_v5.csv", index=False)
        print(f"  WARNING v5: {len(bad):,} rows remain without dates across {bad['game_id'].nunique():,} games.")
    return d


def build_v5_qc_tables(pg: pd.DataFrame, injury_ep: pd.DataFrame,
                       official_reports_raw=None) -> dict:
    """v4 QC + structural completeness diagnostics."""
    tables = build_v4_qc_tables(pg, injury_ep, official_reports_raw)
    x = pg.copy()
    injury_n = int((x["absence_reason_family"] == "injury").sum())
    report_only_n = int(x.get("synthetic_report_only_row", pd.Series(0, index=x.index)).fillna(0).sum())
    report_only_injury_n = int(((x.get("synthetic_report_only_row", 0) == 1) & (x["absence_reason_family"] == "injury")).sum())

    benchmark = pd.DataFrame([
        {"metric": "injury missed player-games captured", "value": injury_n},
        {"metric": "published 2024-25 injury missed-game benchmark", "value": V5_INJURY_MISSED_BENCHMARK_2425},
        {"metric": "capture ratio vs published benchmark", "value": injury_n / V5_INJURY_MISSED_BENCHMARK_2425},
        {"metric": "synthetic official-report-only rows added", "value": report_only_n},
        {"metric": "synthetic injury rows added", "value": report_only_injury_n},
        {"metric": "remaining undated games", "value": x.loc[x["game_date"].isna(), "game_id"].nunique()},
    ])
    tables["qc_structural_completeness"] = benchmark

    team = x.groupby("team", as_index=False).agg(
        player_game_rows=("game_id", "size"),
        games=("game_id", "nunique"),
        synthetic_report_only_rows=("synthetic_report_only_row", "sum"),
        official_report_matches=("official_report_matched", "sum"),
        injury_absence_rows=("absence_reason_family", lambda s: int((s == "injury").sum())),
        illness_absence_rows=("absence_reason_family", lambda s: int((s == "illness").sum())),
        g_league_absence_rows=("absence_reason_family", lambda s: int((s == "g_league").sum())),
        not_on_report_rows=("absence_reason_family", lambda s: int((s == "not_on_report").sum())),
    )
    team["injury_zero_warning"] = (team["injury_absence_rows"] == 0).astype(int)
    tables["qc_team_structural_coverage"] = team.sort_values("injury_absence_rows")

    # The v4 unmatched-report table should collapse dramatically. Recompute using
    # game_id/team/player_key because date-only joins are no longer the core key.
    if official_reports_raw is not None and len(official_reports_raw):
        rr = collapse_reports_to_player_game(normalize_official_injury_reports(official_reports_raw))
        player_base = normalize_playerlog(get_public_playerlog("2024-25"), "2024-25")
        player_base["game_date"] = pd.to_datetime(player_base.get("game_date"), errors="coerce").dt.normalize()
        sched = build_full_schedule_v5(player_base, get_public_teamlog("2024-25"), normalize_official_injury_reports(official_reports_raw), "2024-25")
        rg = _map_reports_to_game_ids_v5(rr, player_base, sched)
        keys = x[["game_id", "team", "player_key"]].drop_duplicates().assign(_master_key=1)
        ur = rg.merge(keys, on=["game_id", "team", "player_key"], how="left")
        tables["qc_unmatched_report_keys_v5"] = ur[ur["_master_key"].isna()].drop(columns=["_master_key"])

    for name, df in tables.items():
        if isinstance(df, pd.DataFrame):
            df.to_csv(ROOT / f"{name}.csv", index=False)
    return tables


def run_maximal_pilot_2024_25(fetch_live=True, min_games_missed=2,
                              make_excel=True, deep_mode="off",
                              deep_scope="episode_windows"):
    season = "2024-25"
    reports = get_2425_official_injury_designations()
    print(f"Official injury designation rows: {len(reports):,}")

    pg = build_player_game_master_maximal(season, fetch_live=fetch_live, official_reports=reports)
    pg_path = ROOT / "player_game_master_2024_25_MAXIMAL_v5.csv"
    pg.to_csv(pg_path, index=False)
    print(f"Player-game rows: {len(pg):,} x {pg.shape[1]:,} columns -> {pg_path}")

    injury_ep = build_injury_episodes(pg, min_games_missed=min_games_missed, pre_games=10, post_games=20)
    injury_ep = add_player_age(injury_ep, season)
    injury_ep.to_csv(ROOT / "rtp_injury_episodes_2024_25_v5.csv", index=False)

    all_abs = build_all_absence_episodes(pg, min_team_games_missed=1)
    all_abs.to_csv(ROOT / "all_absence_episodes_2024_25_v5.csv", index=False)

    windows = build_episode_game_windows(pg, injury_ep, pre_played_games=10, post_team_games=20)
    windows.to_csv(ROOT / "rtp_episode_game_windows_2024_25_v5.csv", index=False)

    qc = build_v5_qc_tables(pg, injury_ep, reports)

    profiles = pd.DataFrame(); deep_audit = pd.DataFrame()
    if deep_mode in {"focused", "maximal"}:
        try:
            profiles = player_profile_archive(pg.loc[pg["player_id"] > 0, "player_id"].dropna().unique(), season=season)
        except Exception as e:
            print("Player profile archive skipped:", e)
        endpoint_names = FOCUSED_DEEP_ENDPOINTS if deep_mode == "focused" else MAXIMAL_DEEP_ENDPOINTS
        gids = episode_window_game_ids(windows) if deep_scope == "episode_windows" else pg["game_id"].dropna().astype(str).unique().tolist()
        print(f"Deep archive target: {len(gids):,} games x {len(endpoint_names)} endpoint families")
        deep_audit = archive_deep_endpoints(gids, endpoint_names)

    if make_excel:
        tables = {
            "player_game_master": pg,
            "injury_episodes": injury_ep,
            "all_absence_episodes": all_abs,
            "rtp_game_windows": windows,
            "player_profiles": profiles,
            "deep_archive_audit": deep_audit,
            **qc,
        }
        export_excel_bundle(tables, filename="NBA_Injury_RTP_Pilot_2024_25_v5.xlsx")

    dated_games = pg.loc[pg["game_date"].notna(), "game_id"].nunique()
    all_games = pg["game_id"].nunique()
    injury_abs = int((pg["absence_reason_family"] == "injury").sum())
    audit = {
        "season": season,
        "player_game_rows": int(len(pg)),
        "player_game_columns": int(pg.shape[1]),
        "unique_games": int(all_games),
        "dated_games": int(dated_games),
        "undated_games": int(all_games - dated_games),
        "synthetic_report_only_rows": int(pg["synthetic_report_only_row"].fillna(0).sum()),
        "injury_episode_rows": int(len(injury_ep)),
        "primary_analysis_eligible_episodes": int(injury_ep.get("primary_analysis_eligible", pd.Series(dtype=int)).fillna(0).sum()) if len(injury_ep) else 0,
        "all_absence_episode_rows": int(len(all_abs)),
        "episode_window_rows": int(len(windows)),
        "deep_mode": deep_mode,
        "deep_scope": deep_scope,
        "injury_absence_rows": injury_abs,
        "injury_absence_capture_vs_6396_benchmark": injury_abs / V5_INJURY_MISSED_BENCHMARK_2425,
        "not_on_report_absence_rows": int((pg["absence_reason_family"] == "not_on_report").sum()),
        "official_report_matches": int(pg["official_report_matched"].fillna(False).sum()),
        "substantive_official_report_matches": int(pg["report_reason_substantive"].fillna(False).sum()),
    }
    (ROOT / "maximal_pilot_audit_2024_25_v5.json").write_text(json.dumps(audit, indent=2, default=str))
    print(json.dumps(audit, indent=2, default=str))
    return pg, injury_ep, all_abs, windows, profiles, deep_audit


def self_test_v5():
    """Network-free regression tests for report-only augmentation and date fallback."""
    season = "2024-25"
    player = pd.DataFrame({
        "season": [season, season],
        "game_id": ["0022400001", "0022400001"],
        "game_date": [pd.Timestamp("2024-10-22"), pd.Timestamp("2024-10-22")],
        "team": ["BOS", "NYK"],
        "player_id": pd.Series([1, 2], dtype="Int64"),
        "player_name": ["Healthy Celtic", "Healthy Knick"],
        "player_key": ["healthy celtic", "healthy knick"],
        "minutes_raw": ["20:00", "20:00"], "minutes": [20.0, 20.0],
        "played": [True, True], "comment": ["", ""],
    })
    rr = pd.DataFrame({
        "game_date": [pd.Timestamp("2024-10-22")], "team": ["BOS"],
        "player_key": ["injured celtic"], "report_reason_family": ["injury"],
        "report_reason": ["Injury/Illness - Left Ankle; Sprain"],
    })
    sched = pd.DataFrame({
        "season": [season], "game_id": ["0022400001"], "game_date": [pd.Timestamp("2024-10-22")],
        "home_team": ["BOS"], "away_team": ["NYK"], "schedule_source": ["test"]
    })
    rg = _map_reports_to_game_ids_v5(rr, player, sched)
    assert rg.iloc[0]["game_id"] == "0022400001", rg
    aug, added = _augment_playerlog_with_report_only_rows_v5(player, rg, season)
    assert len(added) == 1 and int(aug["synthetic_report_only_row"].sum()) == 1, aug
    syn = aug[aug["synthetic_report_only_row"] == 1].iloc[0]
    assert syn["minutes"] == 0 and not bool(syn["played"]), syn

    # Schedule-date fallback should preserve a raw player date when v4 leaves NaT.
    fake_sched = pd.DataFrame({
        "season": [season], "game_id": ["0042400216"], "game_date": [pd.NaT],
        "home_team": [np.nan], "away_team": [np.nan], "schedule_source": ["unresolved"]
    })
    raw_dates = player.iloc[[0]].copy(); raw_dates["game_id"] = "0042400216"; raw_dates["game_date"] = pd.Timestamp("2025-05-16")
    # Test the combine rule directly to avoid monkeypatching build_full_schedule_v4.
    merged = fake_sched.merge(raw_dates[["game_id", "game_date"]].rename(columns={"game_date": "raw_date"}), on="game_id", how="left")
    fixed = pd.to_datetime(merged["game_date"], errors="coerce").where(
        pd.to_datetime(merged["game_date"], errors="coerce").notna(), pd.to_datetime(merged["raw_date"], errors="coerce")
    )
    assert fixed.notna().all()
    print("v5 self-test passed: report-only row augmentation + raw-date fallback are working.")
    return True

self_test_v5()


In [ ]:
"""
NBA RTP v5.1 date-source/schema fix

Load AFTER the v5 structural-completeness patch.

Why this patch exists:
- llimllib/nba_data playerlog_<season>.parquet does NOT contain game_date.
- v5 incorrectly assumed normalize_playerlog() would always return game_date and
  called .dt on pd.to_datetime(None), causing an immediate crash.
- This patch makes normalize_playerlog schema-safe and adds an optional, cached
  LeagueGameLog playoff schedule fallback for any postseason game that the
  regular-season team log + official injury-report bridge cannot date.
"""

from pathlib import Path
import time
import re
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1) Schema-safe player-log normalization.
# -----------------------------------------------------------------------------
_normalize_playerlog_before_v51 = normalize_playerlog


def normalize_playerlog(df: pd.DataFrame, season: str) -> pd.DataFrame:
    """Call the existing normalizer, but guarantee a datetime64 game_date Series.

    The public player-log parquet is not expected to contain dates. Downstream code
    may still safely reference game_date; it will simply be NaT until the schedule
    merge supplies the authoritative date.
    """
    d = _normalize_playerlog_before_v51(df, season)
    if "game_date" not in d.columns:
        d["game_date"] = pd.Series(pd.NaT, index=d.index, dtype="datetime64[ns]")
    else:
        d["game_date"] = pd.to_datetime(d["game_date"], errors="coerce").dt.normalize()
    return d


# -----------------------------------------------------------------------------
# 2) Optional playoff schedule fallback from stats.nba.com.
# -----------------------------------------------------------------------------
def _normalize_leaguegamelog_schedule_v51(raw: pd.DataFrame, season: str) -> pd.DataFrame:
    """Convert team-level LeagueGameLog rows to one row per game."""
    if raw is None or raw.empty:
        return pd.DataFrame(columns=[
            "season", "game_id", "game_date", "home_team", "away_team",
            "schedule_matchup_resolved", "schedule_source", "game_type",
        ])

    d = raw.copy()
    ren = {}
    for c in d.columns:
        u = str(c).upper()
        if u == "GAME_ID": ren[c] = "game_id"
        elif u == "GAME_DATE": ren[c] = "game_date"
        elif u == "TEAM_ABBREVIATION": ren[c] = "team"
        elif u == "MATCHUP": ren[c] = "matchup"
    d = d.rename(columns=ren)
    needed = {"game_id", "game_date", "team", "matchup"}
    if not needed.issubset(d.columns):
        return pd.DataFrame(columns=[
            "season", "game_id", "game_date", "home_team", "away_team",
            "schedule_matchup_resolved", "schedule_source", "game_type",
        ])

    d["game_id"] = d["game_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(10)
    d["game_date"] = pd.to_datetime(d["game_date"], errors="coerce").dt.normalize()
    d["team"] = d["team"].astype(str).str.upper()
    d["is_home"] = d["matchup"].astype(str).str.contains(r"\bvs\.?\b", case=False, regex=True)

    rows = []
    for gid, g in d.groupby("game_id", sort=False):
        home = g[g["is_home"]]
        away = g[~g["is_home"]]
        # Normally LeagueGameLog has both team rows. If only one is available,
        # infer the opponent from the matchup text when possible.
        h = str(home.iloc[0]["team"]) if not home.empty else None
        a = str(away.iloc[0]["team"]) if not away.empty else None
        if h is None or a is None:
            m = str(g.iloc[0]["matchup"])
            toks = re.findall(r"\b[A-Z]{3}\b", m.upper()) if "re" in globals() else []
            if len(toks) >= 2:
                if "@" in m:
                    a, h = toks[0], toks[-1]
                else:
                    h, a = toks[0], toks[-1]
        rows.append({
            "season": season,
            "game_id": str(gid),
            "game_date": g["game_date"].dropna().iloc[0] if g["game_date"].notna().any() else pd.NaT,
            "home_team": h,
            "away_team": a,
            "schedule_matchup_resolved": int(h is not None and a is not None),
            "schedule_source": "nba_stats_leaguegamelog_playoffs",
            "game_type": _game_type_from_id(str(gid)),
        })
    return pd.DataFrame(rows)


def _get_playoff_schedule_v51(season: str, retries: int = 2) -> pd.DataFrame:
    """Fetch/cached LeagueGameLog playoff schedule; failure is nonfatal."""
    cache = RAW / f"leaguegamelog_playoffs_{str(season).replace('-', '')}.csv"
    if cache.exists():
        try:
            raw = pd.read_csv(cache, dtype={"GAME_ID": str, "game_id": str})
            return _normalize_leaguegamelog_schedule_v51(raw, season)
        except Exception:
            pass

    last = None
    for attempt in range(retries):
        try:
            from nba_api.stats.endpoints import leaguegamelog
            obj = leaguegamelog.LeagueGameLog(
                season=season,
                season_type_all_star="Playoffs",
                player_or_team_abbreviation="T",
                sorter="DATE",
                direction="ASC",
                timeout=45,
            )
            raw = obj.get_data_frames()[0]
            raw.to_csv(cache, index=False)
            return _normalize_leaguegamelog_schedule_v51(raw, season)
        except Exception as e:
            last = e
            time.sleep(1.5 * (attempt + 1))
    print(f"  NOTE v5.1: playoff LeagueGameLog fallback unavailable ({last}); continuing with report bridge only.")
    return pd.DataFrame()


# Keep the old v5 schedule builder available for comparison/fallback.
_build_full_schedule_v5_before_v51 = build_full_schedule_v5


def build_full_schedule_v5(player: pd.DataFrame, teamlog: pd.DataFrame,
                           normalized_reports: pd.DataFrame, season: str) -> pd.DataFrame:
    """Schema-safe v5 schedule + optional LeagueGameLog repair of unresolved playoffs."""
    p = player.copy()
    if "game_date" not in p.columns:
        p["game_date"] = pd.Series(pd.NaT, index=p.index, dtype="datetime64[ns]")
    else:
        p["game_date"] = pd.to_datetime(p["game_date"], errors="coerce").dt.normalize()

    # The existing v5 routine is now safe because game_date always exists as a Series.
    sched = _build_full_schedule_v5_before_v51(p, teamlog, normalized_reports, season).copy()
    sched["game_date"] = pd.to_datetime(sched["game_date"], errors="coerce").dt.normalize()

    unresolved = sched["game_date"].isna() | sched["home_team"].isna() | sched["away_team"].isna()
    n_before = int(unresolved.sum())
    repaired = 0

    if n_before:
        playoff = _get_playoff_schedule_v51(season)
        if playoff is not None and not playoff.empty:
            pp = playoff.drop_duplicates("game_id").set_index("game_id")
            for i in sched.index[unresolved]:
                gid = str(sched.at[i, "game_id"])
                if gid not in pp.index:
                    continue
                r = pp.loc[gid]
                changed = False
                if pd.isna(sched.at[i, "game_date"]) and pd.notna(r.get("game_date")):
                    sched.at[i, "game_date"] = pd.Timestamp(r.get("game_date")).normalize(); changed = True
                if pd.isna(sched.at[i, "home_team"]) and pd.notna(r.get("home_team")):
                    sched.at[i, "home_team"] = r.get("home_team"); changed = True
                if pd.isna(sched.at[i, "away_team"]) and pd.notna(r.get("away_team")):
                    sched.at[i, "away_team"] = r.get("away_team"); changed = True
                if changed:
                    sched.at[i, "schedule_source"] = str(sched.at[i, "schedule_source"]) + "+leaguegamelog_playoffs"
                    sched.at[i, "schedule_matchup_resolved"] = int(
                        pd.notna(sched.at[i, "home_team"]) and pd.notna(sched.at[i, "away_team"])
                    )
                    repaired += 1

    sched["game_date"] = pd.to_datetime(sched["game_date"], errors="coerce").dt.normalize()
    unresolved_after = int((sched["game_date"].isna() | sched["home_team"].isna() | sched["away_team"].isna()).sum())
    print(
        f"  v5.1 schedule repair: unresolved_before={n_before:,}; "
        f"repaired={repaired:,}; unresolved_after={unresolved_after:,}"
    )
    return sched


# -----------------------------------------------------------------------------
# 3) Regression test reproducing the user's exact failure condition.
# -----------------------------------------------------------------------------
def self_test_v51():
    # Raw-like player log with NO game_date column.
    raw = pd.DataFrame({
        "gameId": ["0022400001", "0022400001"],
        "teamTricode": ["BOS", "NYK"],
        "personId": [1, 2],
        "firstName": ["A", "B"],
        "familyName": ["One", "Two"],
        "minutes": ["20:00", "20:00"],
    })
    d = normalize_playerlog(raw, "2024-25")
    assert "game_date" in d.columns
    assert pd.api.types.is_datetime64_any_dtype(d["game_date"])
    assert d["game_date"].isna().all()

    # These are the two expressions that crashed in v5; both must now be safe.
    x = pd.to_datetime(d.get("game_date"), errors="coerce").dt.normalize()
    assert len(x) == len(d)
    y = d.copy(); y["playerlog_game_date_raw"] = y["game_date"]
    z = pd.to_datetime(y.get("playerlog_game_date_raw"), errors="coerce").dt.normalize()
    assert len(z) == len(y)
    print("v5.1 self-test passed: player logs without game_date are schema-safe.")
    return True


self_test_v51()
print("NBA RTP v5.1 date-source fix loaded.")


## 4G. Medical reason-family / final QC patch (v5.2)

This fixes a workbook-audit finding: the NBA umbrella label `Injury/Illness` caused uncommon orthopedic terms (e.g. ACL, fibula, metatarsal, patella) to be mislabeled as illness. It also supplies the one audited 2024–25 playoff schedule override and replaces the misleading benchmark “capture ratio” with reference-only QC.


In [ ]:
"""
NBA RTP v5.2 medical-reason classification + final pilot QC fix

Load AFTER v5.1 and BEFORE Step 6.

Fixes discovered by auditing the actual v5 Excel workbook:
1) reason_family() treated the generic NBA label "Injury/Illness" as evidence of
   illness whenever an anatomy word was not in a short keyword list. This caused
   obvious orthopedic diagnoses such as ACL tears, fibula fractures, patellar
   tendinopathy, metatarsal repairs, plantar fasciitis, and thumb sprains to be
   mislabeled as illness.
2) 0042400216 (BOS @ NYK, 2025-05-16) remained the one undated 2024-25 game when
   stats.nba.com timed out. It is added here as an explicit audited override.
3) The published 6,396 injury / 328 illness figures come from an independent
   Basketball-Reference-based participation pipeline and should be treated as a
   reference comparison, not a literal capture-rate denominator for this master.
"""

import json
import re
from pathlib import Path
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1) Clinically safer reason-family classifier.
# -----------------------------------------------------------------------------
_V52_ILLNESS_TERMS = [
    "illness", "non-covid", "non covid", "covid", "influenza", " flu",
    "viral", "virus", "fever", "gastro", "stomach bug", "stomach illness",
    "migraine", "headache", "sinus", "common cold", " cold", "upper respiratory",
    "respiratory illness", "food poisoning", "nausea", "vomit", "diarrhea",
    "infection", "allergic reaction",
]

# Injury/orthopedic diagnosis language. If one of these is present, it should not
# be converted to illness merely because the NBA umbrella prefix says Injury/Illness.
_V52_INJURY_DX_TERMS = [
    "fracture", "broken", "sprain", "strain", "tear", "torn", "rupture",
    "contusion", "bruise", "soreness", "tightness", "tendin", "fasciitis",
    "impingement", "inflammation", "surgery", "repair", "rehab", "recovery",
    "disloc", "sublux", "laceration", "bone bruise", "stress reaction",
    "injury management", "cartilage", "disc", "fissure",
]


def _v52_detail_text(text: str) -> str:
    t = str(text or "").lower()
    if "injury/illness" in t:
        return t.split("injury/illness", 1)[1]
    return t


def reason_family(*parts) -> str:
    """Classify the *reason*, not the NBA umbrella label "Injury/Illness".

    Priority: explicit nonmedical reason -> clear illness -> orthopedic injury.
    For an Injury/Illness row, an explicit orthopedic diagnosis wins over the
    generic word 'illness' contained in the prefix.
    """
    text = " ".join(str(x or "") for x in parts).strip().lower()
    if not text or text in {"-", "nan", "none"}:
        return "unknown"

    # Explicit nonmedical categories first.
    if any(k in text for k in ["g league", "g-league", "gleague", "two-way", "assignment"]):
        return "g_league"
    if "rest" in text or "load management" in text:
        return "rest"
    if "susp" in text:
        return "suspension"
    if "personal" in text or "family matter" in text:
        return "personal"
    if "trade" in text or "not with team" in text:
        return "roster_transaction"
    if "coach" in text:
        return "coach"

    detail = _v52_detail_text(text)
    has_injury_container = any(k in text for k in [
        "injury/illness", "injury", "dnd_injury", "dnp_injury",
        "inactive_injury", "nwt_injury",
    ])
    has_injury_dx = any(k in detail for k in _V52_INJURY_DX_TERMS)
    has_illness = any(k in detail for k in _V52_ILLNESS_TERMS)

    if has_injury_container:
        # e.g. "Right ACL; Tear" or "Right Fibula; Fracture" must be injury.
        if has_injury_dx:
            return "injury"
        # e.g. "Non-Covid; Illness" or "Head; Cold" is illness.
        if has_illness:
            return "illness"
        # The NBA's umbrella prefix is used for medical injuries even when the free
        # text uses uncommon anatomy/diagnosis wording. Default remaining cases to injury.
        return "injury"

    if any(k in text for k in _V52_ILLNESS_TERMS) or "health_and_safety" in text or "health and safety" in text:
        return "illness"
    return "unknown"


# -----------------------------------------------------------------------------
# 2) Explicit audited schedule override for the sole unresolved 2024-25 game.
# -----------------------------------------------------------------------------
_build_full_schedule_v5_before_v52 = build_full_schedule_v5

V52_GAME_OVERRIDES = {
    # Official NBA game 0042400216: Boston Celtics at New York Knicks,
    # Eastern Conference Semifinals Game 6, May 16, 2025.
    ("2024-25", "0042400216"): {
        "game_date": pd.Timestamp("2025-05-16"),
        "home_team": "NYK",
        "away_team": "BOS",
        "source": "audited_manual_override_nba_game_page",
    },
}


def build_full_schedule_v5(player: pd.DataFrame, teamlog: pd.DataFrame,
                           normalized_reports: pd.DataFrame, season: str) -> pd.DataFrame:
    sched = _build_full_schedule_v5_before_v52(player, teamlog, normalized_reports, season).copy()
    sched["game_id"] = sched["game_id"].astype(str)
    fixed = 0
    for (ssn, gid), ov in V52_GAME_OVERRIDES.items():
        if ssn != season:
            continue
        mask = sched["game_id"] == str(gid)
        if not mask.any():
            continue
        i = sched.index[mask][0]
        changed = False
        if pd.isna(sched.at[i, "game_date"]):
            sched.at[i, "game_date"] = ov["game_date"]; changed = True
        if pd.isna(sched.at[i, "home_team"]):
            sched.at[i, "home_team"] = ov["home_team"]; changed = True
        if pd.isna(sched.at[i, "away_team"]):
            sched.at[i, "away_team"] = ov["away_team"]; changed = True
        if changed:
            sched.at[i, "schedule_source"] = ov["source"]
            sched.at[i, "schedule_matchup_resolved"] = 1
            fixed += 1
    sched["game_date"] = pd.to_datetime(sched["game_date"], errors="coerce").dt.normalize()
    unresolved = int((sched["game_date"].isna() | sched["home_team"].isna() | sched["away_team"].isna()).sum())
    print(f"  v5.2 audited schedule overrides: fixed={fixed:,}; unresolved_after={unresolved:,}")
    return sched


# -----------------------------------------------------------------------------
# 3) Better QC: reason-family validation and reference comparisons.
# -----------------------------------------------------------------------------
_build_v5_qc_tables_before_v52 = build_v5_qc_tables


def build_v5_qc_tables(pg: pd.DataFrame, injury_ep: pd.DataFrame,
                       official_reports_raw=None) -> dict:
    tables = _build_v5_qc_tables_before_v52(pg, injury_ep, official_reports_raw)
    x = pg.copy()
    x["_gid"] = x["game_id"].astype(str)
    missed = ~x["played_actual"].fillna(False).astype(bool)
    reg = x["_gid"].str.startswith("002")

    def nmask(mask):
        return int(mask.fillna(False).sum())

    all_injury = nmask(missed & (x["absence_reason_family"] == "injury"))
    all_illness = nmask(missed & (x["absence_reason_family"] == "illness"))
    reg_injury = nmask(missed & reg & (x["absence_reason_family"] == "injury"))
    reg_illness = nmask(missed & reg & (x["absence_reason_family"] == "illness"))

    reason_txt = x.get("report_reason", pd.Series("", index=x.index)).fillna("").astype(str).str.lower()
    dx = x.get("injury_diagnosis_class", pd.Series("unknown", index=x.index)).fillna("unknown").astype(str).str.lower()
    orthopedic_dx = ~dx.isin(["", "unknown", "nan", "none"])
    suspicious_illness = missed & (x["absence_reason_family"] == "illness") & orthopedic_dx

    tables["qc_reason_family_validation"] = pd.DataFrame([
        {"metric": "all-season injury absences", "value": all_injury, "note": "regular + play-in + playoffs"},
        {"metric": "all-season illness absences", "value": all_illness, "note": "regular + play-in + playoffs"},
        {"metric": "regular-season injury absences", "value": reg_injury, "note": "game_id prefix 002"},
        {"metric": "regular-season illness absences", "value": reg_illness, "note": "game_id prefix 002"},
        {"metric": "illness absences with parsed orthopedic diagnosis", "value": int(suspicious_illness.sum()), "note": "should be near zero after v5.2"},
        {"metric": "published reference injury missed games", "value": 6396, "note": "Professor-Pete/Basketball-Reference participation pipeline; reference only"},
        {"metric": "published reference illness missed games", "value": 328, "note": "Professor-Pete/Basketball-Reference participation pipeline; reference only"},
    ])

    # Replace the old 'capture ratio' framing: the reference uses a different
    # participation grid and is not a formal denominator for our all-season master.
    tables["qc_structural_completeness"] = pd.DataFrame([
        {"metric": "synthetic official-report-only rows added", "value": int(x.get("synthetic_report_only_row", pd.Series(0, index=x.index)).fillna(0).sum())},
        {"metric": "remaining undated games", "value": int(x.loc[x["game_date"].isna(), "game_id"].nunique())},
        {"metric": "all-season injury absences", "value": all_injury},
        {"metric": "regular-season injury absences", "value": reg_injury},
        {"metric": "all-season illness absences", "value": all_illness},
        {"metric": "regular-season illness absences", "value": reg_illness},
        {"metric": "published injury reference (different pipeline)", "value": 6396},
        {"metric": "published illness reference (different pipeline)", "value": 328},
    ])

    for name, df in tables.items():
        if isinstance(df, pd.DataFrame):
            df.to_csv(ROOT / f"{name}.csv", index=False)
    return tables


# -----------------------------------------------------------------------------
# 4) v5.2 runner with non-misleading manifest fields.
# -----------------------------------------------------------------------------
def run_maximal_pilot_2024_25(fetch_live=True, min_games_missed=2,
                              make_excel=True, deep_mode="off",
                              deep_scope="episode_windows"):
    season = "2024-25"
    reports = get_2425_official_injury_designations()
    print(f"Official injury designation rows: {len(reports):,}")

    pg = build_player_game_master_maximal(season, fetch_live=fetch_live, official_reports=reports)
    pg_path = ROOT / "player_game_master_2024_25_MAXIMAL_v5_2.csv"
    pg.to_csv(pg_path, index=False)
    print(f"Player-game rows: {len(pg):,} x {pg.shape[1]:,} columns -> {pg_path}")

    injury_ep = build_injury_episodes(pg, min_games_missed=min_games_missed, pre_games=10, post_games=20)
    injury_ep = add_player_age(injury_ep, season)
    injury_ep.to_csv(ROOT / "rtp_injury_episodes_2024_25_v5_2.csv", index=False)

    all_abs = build_all_absence_episodes(pg, min_team_games_missed=1)
    all_abs.to_csv(ROOT / "all_absence_episodes_2024_25_v5_2.csv", index=False)

    windows = build_episode_game_windows(pg, injury_ep, pre_played_games=10, post_team_games=20)
    windows.to_csv(ROOT / "rtp_episode_game_windows_2024_25_v5_2.csv", index=False)

    qc = build_v5_qc_tables(pg, injury_ep, reports)

    profiles = pd.DataFrame(); deep_audit = pd.DataFrame()
    if deep_mode in {"focused", "maximal"}:
        try:
            profiles = player_profile_archive(pg.loc[pg["player_id"] > 0, "player_id"].dropna().unique(), season=season)
        except Exception as e:
            print("Player profile archive skipped:", e)
        endpoint_names = FOCUSED_DEEP_ENDPOINTS if deep_mode == "focused" else MAXIMAL_DEEP_ENDPOINTS
        gids = episode_window_game_ids(windows) if deep_scope == "episode_windows" else pg["game_id"].dropna().astype(str).unique().tolist()
        print(f"Deep archive target: {len(gids):,} games x {len(endpoint_names)} endpoint families")
        deep_audit = archive_deep_endpoints(gids, endpoint_names)

    if make_excel:
        tables = {
            "player_game_master": pg,
            "injury_episodes": injury_ep,
            "all_absence_episodes": all_abs,
            "rtp_game_windows": windows,
            "player_profiles": profiles,
            "deep_archive_audit": deep_audit,
            **qc,
        }
        export_excel_bundle(tables, filename="NBA_Injury_RTP_Pilot_2024_25_v5_2.xlsx")

    gid = pg["game_id"].astype(str)
    reg = gid.str.startswith("002")
    missed = ~pg["played_actual"].fillna(False).astype(bool)
    all_games = int(pg["game_id"].nunique())
    dated_games = int(pg.loc[pg["game_date"].notna(), "game_id"].nunique())
    audit = {
        "season": season,
        "player_game_rows": int(len(pg)),
        "player_game_columns": int(pg.shape[1]),
        "unique_games": all_games,
        "dated_games": dated_games,
        "undated_games": int(all_games - dated_games),
        "synthetic_report_only_rows": int(pg["synthetic_report_only_row"].fillna(0).sum()),
        "injury_episode_rows": int(len(injury_ep)),
        "primary_analysis_eligible_episodes": int(injury_ep.get("primary_analysis_eligible", pd.Series(dtype=int)).fillna(0).sum()) if len(injury_ep) else 0,
        "strict_rotation_eligible_episodes": int(injury_ep.get("strict_rotation_analysis_eligible", pd.Series(dtype=int)).fillna(0).sum()) if len(injury_ep) else 0,
        "all_absence_episode_rows": int(len(all_abs)),
        "episode_window_rows": int(len(windows)),
        "deep_mode": deep_mode,
        "deep_scope": deep_scope,
        "all_season_injury_absence_rows": int((missed & (pg["absence_reason_family"] == "injury")).sum()),
        "regular_season_injury_absence_rows": int((missed & reg & (pg["absence_reason_family"] == "injury")).sum()),
        "all_season_illness_absence_rows": int((missed & (pg["absence_reason_family"] == "illness")).sum()),
        "regular_season_illness_absence_rows": int((missed & reg & (pg["absence_reason_family"] == "illness")).sum()),
        "not_on_report_absence_rows": int((missed & (pg["absence_reason_family"] == "not_on_report")).sum()),
        "official_report_matches": int(pg["official_report_matched"].fillna(False).sum()),
        "substantive_official_report_matches": int(pg["report_reason_substantive"].fillna(False).sum()),
        "published_reference_injury_missed_games": 6396,
        "published_reference_illness_missed_games": 328,
        "reference_note": "Published counts use a different Basketball-Reference participation pipeline; comparison only, not a capture denominator.",
    }
    (ROOT / "maximal_pilot_audit_2024_25_v5_2.json").write_text(json.dumps(audit, indent=2, default=str))
    print(json.dumps(audit, indent=2, default=str))
    return pg, injury_ep, all_abs, windows, profiles, deep_audit


# -----------------------------------------------------------------------------
# 5) Regression tests for the exact classifier failures found in the workbook.
# -----------------------------------------------------------------------------
def self_test_v52():
    assert reason_family("Injury/Illness - Right ACL; Tear") == "injury"
    assert reason_family("Injury/Illness - Right Fibula; Fracture") == "injury"
    assert reason_family("Injury/Illness - Left Plantar; Fasciitis") == "injury"
    assert reason_family("Injury/Illness - Right Patella; Tendinopathy") == "injury"
    assert reason_family("Injury/Illness - Right Thumb; Sprain") == "injury"
    assert reason_family("Injury/Illness - Non-Covid; Illness") == "illness"
    assert reason_family("Injury/Illness - Head; Cold") == "illness"
    assert reason_family("DNP_REST") == "rest"
    assert reason_family("G League - Two-Way") == "g_league"
    print("v5.2 self-test passed: orthopedic injury vs illness classification is corrected.")
    return True

self_test_v52()
print("NBA RTP v5.2 reason-family/QC fix loaded.")


## 5. Run self-tests

These tests check the core parsing logic before any network-heavy work starts.


In [ ]:
self_test()
self_test_v4()
self_test_v52()


## 6. Historical official injury reports + multi-season orchestration

For 2024–25, this notebook reuses the public parsed `season_rows.csv` used in the pilot. For earlier seasons it downloads official NBA injury-report PDFs from `ak-static.cms.nba.com`, **sequentially and cache-first**, then parses the ruled PDF tables with `pdfplumber`.

The historical fetch is intentionally conservative: sustained concurrent scraping can trigger CDN edge blocks. A known-good PDF is probed before a long fetch, every attempted slot is indexed on disk, and a blocked season fails cleanly without stopping the rest of the batch.


In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Optional
import datetime as dt
import json, re, time, traceback
import numpy as np
import pandas as pd
import requests
import pdfplumber

# -----------------------------------------------------------------------------
# Season-isolated paths. Existing validated functions read these globals.
# -----------------------------------------------------------------------------
def configure_season_paths(season: str):
    global ROOT, CACHE, RAW, DEEP_RAW, DEEP_CACHE
    ROOT = MULTI_ROOT / season
    CACHE = ROOT / "cache"
    RAW = ROOT / "raw"
    DEEP_RAW = ROOT / "deep_raw"
    DEEP_CACHE = CACHE / "deep_endpoints"
    for p in [ROOT, CACHE, RAW, DEEP_RAW, DEEP_CACHE]:
        p.mkdir(parents=True, exist_ok=True)
    return ROOT


def season_tag(season: str) -> str:
    return season.replace("-", "_")


# -----------------------------------------------------------------------------
# Historical NBA official injury-report PDF parser.
# Geometry follows the ruled-table strategy documented by Professor-Pete's
# MIT-licensed 2024-2025-injury_analysis parser; adapted here for batch use.
# -----------------------------------------------------------------------------
_IR_HEADER_TOKENS = {"GameDate", "GameTime", "Matchup", "Team", "PlayerName", "CurrentStatus", "Reason"}
_IR_CARRY = ("game_date", "game_time", "matchup", "team")


def _ir_clean(v):
    return re.sub(r"\s+", " ", (v or "").replace("\n", " ")).strip()


def _ir_x_grid(page):
    return sorted({round(e["x0"], 1) for e in page.edges
                   if e.get("orientation") == "v" and 95 < e.get("top", 0) < 130})


def _ir_y_grid(page):
    tops = sorted({round(e["top"], 1) for e in page.edges
                   if e.get("orientation") == "h" and e.get("top", 0) >= 75})
    merged = []
    for t in tops:
        if not merged or t - merged[-1] >= 2:
            merged.append(t)
    return merged


def parse_official_injury_pdf(path: Path):
    rows, carry, xs = [], dict.fromkeys(_IR_CARRY, ""), None
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            xs = _ir_x_grid(page) or xs
            ys = _ir_y_grid(page)
            if not xs or len(xs) < 8 or len(ys) < 2:
                continue
            table = page.extract_table({
                "vertical_strategy": "explicit",
                "explicit_vertical_lines": xs,
                "horizontal_strategy": "explicit",
                "explicit_horizontal_lines": ys,
                "text_x_tolerance": 1.5,
            }) or []
            for raw in table:
                cells = [_ir_clean(c) for c in raw]
                if len(cells) < 7 or not any(cells):
                    continue
                game_date, game_time, matchup, team, player, status, reason = cells[:7]
                # repeated page header
                if {v.replace(" ", "") for v in (game_date, player, status)} & _IR_HEADER_TOKENS:
                    continue
                for k, v in zip(_IR_CARRY, (game_date, game_time, matchup, team)):
                    if v:
                        carry[k] = v
                if not player and not status:
                    if "NOT YET SUBMITTED" in reason.upper():
                        rows.append({**carry, "player": "", "status": "NOT YET SUBMITTED", "reason": ""})
                    continue
                rows.append({**carry, "player": player, "status": status, "reason": reason})
    for r in rows:
        r["game_time"] = str(r.get("game_time", "")).replace("(ET)", "").strip()
    return rows


# -----------------------------------------------------------------------------
# Candidate snapshot dates: regular-season game dates (+ prior day) from the
# public team schedule, then every day through the season's postseason end.
# -----------------------------------------------------------------------------
def candidate_ir_snapshot_dates(season: str):
    teamlog = get_public_teamlog(season)
    sched = normalize_team_schedule(teamlog, season)
    game_dates = pd.to_datetime(sched["game_date"], errors="coerce").dropna().dt.normalize().unique()
    if len(game_dates) == 0:
        raise RuntimeError(f"{season}: no game dates found in public team log")
    dates = {pd.Timestamp(x).date() for x in game_dates}
    dates |= {d - dt.timedelta(days=1) for d in list(dates)}
    last_reg = max(pd.Timestamp(x).date() for x in game_dates)
    end = pd.Timestamp(SEASON_ARCHIVE_END[season]).date()
    d = last_reg + dt.timedelta(days=1)
    while d <= end:
        dates.add(d)
        dates.add(d - dt.timedelta(days=1))
        d += dt.timedelta(days=1)
    return sorted(dates)


def _ir_url(day, slot):
    return IR_ARCHIVE_BASE.format(date=str(day), slot=slot)


def _ir_fetch_one(session: requests.Session, day, slot, dest: Path):
    url = _ir_url(day, slot)
    try:
        r = session.get(url, headers={"User-Agent":"research-script", "Accept":"application/pdf,*/*"}, timeout=IR_TIMEOUT)
        body = r.content
        ok = r.status_code == 200 and body.startswith(b"%PDF") and len(body) > 2000
        if ok:
            tmp = dest.with_suffix(dest.suffix + ".tmp")
            tmp.write_bytes(body)
            tmp.replace(dest)
            return {"date":str(day), "slot":slot, "status":"pdf", "http_status":r.status_code, "bytes":len(body), "url":url}
        return {"date":str(day), "slot":slot, "status":"missing", "http_status":r.status_code, "bytes":len(body), "url":url}
    except Exception as e:
        return {"date":str(day), "slot":slot, "status":"error", "http_status":None, "bytes":0, "url":url, "error":repr(e)}


def ir_archive_probe(session=None):
    s = session or requests.Session()
    probe_path = RAW / "_ir_probe.pdf"
    rec = _ir_fetch_one(s, IR_PROBE_DATE, IR_PROBE_SLOT, probe_path)
    if probe_path.exists():
        try: probe_path.unlink()
        except Exception: pass
    return rec


def fetch_official_injury_pdfs(season: str):
    pdf_dir = RAW / "official_injury_pdfs"
    pdf_dir.mkdir(parents=True, exist_ok=True)
    index_path = RAW / "official_injury_fetch_index.csv"
    old = pd.read_csv(index_path) if index_path.exists() else pd.DataFrame()
    attempted = set()
    if len(old) and not IR_RECHECK_MISSING:
        # Only trust prior misses that were recorded while a known-good probe was
        # simultaneously reachable. This prevents a temporary CDN/WAF block from
        # poisoning the resume index with false "missing" slots.
        verified = old.get("verified_unblocked", pd.Series(False, index=old.index)).fillna(False).astype(bool)
        reusable = (old.get("status", pd.Series("", index=old.index)).astype(str) == "pdf") | verified
        attempted = set(zip(old.loc[reusable, "date"].astype(str), old.loc[reusable, "slot"].astype(str)))

    session = requests.Session()
    probe = ir_archive_probe(session)
    if probe.get("status") != "pdf":
        raise RuntimeError(
            f"Official NBA injury-report CDN probe failed before {season}: {probe}. "
            "This usually means the runtime/IP is temporarily edge-blocked."
        )

    dates = candidate_ir_snapshot_dates(season)
    jobs = []
    for day in dates:
        for slot in IR_SLOTS:
            dest = pdf_dir / f"{day}_{slot}.pdf"
            if dest.exists() and dest.stat().st_size > 2000:
                continue
            if (str(day), slot) in attempted:
                continue
            jobs.append((day, slot, dest))

    print(f"  injury-report archive {season}: {len(dates):,} candidate dates; {len(jobs):,} uncached slots")
    new = []
    batch_start = 0
    consecutive_errors = 0
    for i, (day, slot, dest) in enumerate(jobs, 1):
        rec = _ir_fetch_one(session, day, slot, dest)
        rec["verified_unblocked"] = False
        new.append(rec)
        if rec["status"] == "error":
            consecutive_errors += 1
        else:
            consecutive_errors = 0
        if i % IR_PROGRESS_EVERY == 0 or i == len(jobs):
            # Verify the CDN is still serving a known-good report before trusting
            # this batch's 403/missing results. Successful PDFs are safe regardless.
            probe2 = ir_archive_probe(session)
            batch_ok = probe2.get("status") == "pdf"
            for r in new[batch_start:i]:
                r["verified_unblocked"] = bool(batch_ok)
            merged = pd.concat([old, pd.DataFrame(new)], ignore_index=True) if len(old) else pd.DataFrame(new)
            merged.drop_duplicates(["date","slot"], keep="last").to_csv(index_path, index=False)
            ok = sum(r["status"] == "pdf" for r in new)
            print(f"    {i:,}/{len(jobs):,} attempted; new_pdfs={ok:,}; probe_ok={batch_ok}")
            batch_start = i
            if not batch_ok:
                raise RuntimeError(
                    f"Official injury-report CDN became blocked during {season}; "
                    f"successful PDFs are cached and unverified misses will be retried. Probe={probe2}"
                )
        if consecutive_errors >= 20:
            raise RuntimeError(f"{season}: 20 consecutive injury-report network errors; stopping safely")
        time.sleep(float(IR_POLITE_DELAY))
    pdfs = sorted(pdf_dir.glob("*.pdf"))
    print(f"  cached official PDFs for {season}: {len(pdfs):,}")
    return pdfs


def parse_cached_injury_pdfs(season: str, force=False):
    out_csv = RAW / f"official_injury_designations_{season_tag(season)}.csv"
    if out_csv.exists() and not force:
        return pd.read_csv(out_csv, low_memory=False)
    pdfs = sorted((RAW / "official_injury_pdfs").glob("*.pdf"))
    if not pdfs:
        raise RuntimeError(f"{season}: no cached injury-report PDFs to parse")
    all_rows, errors = [], []
    for i, path in enumerate(pdfs, 1):
        stem = path.stem
        day, slot = stem.rsplit("_", 1)
        try:
            rows = parse_official_injury_pdf(path)
            all_rows.extend([{"snapshot_date":day, "snapshot_time":slot, **r} for r in rows])
        except Exception as e:
            errors.append({"file":path.name, "error":repr(e)})
        if i % 100 == 0 or i == len(pdfs):
            print(f"    parsed {i:,}/{len(pdfs):,} PDFs; designation_rows={len(all_rows):,}; errors={len(errors):,}")
    df = pd.DataFrame(all_rows, columns=[
        "snapshot_date","snapshot_time","game_date","game_time","matchup","team","player","status","reason"
    ])
    df.to_csv(out_csv, index=False)
    if errors:
        pd.DataFrame(errors).to_csv(RAW / "official_injury_parse_errors.csv", index=False)
    return df


def get_official_injury_designations_multiseason(season: str):
    if season == "2024-25" and USE_PREPARSED_2425:
        df = get_2425_official_injury_designations()
        # Save a season-local copy so the source layer is reproducible beside outputs.
        p = RAW / f"official_injury_designations_{season_tag(season)}.csv"
        if not p.exists():
            df.to_csv(p, index=False)
        return df
    parsed = RAW / f"official_injury_designations_{season_tag(season)}.csv"
    if parsed.exists():
        return pd.read_csv(parsed, low_memory=False)
    fetch_official_injury_pdfs(season)
    return parse_cached_injury_pdfs(season)


# -----------------------------------------------------------------------------
# Generic season QC. Avoids the 2024-25 benchmark-specific pilot functions.
# -----------------------------------------------------------------------------
def build_multiseason_qc_tables(season: str, pg: pd.DataFrame, injury_ep: pd.DataFrame,
                                reports_raw: pd.DataFrame):
    x = pg.copy()
    x["game_id"] = x["game_id"].astype(str)
    x["game_date"] = pd.to_datetime(x["game_date"], errors="coerce").dt.normalize()
    missed = ~x["played_actual"].fillna(False).astype(bool)

    rrn = normalize_official_injury_reports(reports_raw)
    rrc = collapse_reports_to_player_game(rrn)

    # Map report team/date to the observed season master; postponed / non-counting
    # games naturally remain unmatched and are exposed for review.
    gmap = x[["game_id","game_date","team"]].dropna(subset=["game_date","team"]).drop_duplicates()
    counts = gmap.groupby(["game_date","team"])["game_id"].nunique().rename("_n").reset_index()
    gmap = gmap.merge(counts, on=["game_date","team"], how="left")
    gmap = gmap[gmap["_n"] == 1].drop(columns="_n").drop_duplicates(["game_date","team"])
    rg = rrc.merge(gmap, on=["game_date","team"], how="left")
    keys = x[["game_id","team","player_key"]].drop_duplicates().assign(_master=1)
    rg2 = rg.merge(keys, on=["game_id","team","player_key"], how="left")
    unmatched = rg2[rg2["_master"].isna()].drop(columns=["_master"], errors="ignore")

    team = x.groupby("team", as_index=False).agg(
        player_game_rows=("game_id","size"),
        games=("game_id","nunique"),
        synthetic_report_only_rows=("synthetic_report_only_row","sum"),
        official_report_matches=("official_report_matched","sum"),
        injury_absence_rows=("absence_reason_family", lambda s: int((s == "injury").sum())),
        illness_absence_rows=("absence_reason_family", lambda s: int((s == "illness").sum())),
        not_on_report_rows=("absence_reason_family", lambda s: int((s == "not_on_report").sum())),
    ).sort_values("injury_absence_rows")
    team["injury_zero_warning"] = (team["injury_absence_rows"] == 0).astype(int)

    dx = x.get("injury_diagnosis_class", pd.Series("unknown", index=x.index)).fillna("unknown").astype(str).str.lower()
    suspicious_illness = missed & (x["absence_reason_family"] == "illness") & ~dx.isin(["", "unknown", "nan", "none"])

    unique_games = int(x["game_id"].nunique())
    dated_games = int(x.loc[x["game_date"].notna(), "game_id"].nunique())
    team_count = int(x["team"].nunique())
    mapped_report_rate = float(rg["game_id"].notna().mean()) if len(rg) else np.nan
    unmatched_rate = float(len(unmatched) / len(rrc)) if len(rrc) else np.nan
    primary_n = int(injury_ep.get("primary_analysis_eligible", pd.Series(dtype=int)).fillna(0).sum()) if len(injury_ep) else 0

    gates = [
        ("all player-log games dated", unique_games == dated_games, f"{dated_games}/{unique_games}"),
        ("30 NBA teams represented", team_count == 30, str(team_count)),
        ("no team has zero injury absences", int(team["injury_zero_warning"].sum()) == 0, str(int(team["injury_zero_warning"].sum()))),
        ("official report team/date mapping >=97%", (mapped_report_rate >= 0.97) if pd.notna(mapped_report_rate) else False, f"{mapped_report_rate:.3%}" if pd.notna(mapped_report_rate) else "NA"),
        ("unmatched collapsed report keys <=2%", (unmatched_rate <= 0.02) if pd.notna(unmatched_rate) else False, f"{unmatched_rate:.3%}" if pd.notna(unmatched_rate) else "NA"),
        ("orthopedic-looking illness rows <=5", int(suspicious_illness.sum()) <= 5, str(int(suspicious_illness.sum()))),
        (">=50 primary RTP episodes", primary_n >= 50, str(primary_n)),
    ]
    gate_df = pd.DataFrame(gates, columns=["gate","pass","observed"])
    overall = "PASS" if bool(gate_df["pass"].all()) else "REVIEW"

    summary = pd.DataFrame([{
        "season": season,
        "qc_status": overall,
        "player_game_rows": len(x),
        "unique_games": unique_games,
        "dated_games": dated_games,
        "undated_games": unique_games - dated_games,
        "teams": team_count,
        "official_designation_rows_raw": len(reports_raw),
        "collapsed_report_player_games": len(rrc),
        "report_game_mapping_rate": mapped_report_rate,
        "unmatched_collapsed_report_keys": len(unmatched),
        "unmatched_report_rate": unmatched_rate,
        "synthetic_report_only_rows": int(x["synthetic_report_only_row"].fillna(0).sum()),
        "injury_absence_rows": int((missed & (x["absence_reason_family"] == "injury")).sum()),
        "illness_absence_rows": int((missed & (x["absence_reason_family"] == "illness")).sum()),
        "not_on_report_absence_rows": int((missed & (x["absence_reason_family"] == "not_on_report")).sum()),
        "injury_episode_rows": len(injury_ep),
        "primary_analysis_eligible_episodes": primary_n,
        "strict_rotation_eligible_episodes": int(injury_ep.get("strict_rotation_analysis_eligible", pd.Series(dtype=int)).fillna(0).sum()) if len(injury_ep) else 0,
        "suspicious_orthopedic_illness_rows": int(suspicious_illness.sum()),
    }])

    schedule_missing = x.loc[x["game_date"].isna(), [c for c in ["game_id","team","player_id","player_name","comment"] if c in x.columns]].drop_duplicates()
    tables = {
        "qc_summary": summary,
        "qc_gates": gate_df,
        "qc_team_structural_coverage": team,
        "qc_unmatched_report_keys": unmatched,
        "qc_schedule_missing_games": schedule_missing,
    }
    for name, df in tables.items():
        df.to_csv(ROOT / f"{name}.csv", index=False)
    return overall, tables


# -----------------------------------------------------------------------------
# One-season production runner.
# -----------------------------------------------------------------------------
def run_one_season(season: str, fetch_live: bool = FETCH_LIVE,
                   deep_mode: str = DEEP_MODE, deep_scope: str = DEEP_SCOPE):
    configure_season_paths(season)
    manifest_path = ROOT / "season_manifest.json"
    if (SKIP_COMPLETED_PASS_SEASONS and season not in FORCE_RERUN_SEASONS and manifest_path.exists()):
        try:
            old = json.loads(manifest_path.read_text())
            if old.get("qc_status") == "PASS":
                print(f"[{season}] already PASS; skipping (set FORCE_RERUN_SEASONS to rerun).")
                return old
        except Exception:
            pass

    started = pd.Timestamp.utcnow().isoformat()
    print("\n" + "="*78)
    print(f"[{season}] START")
    print("="*78)
    try:
        reports = get_official_injury_designations_multiseason(season)
        print(f"[{season}] official injury designation rows: {len(reports):,}")

        pg = build_player_game_master_maximal(season, fetch_live=fetch_live, official_reports=reports)
        injury_ep = build_injury_episodes(pg, min_games_missed=MIN_GAMES_MISSED, pre_games=10, post_games=20)
        injury_ep = add_player_age(injury_ep, season)
        all_abs = build_all_absence_episodes(pg, min_team_games_missed=1)
        windows = build_episode_game_windows(pg, injury_ep, pre_played_games=10, post_team_games=20)

        # Parquet-first production outputs.
        pg.to_parquet(ROOT / "player_game_master.parquet", index=False)
        injury_ep.to_parquet(ROOT / "injury_episodes.parquet", index=False)
        all_abs.to_parquet(ROOT / "all_absence_episodes.parquet", index=False)
        windows.to_parquet(ROOT / "rtp_episode_game_windows.parquet", index=False)
        if WRITE_SEASON_CSV:
            pg.to_csv(ROOT / "player_game_master.csv", index=False)
            injury_ep.to_csv(ROOT / "injury_episodes.csv", index=False)
            all_abs.to_csv(ROOT / "all_absence_episodes.csv", index=False)
            windows.to_csv(ROOT / "rtp_episode_game_windows.csv", index=False)

        qc_status, qc_tables = build_multiseason_qc_tables(season, pg, injury_ep, reports)

        if WRITE_SEASON_EXCEL:
            excel_tables = {
                "injury_episodes": injury_ep,
                "all_absence_episodes": all_abs,
                "rtp_game_windows": windows,
                **qc_tables,
            }
            export_excel_bundle(excel_tables, filename=f"NBA_Injury_RTP_{season_tag(season)}_QC.xlsx")

        if deep_mode in {"focused", "maximal"}:
            endpoint_names = FOCUSED_DEEP_ENDPOINTS if deep_mode == "focused" else MAXIMAL_DEEP_ENDPOINTS
            gids = episode_window_game_ids(windows) if deep_scope == "episode_windows" else pg["game_id"].dropna().astype(str).unique().tolist()
            archive_deep_endpoints(gids, endpoint_names)

        summary = qc_tables["qc_summary"].iloc[0].to_dict()
        manifest = {
            **summary,
            "season": season,
            "qc_status": qc_status,
            "started_utc": started,
            "finished_utc": pd.Timestamp.utcnow().isoformat(),
            "deep_mode": deep_mode,
            "deep_scope": deep_scope,
            "error": None,
        }
        manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
        print(f"[{season}] DONE — QC {qc_status}")
        print(json.dumps(manifest, indent=2, default=str))
        return manifest

    except Exception as e:
        manifest = {
            "season": season,
            "qc_status": "FAILED",
            "started_utc": started,
            "finished_utc": pd.Timestamp.utcnow().isoformat(),
            "error": repr(e),
            "traceback": traceback.format_exc(),
        }
        manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
        print(f"[{season}] FAILED: {e}")
        return manifest


def run_multiseason(seasons=None):
    seasons = list(seasons or SEASONS)
    manifests = []
    for season in seasons:
        manifests.append(run_one_season(season))
    summary = pd.DataFrame(manifests)
    summary.to_csv(MULTI_ROOT / "batch_manifest.csv", index=False)
    display_cols = [c for c in [
        "season","qc_status","unique_games","dated_games","injury_absence_rows",
        "injury_episode_rows","primary_analysis_eligible_episodes",
        "report_game_mapping_rate","unmatched_report_rate","error"
    ] if c in summary.columns]
    display(summary[display_cols])
    return summary


## 6B. FAST historical archive patch (v1.2)

Overrides the historical downloader with modest rate-capped concurrency. Individual archive-slot HTTP 403 responses are treated as provisional missing reports and validated by a known-good health probe; only HTTP 429 or a failed health probe triggers slowdown/stop. Existing v1/v1.1 caches are reused.


In [ ]:
"""
NBA RTP historical injury-report speed patch v1.2

Load AFTER Section 6 of NBA_Injury_RTP_MultiSeason_v1_1_FAST.ipynb (or v1)
and BEFORE the batch/run cell. It preserves the same season-isolated cache/index
paths, so interrupted v1/v1.1 runs resume from already downloaded PDFs.

Why v1.2 exists:
  * The NBA archive often returns HTTP 403 for an individual date/time slot that
    simply has no PDF, while a known-good archive PDF remains reachable.
  * v1.1 treated every slot-level 403 as global throttling and ratcheted itself
    to a 3-second delay even while the health probe stayed green.

v1.2 fixes that and uses modest, rate-capped concurrency:
  * 4 download workers by default;
  * globally spaced request starts (~4.5 starts/sec by default);
  * target-slot 403 is provisional_missing, NOT automatic throttling;
  * HTTP 429 remains a true rate-limit signal;
  * known-good CDN probe after each small batch validates provisional misses;
  * if the probe fails, successful PDFs stay cached and the run stops safely;
  * old v1.1 blocked/403 index rows are retried, while verified misses/PDFs are reused;
  * concurrent local PDF parsing remains enabled.
"""

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import time
import pandas as pd
import requests

# ------------------------------ v1.2 knobs ----------------------------------
IR_DL_WORKERS = 4
IR_START_INTERVAL = 0.22       # global spacing between request starts (~4.5/s)
IR_MIN_START_INTERVAL = 0.16
IR_MAX_START_INTERVAL = 1.50
IR_BATCH_SIZE = 40             # health-probe/checkpoint cadence
IR_CHECKPOINT_EVERY_BATCH = True
IR_PARSE_WORKERS = 4
IR_MAX_429_RETRIES = 2
IR_429_BACKOFF_MULTIPLIER = 1.7
IR_HEALTHY_RECOVERY_MULTIPLIER = 0.92

# Compatibility values used by other notebook cells / display code.
IR_POLITE_DELAY = 0.0
IR_PROGRESS_EVERY = IR_BATCH_SIZE


class _IRStartLimiter:
    def __init__(self, interval):
        self.interval = float(interval)
        self.lock = threading.Lock()
        self.next_start = 0.0

    def set_interval(self, interval):
        with self.lock:
            self.interval = float(interval)

    def wait(self):
        with self.lock:
            now = time.monotonic()
            start = max(now, self.next_start)
            self.next_start = start + self.interval
            wait = start - now
        if wait > 0:
            time.sleep(wait)


_IR_LIMITER_V12 = _IRStartLimiter(IR_START_INTERVAL)
_TLS_V12 = threading.local()


def _v12_session():
    s = getattr(_TLS_V12, "session", None)
    if s is None:
        s = requests.Session()
        s.headers.update({
            "User-Agent": "Mozilla/5.0 research-use",
            "Accept": "application/pdf,*/*",
            "Referer": "https://official.nba.com/",
            "Connection": "keep-alive",
        })
        _TLS_V12.session = s
    return s


def _ir_fetch_one_v12(day, slot, dest):
    """Fetch one candidate slot; 403 is ambiguous/provisional, 429 is throttling."""
    _IR_LIMITER_V12.wait()
    url = _ir_url(day, slot)
    try:
        r = _v12_session().get(url, timeout=IR_TIMEOUT)
        body = r.content
        is_pdf = r.status_code == 200 and body.startswith(b"%PDF") and len(body) > 2000
        if is_pdf:
            tmp = dest.with_suffix(dest.suffix + ".tmp")
            tmp.write_bytes(body)
            tmp.replace(dest)
            return {
                "date": str(day), "slot": slot, "status": "pdf",
                "http_status": int(r.status_code), "bytes": len(body), "url": url,
                "verified_unblocked": True,
            }

        if r.status_code == 429:
            status = "rate_limited"
        elif r.status_code == 403:
            # Important: this archive commonly uses 403 for a nonexistent slot.
            # We validate these at batch level with a known-good PDF probe.
            status = "provisional_missing"
        else:
            status = "missing"

        return {
            "date": str(day), "slot": slot, "status": status,
            "http_status": int(r.status_code), "bytes": len(body), "url": url,
            "verified_unblocked": False,
        }
    except Exception as e:
        return {
            "date": str(day), "slot": slot, "status": "error",
            "http_status": None, "bytes": 0, "url": url,
            "verified_unblocked": False, "error": repr(e),
        }


def _v12_atomic_index(index_path, old, new):
    if len(old) == 0 and len(new) == 0:
        return
    merged = pd.concat([old, pd.DataFrame(new)], ignore_index=True, sort=False) if len(old) else pd.DataFrame(new)
    merged = merged.drop_duplicates(["date", "slot"], keep="last")
    tmp = index_path.with_suffix(index_path.suffix + ".tmp")
    merged.to_csv(tmp, index=False)
    tmp.replace(index_path)


def _v12_reusable_keys(old):
    if old is None or len(old) == 0:
        return set()
    status = old.get("status", pd.Series("", index=old.index)).astype(str)
    verified = old.get("verified_unblocked", pd.Series(False, index=old.index)).fillna(False).astype(bool)
    http = pd.to_numeric(old.get("http_status", pd.Series(float("nan"), index=old.index)), errors="coerce")

    # PDF is reusable (disk existence is checked separately too). Verified missing is reusable.
    # v1.1 'blocked' / 403 rows are deliberately NOT reusable and will be retried.
    reusable = (status == "pdf") | ((status == "missing") & verified & ~http.eq(429))
    return set(zip(old.loc[reusable, "date"].astype(str), old.loc[reusable, "slot"].astype(str)))


def _v12_probe():
    # Use the notebook's known-good probe, but with a fresh sequential session so
    # worker-session state cannot influence the result.
    s = requests.Session()
    return ir_archive_probe(s)


def _v12_fetch_batch(batch):
    """Concurrent fetch of one small batch; returns results in completion order."""
    out = []
    with ThreadPoolExecutor(max_workers=int(IR_DL_WORKERS)) as ex:
        futs = {ex.submit(_ir_fetch_one_v12, day, slot, dest): (day, slot, dest)
                for day, slot, dest in batch}
        for fut in as_completed(futs):
            try:
                out.append(fut.result())
            except Exception as e:
                day, slot, _ = futs[fut]
                out.append({
                    "date": str(day), "slot": slot, "status": "error",
                    "http_status": None, "bytes": 0, "url": _ir_url(day, slot),
                    "verified_unblocked": False, "error": repr(e),
                })
    return out


def fetch_official_injury_pdfs(season: str):
    """Fast, resume-safe historical NBA injury-report downloader."""
    pdf_dir = RAW / "official_injury_pdfs"
    pdf_dir.mkdir(parents=True, exist_ok=True)
    index_path = RAW / "official_injury_fetch_index.csv"
    old = pd.read_csv(index_path) if index_path.exists() else pd.DataFrame()
    attempted = set() if IR_RECHECK_MISSING else _v12_reusable_keys(old)

    probe = _v12_probe()
    if probe.get("status") != "pdf":
        raise RuntimeError(
            f"Official NBA injury-report CDN probe failed before {season}: {probe}. "
            "Existing cached PDFs are preserved; retry later/from another runtime."
        )

    dates = candidate_ir_snapshot_dates(season)
    jobs = []
    cached_pdf_count = 0
    for day in dates:
        for slot in IR_SLOTS:
            dest = pdf_dir / f"{day}_{slot}.pdf"
            if dest.exists() and dest.stat().st_size > 2000:
                cached_pdf_count += 1
                continue
            if (str(day), slot) in attempted:
                continue
            jobs.append((day, slot, dest))

    print(
        f"  injury-report archive {season}: {len(dates):,} candidate dates; "
        f"cached_pdfs={cached_pdf_count:,}; uncached_slots={len(jobs):,}; "
        f"workers={IR_DL_WORKERS}; start_interval={IR_START_INTERVAL:.2f}s"
    )
    if not jobs:
        pdfs = sorted(pdf_dir.glob("*.pdf"))
        print(f"  cached official PDFs for {season}: {len(pdfs):,}")
        return pdfs

    new = []
    start = time.monotonic()
    done = 0
    pdf_count = 0
    miss_count = 0
    http403_count = 0
    rate_limit_count = 0
    error_count = 0
    current_interval = float(IR_START_INTERVAL)
    retry_429 = []

    # Process small batches so a health failure cannot invalidate a huge tail.
    for b0 in range(0, len(jobs), int(IR_BATCH_SIZE)):
        batch = jobs[b0:b0 + int(IR_BATCH_SIZE)]
        _IR_LIMITER_V12.set_interval(current_interval)
        results = _v12_fetch_batch(batch)

        # Probe AFTER the batch. This is the decisive global health signal.
        probe2 = _v12_probe()
        batch_ok = probe2.get("status") == "pdf"

        batch_429 = 0
        for r in results:
            if r.get("status") == "pdf":
                pdf_count += 1
                r["verified_unblocked"] = True
            elif r.get("status") == "provisional_missing":
                http403_count += 1
                if batch_ok:
                    r["status"] = "missing"
                    r["verified_unblocked"] = True
                    miss_count += 1
            elif r.get("status") == "missing":
                if batch_ok:
                    r["verified_unblocked"] = True
                    miss_count += 1
            elif r.get("status") == "rate_limited":
                rate_limit_count += 1
                batch_429 += 1
                retry_429.append(r)
            elif r.get("status") == "error":
                error_count += 1
            new.append(r)

        done += len(batch)
        _v12_atomic_index(index_path, old, new)

        if not batch_ok:
            raise RuntimeError(
                f"Official injury-report CDN health probe failed during {season}. "
                f"Successful PDFs are cached; provisional misses in this batch will be retried. Probe={probe2}"
            )

        # Only a real 429 slows the controller. Plain slot-level 403s do NOT.
        if batch_429:
            current_interval = min(
                IR_MAX_START_INTERVAL,
                max(current_interval * IR_429_BACKOFF_MULTIPLIER, 0.35),
            )
        else:
            current_interval = max(
                IR_MIN_START_INTERVAL,
                current_interval * IR_HEALTHY_RECOVERY_MULTIPLIER,
            )

        elapsed = max(time.monotonic() - start, 1e-9)
        rate = done / elapsed
        remaining = len(jobs) - done
        eta_min = remaining / max(rate, 1e-9) / 60.0
        print(
            f"    {done:,}/{len(jobs):,} attempted; new_pdfs={pdf_count:,}; "
            f"missing={miss_count:,} (403={http403_count:,}); 429={rate_limit_count:,}; "
            f"errors={error_count:,}; rate={rate:.2f}/s; interval={current_interval:.2f}s; "
            f"ETA≈{eta_min:.1f} min; probe_ok=True"
        )

    # Retry actual 429s only, with a slower interval; do not re-download successful/missing slots.
    if retry_429:
        unique_retry = {}
        for r in retry_429:
            dest = pdf_dir / f"{r['date']}_{r['slot']}.pdf"
            if not dest.exists():
                unique_retry[(r["date"], r["slot"])] = (r["date"], r["slot"], dest)
        pending = list(unique_retry.values())
        for attempt in range(int(IR_MAX_429_RETRIES)):
            if not pending:
                break
            current_interval = min(IR_MAX_START_INTERVAL, max(current_interval, 0.5))
            _IR_LIMITER_V12.set_interval(current_interval)
            time.sleep(min(3.0, 1.0 + attempt))
            results = _v12_fetch_batch(pending)
            probe2 = _v12_probe()
            if probe2.get("status") != "pdf":
                _v12_atomic_index(index_path, old, new)
                break
            pending2 = []
            for r in results:
                if r.get("status") == "pdf":
                    r["verified_unblocked"] = True
                elif r.get("status") == "provisional_missing":
                    r["status"] = "missing"; r["verified_unblocked"] = True
                elif r.get("status") == "missing":
                    r["verified_unblocked"] = True
                elif r.get("status") == "rate_limited":
                    dest = pdf_dir / f"{r['date']}_{r['slot']}.pdf"
                    pending2.append((r["date"], r["slot"], dest))
                new.append(r)
            _v12_atomic_index(index_path, old, new)
            pending = pending2
            current_interval = min(IR_MAX_START_INTERVAL, current_interval * IR_429_BACKOFF_MULTIPLIER)

    _v12_atomic_index(index_path, old, new)
    pdfs = sorted(pdf_dir.glob("*.pdf"))
    print(f"  cached official PDFs for {season}: {len(pdfs):,}")
    return pdfs


# Keep / replace the fast concurrent local parser from v1.1.
def _parse_one_ir_pdf_v12(path):
    stem = path.stem
    day, slot = stem.rsplit("_", 1)
    try:
        rows = parse_official_injury_pdf(path)
        return [{"snapshot_date": day, "snapshot_time": slot, **r} for r in rows], None
    except Exception as e:
        return [], {"file": path.name, "error": repr(e)}


def parse_cached_injury_pdfs(season: str, force=False):
    out_csv = RAW / f"official_injury_designations_{season_tag(season)}.csv"
    if out_csv.exists() and not force:
        return pd.read_csv(out_csv, low_memory=False)

    pdfs = sorted((RAW / "official_injury_pdfs").glob("*.pdf"))
    if not pdfs:
        raise RuntimeError(f"{season}: no cached injury-report PDFs to parse")

    all_rows, errors = [], []
    done = 0
    with ThreadPoolExecutor(max_workers=int(IR_PARSE_WORKERS)) as ex:
        futs = {ex.submit(_parse_one_ir_pdf_v12, p): p for p in pdfs}
        for fut in as_completed(futs):
            rows, err = fut.result()
            all_rows.extend(rows)
            if err is not None:
                errors.append(err)
            done += 1
            if done % 100 == 0 or done == len(pdfs):
                print(
                    f"    parsed {done:,}/{len(pdfs):,} PDFs; "
                    f"designation_rows={len(all_rows):,}; errors={len(errors):,}"
                )

    df = pd.DataFrame(all_rows, columns=[
        "snapshot_date", "snapshot_time", "game_date", "game_time", "matchup",
        "team", "player", "status", "reason"
    ])
    if len(df):
        df = df.sort_values(["snapshot_date", "snapshot_time", "game_date", "team", "player"], na_position="last")
    df.to_csv(out_csv, index=False)
    pd.DataFrame(errors).to_csv(RAW / f"official_injury_pdf_parse_errors_{season_tag(season)}.csv", index=False)
    print(f"  parsed injury designations {season}: {len(df):,} rows -> {out_csv.name}")
    return df


print("NBA RTP historical speed patch v1.2 loaded.")
print(
    f"  downloader: {IR_DL_WORKERS} workers, globally rate-capped at ~{1/IR_START_INTERVAL:.1f} starts/s, "
    f"batch probe every {IR_BATCH_SIZE} slots"
)
print("  slot-level HTTP 403 is treated as provisional missing; only a failed health probe or 429 triggers slowdown.")
print(f"  local PDF parser workers: {IR_PARSE_WORKERS}")
print("  Existing v1/v1.1 caches are reused; stop the old run without resetting the runtime.")


## 6C. v2 historical player-game source fix

For **2021–22 through 2023–24**, this layer replaces the sparse per-season NBA player-log parquet with ESPN-backed data. It builds a complete event/date/home-away map from the ESPN scoreboard and joins that to the repository’s aggregate ESPN `player_box.parquet`. The validated 2024–25 NBA player-log path is unchanged.

The source layer is schema-aware and adds a hard QC gate requiring ≥95% observed game coverage before a historical season can pass.


In [ ]:
# ==================== v2 HISTORICAL SOURCE FIX ====================
# Historical player-game source for 2021-22 through 2023-24:
# ESPN scoreboard (date/home-away/event IDs) + llimllib ESPN player_box parquet.
# 2024-25 keeps the validated NBA playerlog source.

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path

# Preserve validated source functions before overriding them.
_GET_PUBLIC_PLAYERLOG_MAIN_V2 = get_public_playerlog
_GET_PUBLIC_TEAMLOG_MAIN_V2 = get_public_teamlog
_ADD_PLAYER_AGE_MAIN_V2 = add_player_age
_GAME_TYPE_MAIN_V2 = _game_type_from_id
_BUILD_MULTI_QC_MAIN_V2 = build_multiseason_qc_tables

ESPN_SCOREBOARD_URL = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard"
ESPN_PLAYER_BOX_URL = f"{LLIMLLIB_RAW}/espn/player_box.parquet"

_ESPN_GAME_TYPE_MAP_V2 = {}
_ESPN_SCOREBOARD_RATE_LOCK_V2 = threading.Lock()
_ESPN_SCOREBOARD_NEXT_START_V2 = 0.0

# ESPN sometimes uses shorter abbreviations than NBA.com.
_ESPN_ABBR_TO_NBA_V2 = {
    "ATL":"ATL", "BOS":"BOS", "BKN":"BKN", "BRK":"BKN",
    "CHA":"CHA", "CHI":"CHI", "CLE":"CLE", "DAL":"DAL", "DEN":"DEN",
    "DET":"DET", "GS":"GSW", "GSW":"GSW", "HOU":"HOU", "IND":"IND",
    "LAC":"LAC", "LAL":"LAL", "MEM":"MEM", "MIA":"MIA", "MIL":"MIL",
    "MIN":"MIN", "NO":"NOP", "NOP":"NOP", "NY":"NYK", "NYK":"NYK",
    "OKC":"OKC", "ORL":"ORL", "PHI":"PHI", "PHX":"PHX", "POR":"POR",
    "SAC":"SAC", "SA":"SAS", "SAS":"SAS", "TOR":"TOR", "UTA":"UTA",
    "WSH":"WAS", "WAS":"WAS",
}


def _espn_team_to_nba_v2(team_obj):
    """Map an ESPN team dict/string to the NBA 3-letter code used by the pilot."""
    if isinstance(team_obj, dict):
        full = str(team_obj.get("displayName") or team_obj.get("name") or "").strip()
        abbr = str(team_obj.get("abbreviation") or "").strip().upper()
        if full in TEAM_NAME_TO_ABBR:
            return TEAM_NAME_TO_ABBR[full]
        if abbr in _ESPN_ABBR_TO_NBA_V2:
            return _ESPN_ABBR_TO_NBA_V2[abbr]
        return abbr or None
    s = str(team_obj or "").strip()
    if s in TEAM_NAME_TO_ABBR:
        return TEAM_NAME_TO_ABBR[s]
    su = s.upper()
    if su in _ESPN_ABBR_TO_NBA_V2:
        return _ESPN_ABBR_TO_NBA_V2[su]
    # Normalize punctuation/case for full-name fallbacks.
    sn = re.sub(r"\s+", " ", s).strip().lower()
    for full, abbr in TEAM_NAME_TO_ABBR.items():
        if full.lower() == sn:
            return abbr
    return su if 2 <= len(su) <= 4 else None


def _espn_wait_scoreboard_slot_v2():
    global _ESPN_SCOREBOARD_NEXT_START_V2
    interval = float(globals().get("ESPN_SCOREBOARD_START_INTERVAL", 0.06))
    with _ESPN_SCOREBOARD_RATE_LOCK_V2:
        now = time.monotonic()
        slot = max(now, _ESPN_SCOREBOARD_NEXT_START_V2)
        _ESPN_SCOREBOARD_NEXT_START_V2 = slot + interval
        wait = slot - now
    if wait > 0:
        time.sleep(wait)


def _fetch_espn_scoreboard_day_v2(day, cache_dir: Path, retries=3):
    day = pd.Timestamp(day).normalize()
    key = day.strftime("%Y%m%d")
    path = cache_dir / f"{key}.json"
    if path.exists() and path.stat().st_size > 100:
        try:
            return json.loads(path.read_text())
        except Exception:
            pass
    last = None
    for attempt in range(retries):
        try:
            _espn_wait_scoreboard_slot_v2()
            r = requests.get(
                ESPN_SCOREBOARD_URL,
                params={"dates": key, "limit": 100},
                headers={"User-Agent": HEADERS.get("User-Agent", "Mozilla/5.0"), "Accept":"application/json"},
                timeout=(5, 20),
            )
            r.raise_for_status()
            js = r.json()
            tmp = path.with_suffix(".json.tmp")
            tmp.write_text(json.dumps(js))
            tmp.replace(path)
            return js
        except Exception as e:
            last = e
            time.sleep(0.5 * (attempt + 1))
    raise RuntimeError(f"ESPN scoreboard failed for {key}: {last}")


def _event_game_type_v2(event):
    season_obj = event.get("season", {}) or {}
    st = season_obj.get("type")
    name = str(season_obj.get("slug") or season_obj.get("name") or "").lower()
    if st == 1 or "pre" in name:
        return "preseason"
    if st == 3 or "post" in name or "playoff" in name:
        return "postseason"
    if st == 2:
        return "regular_season"
    return "unknown"


def _parse_espn_scoreboard_events_v2(js, requested_day, season):
    rows = []
    requested_day = pd.Timestamp(requested_day).normalize()
    for event in js.get("events", []) or []:
        comps = event.get("competitions", []) or []
        if not comps:
            continue
        comp = comps[0] or {}
        competitors = comp.get("competitors", []) or []
        if len(competitors) != 2:
            continue
        home = next((c for c in competitors if str(c.get("homeAway", "")).lower() == "home"), None)
        away = next((c for c in competitors if str(c.get("homeAway", "")).lower() == "away"), None)
        if home is None or away is None:
            continue
        home_abbr = _espn_team_to_nba_v2(home.get("team", {}))
        away_abbr = _espn_team_to_nba_v2(away.get("team", {}))
        if not home_abbr or not away_abbr:
            continue
        gid = str(event.get("id") or comp.get("id") or "").strip()
        if not gid:
            continue
        gtype = _event_game_type_v2(event)
        if gtype == "preseason":
            continue
        _ESPN_GAME_TYPE_MAP_V2[gid] = gtype
        rows.append({
            "season": season,
            "game_id": gid,
            "game_date": requested_day,
            "home_team": home_abbr,
            "away_team": away_abbr,
            "game_type_espn": gtype,
            "espn_event_id": gid,
            "event_name": event.get("name"),
        })
    return rows


def build_espn_full_schedule_v2(season: str, force=False):
    """Full regular + postseason ESPN schedule for one historical season."""
    if season not in HISTORICAL_ESPN_SEASONS:
        raise ValueError(f"ESPN historical schedule not configured for {season}")
    cache = RAW / "espn_full_schedule.parquet"
    if cache.exists() and not force:
        out = pd.read_parquet(cache)
        for r in out[["game_id","game_type_espn"]].drop_duplicates().itertuples(index=False):
            _ESPN_GAME_TYPE_MAP_V2[str(r.game_id)] = str(r.game_type_espn)
        return out

    # Use the reliable NBA team log only to locate the beginning of the regular season.
    raw_team = _GET_PUBLIC_TEAMLOG_MAIN_V2(season)
    reg_sched = normalize_team_schedule(raw_team, season)
    start = pd.to_datetime(reg_sched["game_date"], errors="coerce").dropna().min()
    if pd.isna(start):
        raise RuntimeError(f"{season}: could not determine regular-season start date")
    end = pd.Timestamp(SEASON_ARCHIVE_END[season]).normalize()
    days = list(pd.date_range(start.normalize(), end, freq="D"))
    sb_dir = RAW / "espn_scoreboards"
    sb_dir.mkdir(parents=True, exist_ok=True)

    rows, failures = [], []
    with ThreadPoolExecutor(max_workers=int(ESPN_SCOREBOARD_WORKERS)) as ex:
        futs = {ex.submit(_fetch_espn_scoreboard_day_v2, d, sb_dir): d for d in days}
        done = 0
        for fut in as_completed(futs):
            d = futs[fut]
            try:
                js = fut.result()
                rows.extend(_parse_espn_scoreboard_events_v2(js, d, season))
            except Exception as e:
                failures.append({"date": str(pd.Timestamp(d).date()), "error": repr(e)})
            done += 1
            if done % 50 == 0 or done == len(days):
                print(f"  ESPN schedule {season}: {done:,}/{len(days):,} dates; games_found={len(rows):,}; failures={len(failures):,}")

    out = pd.DataFrame(rows)
    if out.empty:
        raise RuntimeError(f"{season}: ESPN scoreboard produced no NBA games")
    out["game_id"] = out["game_id"].astype(str)
    out["game_date"] = pd.to_datetime(out["game_date"], errors="coerce").dt.normalize()
    out = out.drop_duplicates("game_id").sort_values(["game_date","game_id"]).reset_index(drop=True)
    out.to_parquet(cache, index=False)
    pd.DataFrame(failures).to_csv(RAW / "espn_scoreboard_failures.csv", index=False)
    print(f"  ESPN full schedule {season}: {out['game_id'].nunique():,} games; {out['game_date'].min().date()} to {out['game_date'].max().date()}")
    return out


def get_public_teamlog(season: str) -> pd.DataFrame:
    """Historical seasons use ESPN event IDs; 2024-25 keeps validated NBA IDs."""
    if season not in HISTORICAL_ESPN_SEASONS:
        return _GET_PUBLIC_TEAMLOG_MAIN_V2(season)
    sched = build_espn_full_schedule_v2(season)
    rows = []
    for r in sched.itertuples(index=False):
        rows.append({
            "game_id": str(r.game_id), "game_date": r.game_date,
            "team_abbreviation": r.home_team,
            "matchup": f"{r.home_team} vs. {r.away_team}",
        })
        rows.append({
            "game_id": str(r.game_id), "game_date": r.game_date,
            "team_abbreviation": r.away_team,
            "matchup": f"{r.away_team} @ {r.home_team}",
        })
    return pd.DataFrame(rows)


def _shared_espn_player_box_path_v2():
    shared = MULTI_ROOT / "_shared"
    shared.mkdir(parents=True, exist_ok=True)
    return shared / "espn_player_box.parquet"


def _read_espn_player_box_for_games_v2(game_ids):
    path = _shared_espn_player_box_path_v2()
    if not path.exists() or path.stat().st_size < 1000:
        print("  downloading shared ESPN player_box.parquet (one-time)...")
        _download(ESPN_PLAYER_BOX_URL, path)
    ids = {str(x) for x in game_ids}
    # Read schema first because upstream naming has changed historically.
    try:
        import pyarrow.parquet as pq
        names = pq.ParquetFile(path).schema.names
    except Exception:
        names = list(pd.read_parquet(path).columns)
    game_col = "game_id" if "game_id" in names else ("gameId" if "gameId" in names else None)
    if game_col is None:
        raise RuntimeError(f"ESPN player_box schema has no game id column: {names}")
    try:
        df = pd.read_parquet(path, filters=[(game_col, "in", list(ids))])
    except Exception:
        # Safe fallback when the parquet engine cannot push an IN filter.
        df = pd.read_parquet(path)
        df = df[df[game_col].astype(str).isin(ids)].copy()
    df[game_col] = df[game_col].astype(str)
    return df, game_col


def _col_first_v2(df, *names):
    for n in names:
        if n in df.columns:
            return n
    return None


def get_public_playerlog(season: str) -> pd.DataFrame:
    """Complete historical player-game layer from ESPN; validated NBA source for 2024-25."""
    if season not in HISTORICAL_ESPN_SEASONS:
        return _GET_PUBLIC_PLAYERLOG_MAIN_V2(season)

    sched = build_espn_full_schedule_v2(season)
    event_ids = sched["game_id"].astype(str).unique().tolist()
    src, gidc = _read_espn_player_box_for_games_v2(event_ids)
    if src.empty:
        raise RuntimeError(f"{season}: ESPN player_box returned zero rows for {len(event_ids)} schedule games")

    namec = _col_first_v2(src, "name", "player_name", "displayName")
    pidc = _col_first_v2(src, "player_id", "playerId", "athlete_id")
    teamc = _col_first_v2(src, "team", "team_name", "tmName")
    teamidc = _col_first_v2(src, "team_id", "teamId")
    minc = _col_first_v2(src, "minutes_played", "minutes", "MIN")
    playedc = _col_first_v2(src, "played", "active")
    if not all([namec, pidc, teamc, minc]):
        raise RuntimeError(
            f"{season}: ESPN player_box missing required columns; "
            f"name={namec}, player_id={pidc}, team={teamc}, minutes={minc}; columns={list(src.columns)}"
        )

    date_map = sched.set_index("game_id")["game_date"].to_dict()
    out = pd.DataFrame(index=src.index)
    out["gameId"] = src[gidc].astype(str)
    out["GAME_DATE"] = out["gameId"].map(date_map)
    out["teamTricode"] = src[teamc].map(_espn_team_to_nba_v2)
    if teamidc:
        out["teamId"] = pd.to_numeric(src[teamidc], errors="coerce")
    else:
        out["teamId"] = np.nan
    out["PLAYER_NAME"] = src[namec].astype(str)
    out["espn_player_id"] = pd.to_numeric(src[pidc], errors="coerce")
    out["personId"] = [
        _stable_surrogate_player_id(f"espn:{int(x)}") if pd.notna(x)
        else _stable_surrogate_player_id(f"espn_name:{norm_name(n)}")
        for x, n in zip(out["espn_player_id"], out["PLAYER_NAME"])
    ]
    out["minutes"] = src[minc]
    out["comment"] = ""
    out["player_id_source"] = "espn"
    out["player_game_source"] = "llimllib_espn_player_box"

    # Preserve played flag where available; otherwise infer from minutes.
    if playedc:
        out["espn_played_source"] = pd.to_numeric(src[playedc], errors="coerce")
    else:
        out["espn_played_source"] = np.nan

    # Map box-score columns into the canonical names used by the RTP engine.
    mappings = {
        "points": ("pts", "points"),
        "plusMinusPoints": ("plusMinusPoints", "plus_minus"),
        "fieldGoalsMade": ("fgmplyr", "fieldGoalsMade"),
        "fieldGoalsAttempted": ("fgaplyr", "fieldGoalsAttempted"),
        "threePointersMade": ("fg3mplyr", "threePointersMade"),
        "threePointersAttempted": ("fg3aplyr", "threePointersAttempted"),
        "freeThrowsMade": ("ftmplyr", "freeThrowsMade"),
        "freeThrowsAttempted": ("ftaplyr", "freeThrowsAttempted"),
        "assists": ("assister", "assists"),
        "steals": ("stlr", "steals"),
        "blocks": ("blockplyr", "blocks"),
        "turnovers": ("tov1", "turnovers"),
    }
    for dst, candidates in mappings.items():
        c = _col_first_v2(src, *candidates)
        if c:
            out[dst] = pd.to_numeric(src[c], errors="coerce")

    oc = _col_first_v2(src, "orebounder", "reboundsOffensive")
    dc = _col_first_v2(src, "drebounder", "reboundsDefensive")
    if oc:
        out["reboundsOffensive"] = pd.to_numeric(src[oc], errors="coerce")
    if dc:
        out["reboundsDefensive"] = pd.to_numeric(src[dc], errors="coerce")
    if oc or dc:
        out["reboundsTotal"] = out.get("reboundsOffensive", 0).fillna(0) + out.get("reboundsDefensive", 0).fillna(0)

    # Compute TS% from box-score primitives when possible.
    if all(c in out.columns for c in ["points","fieldGoalsAttempted","freeThrowsAttempted"]):
        denom = 2 * (pd.to_numeric(out["fieldGoalsAttempted"], errors="coerce") +
                     0.44 * pd.to_numeric(out["freeThrowsAttempted"], errors="coerce"))
        out["trueShootingPercentage"] = np.where(
            denom > 0, pd.to_numeric(out["points"], errors="coerce") / denom, np.nan
        )

    # Preserve source-specific metrics under explicit names rather than pretending
    # ESPN oUsg/tNetPts are identical to NBA Stats USG%/Net Rating.
    for c in ["oUsg","dUsg","oNetPts","dNetPts","tNetPts","oWPA","dWPA","tWPA","starter"]:
        if c in src.columns:
            out[f"espn_{c}"] = src[c].values

    out = out[out["teamTricode"].notna()].copy()
    # A source row with played=0 is intentionally retained; normalize_playerlog will
    # represent it as a non-played game and official reports can supply the reason.
    print(
        f"  ESPN player source {season}: {len(out):,} player-game rows; "
        f"games={out['gameId'].nunique():,}/{len(event_ids):,}; teams={out['teamTricode'].nunique():,}"
    )
    return out.reset_index(drop=True)


def _game_type_from_id(game_id):
    gid = str(game_id)
    if gid in _ESPN_GAME_TYPE_MAP_V2:
        return _ESPN_GAME_TYPE_MAP_V2[gid]
    return _GAME_TYPE_MAIN_V2(game_id)


def add_player_age(episodes: pd.DataFrame, season: str) -> pd.DataFrame:
    """Historical ESPN IDs are not NBA IDs, so merge age by normalized player name."""
    if season not in HISTORICAL_ESPN_SEASONS:
        return _ADD_PLAYER_AGE_MAIN_V2(episodes, season)
    if episodes is None or episodes.empty:
        return episodes
    try:
        ps = get_public_player_season_stats(season)
    except Exception as e:
        print("age merge skipped:", e)
        return episodes
    namec = _first_col(ps, ["player_name","PLAYER_NAME","display_name"])
    agec = _first_col(ps, ["age","AGE"])
    if not namec or not agec or "player_key" not in episodes.columns:
        return episodes
    x = ps[[namec, agec]].copy().rename(columns={agec:"season_age"})
    x["player_key"] = x[namec].map(norm_name)
    x = x.dropna(subset=["player_key"]).drop_duplicates("player_key")[["player_key","season_age"]]
    return episodes.drop(columns=["season_age"], errors="ignore").merge(x, on="player_key", how="left")


# Harden official-report scope: current independent tooling documents static
# historical report availability beginning in 2021-22.
_GET_OFFICIAL_IR_MAIN_V2 = get_official_injury_designations_multiseason

def get_official_injury_designations_multiseason(season: str):
    if season not in OFFICIAL_IR_SUPPORTED_SEASONS:
        raise RuntimeError(
            f"{season} is outside the v2 primary official-injury-report window. "
            "Use 2021-22 through 2024-25 for the primary multi-season extraction."
        )
    return _GET_OFFICIAL_IR_MAIN_V2(season)


# Add an explicit observed-game coverage gate. Synthetic report-only rows must not
# be allowed to hide an incomplete played-game source.
def build_multiseason_qc_tables(season: str, pg: pd.DataFrame, injury_ep: pd.DataFrame,
                                reports_raw: pd.DataFrame):
    overall, tables = _BUILD_MULTI_QC_MAIN_V2(season, pg, injury_ep, reports_raw)
    x = pg.copy()
    sched = normalize_team_schedule(get_public_teamlog(season), season)
    schedule_games = int(sched["game_id"].astype(str).nunique()) if len(sched) else 0
    observed_mask = pd.to_numeric(
        x.get("playerlog_row_present", pd.Series(1, index=x.index)), errors="coerce"
    ).fillna(0).eq(1)
    observed_games = int(x.loc[observed_mask, "game_id"].astype(str).nunique()) if len(x) else 0
    coverage = observed_games / schedule_games if schedule_games else np.nan

    gate_df = tables["qc_gates"].copy()
    gate_df = pd.concat([gate_df, pd.DataFrame([{
        "gate": "observed player-game source covers >=95% of schedule games",
        "pass": bool(pd.notna(coverage) and coverage >= HISTORICAL_MIN_OBSERVED_GAME_COVERAGE),
        "observed": f"{observed_games}/{schedule_games} ({coverage:.1%})" if pd.notna(coverage) else "NA",
    }])], ignore_index=True)

    summary = tables["qc_summary"].copy()
    summary["schedule_games_expected"] = schedule_games
    summary["observed_player_source_games"] = observed_games
    summary["observed_game_coverage"] = coverage
    summary["player_game_source"] = "espn_player_box" if season in HISTORICAL_ESPN_SEASONS else "nba_playerlog"

    # Soft parser diagnostic: do not silently call a season PASS if historical
    # injury extraction yielded implausibly few designation rows.
    if season in HISTORICAL_ESPN_SEASONS:
        parser_ok = int(len(reports_raw)) >= 5000
        gate_df = pd.concat([gate_df, pd.DataFrame([{
            "gate": "historical official-report parser produced >=5,000 designation rows",
            "pass": parser_ok,
            "observed": f"{len(reports_raw):,}",
        }])], ignore_index=True)

    overall = "PASS" if bool(gate_df["pass"].all()) else "REVIEW"
    summary["qc_status"] = overall
    tables["qc_gates"] = gate_df
    tables["qc_summary"] = summary
    gate_df.to_csv(ROOT / "qc_gates.csv", index=False)
    summary.to_csv(ROOT / "qc_summary.csv", index=False)
    return overall, tables


# Do not combine REVIEW seasons by default. This prevents an old invalid 2019-20
# output from leaking into a future combined analysis if the same filesystem is reused.
print("v2 historical source layer loaded.")
print("  primary seasons:", FULL_SEASONS)
print("  ESPN historical player source:", sorted(HISTORICAL_ESPN_SEASONS))
print("  2024-25 retains validated NBA playerlog source")


## 6D. v2.2 deterministic raw ESPN archive fix

This override removes both the live ESPN scoreboard dependency and the GitHub Contents API dependency. Historical schedule reconstruction now uses deterministic `raw.githubusercontent.com` daily archive URLs only. Non-game dates are harmless 404s; the independent NBA team log remains the coverage benchmark.


In [ ]:
"""
NBA RTP historical ESPN raw-archive fix v2.2

Load AFTER the v2 historical source layer and BEFORE the batch/run cell.

This removes BOTH fragile discovery dependencies used by earlier historical patches:
  * no site.api.espn.com live scoreboard requests;
  * no api.github.com directory-listing requests.

Instead, it constructs deterministic raw.githubusercontent.com URLs for each
calendar date that could contain NBA games in the season, downloads only those
static archived JSON files, and reconstructs ESPN game IDs/date/home-away from
the top-level team_box table.

The existing v2 aggregate espn/player_box.parquet player-game source, official
injury-report downloader/parser, RTP logic, and QC gates remain unchanged.
"""

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import json
import random
import threading
import time
import pandas as pd
import requests

ESPN_RAW_WORKERS_V22 = 4
ESPN_RAW_START_INTERVAL_V22 = 0.10   # globally spaced request starts (~10/s max)
ESPN_RAW_TIMEOUT_V22 = (6, 35)
ESPN_RAW_RETRIES_V22 = 4
ESPN_RAW_PROGRESS_EVERY_V22 = 25

_ESPN_RAW_RATE_LOCK_V22 = threading.Lock()
_ESPN_RAW_NEXT_START_V22 = 0.0


def _espn_raw_wait_v22():
    global _ESPN_RAW_NEXT_START_V22
    interval = float(globals().get("ESPN_RAW_START_INTERVAL_V22", 0.10))
    with _ESPN_RAW_RATE_LOCK_V22:
        now = time.monotonic()
        slot = max(now, _ESPN_RAW_NEXT_START_V22)
        _ESPN_RAW_NEXT_START_V22 = slot + interval
        wait = slot - now
    if wait > 0:
        time.sleep(wait)


def _season_end_int_v22(season: str) -> int:
    return int(str(season)[:4]) + 1


def _raw_daily_dir_v22() -> Path:
    p = RAW / "espn_raw_daily_v22"
    p.mkdir(parents=True, exist_ok=True)
    return p


def _candidate_espn_archive_days_v22(season: str, reg_sched: pd.DataFrame):
    """Dates worth trying in the static ESPN archive.

    Regular season: only dates on which the independent NBA team log says games
    occurred. Postseason/play-in: every calendar day from the day after the final
    regular-season game through the configured season end, because the main team
    parquet used here is regular-season only for some historical years.
    """
    reg_dates = pd.to_datetime(reg_sched["game_date"], errors="coerce").dropna().dt.normalize()
    if reg_dates.empty:
        raise RuntimeError(f"{season}: independent NBA team log has no usable dates")
    reg_unique = sorted(pd.unique(reg_dates))
    reg_end = pd.Timestamp(max(reg_unique)).normalize()
    archive_end = pd.Timestamp(SEASON_ARCHIVE_END[season]).normalize()
    post_days = []
    if archive_end > reg_end:
        post_days = list(pd.date_range(reg_end + pd.Timedelta(days=1), archive_end, freq="D"))
    days = sorted({pd.Timestamp(x).normalize() for x in list(reg_unique) + post_days})
    return days, reg_end, archive_end


def _raw_url_v22(season_end: int, day) -> str:
    filename = pd.Timestamp(day).strftime("%Y-%m-%d.json")
    return f"{LLIMLLIB_RAW}/espn/{season_end}/{filename}"


def _fetch_raw_daily_json_v22(season_end: int, day, dest: Path, retries=None):
    """Fetch one deterministic raw GitHub archive file; 404 is an expected miss."""
    retries = int(retries or ESPN_RAW_RETRIES_V22)
    if dest.exists() and dest.stat().st_size > 100:
        try:
            return json.loads(dest.read_text()), "cache", 200
        except Exception:
            try:
                dest.unlink()
            except Exception:
                pass

    url = _raw_url_v22(season_end, day)
    last = None
    for attempt in range(retries):
        try:
            _espn_raw_wait_v22()
            r = requests.get(
                url,
                headers={
                    "User-Agent": HEADERS.get("User-Agent", "Mozilla/5.0") if isinstance(globals().get("HEADERS"), dict) else "Mozilla/5.0",
                    "Accept": "application/json,text/plain,*/*",
                },
                timeout=ESPN_RAW_TIMEOUT_V22,
            )
            if r.status_code == 404:
                return None, "404", 404
            if r.status_code in (429, 500, 502, 503, 504):
                last = RuntimeError(f"HTTP {r.status_code}")
                if attempt + 1 < retries:
                    time.sleep(min(8.0, 0.8 * (2 ** attempt)) + random.uniform(0, 0.2))
                    continue
            r.raise_for_status()
            js = r.json()
            tmp = dest.with_suffix(dest.suffix + ".tmp")
            tmp.write_text(json.dumps(js))
            tmp.replace(dest)
            return js, "download", int(r.status_code)
        except Exception as exc:
            last = exc
            if attempt + 1 < retries:
                time.sleep(min(8.0, 0.6 * (2 ** attempt)) + random.uniform(0, 0.2))
    raise RuntimeError(f"raw ESPN archive fetch failed {url}: {last}")


def _team_box_schedule_rows_v22(js, day, season):
    """One schedule row per game from llimllib archived ESPN team_box records."""
    if not isinstance(js, dict):
        return []
    team_box = js.get("team_box") or []
    if not isinstance(team_box, list):
        return []

    by_game = {}
    for tb in team_box:
        if not isinstance(tb, dict):
            continue
        gid = str(tb.get("gameId") or tb.get("gmId") or tb.get("game_id") or "").strip()
        if not gid:
            continue
        team_raw = tb.get("tmName") or tb.get("team") or tb.get("deanAbbrev") or tb.get("teamName")
        team = _espn_team_to_nba_v2(team_raw)
        if not team:
            continue
        home_raw = tb.get("homeTm", tb.get("hmTm", tb.get("home")))
        try:
            is_home = int(float(home_raw)) == 1
        except Exception:
            is_home = str(home_raw).strip().lower() in {"1", "true", "home", "h"}
        by_game.setdefault(gid, []).append((team, is_home))

    rows = []
    for gid, vals in by_game.items():
        # Deduplicate exact repeats.
        vals = list(dict.fromkeys((str(t), bool(h)) for t, h in vals))
        distinct = list(dict.fromkeys(t for t, _ in vals))
        if len(distinct) != 2:
            continue
        homes = list(dict.fromkeys(t for t, h in vals if h))
        aways = list(dict.fromkeys(t for t, h in vals if not h))
        if len(homes) == 1:
            home = homes[0]
            away = next((t for t in distinct if t != home), None)
        elif len(aways) == 1:
            away = aways[0]
            home = next((t for t in distinct if t != away), None)
        else:
            continue
        if not home or not away:
            continue
        rows.append({
            "season": season,
            "game_id": str(gid),
            "game_date": pd.Timestamp(day).normalize(),
            "home_team": home,
            "away_team": away,
            "espn_event_id": str(gid),
            "schedule_source": "llimllib_raw_espn_daily_archive_v22",
        })
    return rows


def build_espn_full_schedule_v2(season: str, force=False):
    """v2.2 historical schedule: deterministic static raw GitHub archive URLs."""
    if season not in HISTORICAL_ESPN_SEASONS:
        raise ValueError(f"ESPN historical schedule not configured for {season}")

    cache = RAW / "espn_archive_full_schedule_v22.parquet"
    if cache.exists() and not force:
        out = pd.read_parquet(cache)
        for r in out[["game_id", "game_type_espn"]].drop_duplicates().itertuples(index=False):
            _ESPN_GAME_TYPE_MAP_V2[str(r.game_id)] = str(r.game_type_espn)
        return out

    raw_team = _GET_PUBLIC_TEAMLOG_MAIN_V2(season)
    reg_sched = normalize_team_schedule(raw_team, season)
    reg_expected = int(reg_sched["game_id"].astype(str).nunique())
    if reg_expected <= 0:
        raise RuntimeError(f"{season}: independent NBA schedule contained zero games")

    days, reg_end, archive_end = _candidate_espn_archive_days_v22(season, reg_sched)
    season_end = _season_end_int_v22(season)
    daily_dir = _raw_daily_dir_v22()
    cached_before = sum((daily_dir / (pd.Timestamp(d).strftime("%Y-%m-%d") + ".json")).exists() for d in days)
    print(
        f"  ESPN raw archive {season}: {len(days):,} candidate dates; cached={cached_before:,}; "
        f"workers={ESPN_RAW_WORKERS_V22}; start_interval={ESPN_RAW_START_INTERVAL_V22:.2f}s; GitHub API=OFF"
    )

    rows, failures = [], []
    counts = {"200": 0, "404": 0, "cache": 0}
    with ThreadPoolExecutor(max_workers=int(ESPN_RAW_WORKERS_V22)) as ex:
        futs = {}
        for d in days:
            filename = pd.Timestamp(d).strftime("%Y-%m-%d.json")
            futs[ex.submit(_fetch_raw_daily_json_v22, season_end, d, daily_dir / filename)] = d
        done = 0
        for fut in as_completed(futs):
            day = futs[fut]
            try:
                js, source, http_status = fut.result()
                if source == "cache":
                    counts["cache"] += 1
                elif http_status == 404:
                    counts["404"] += 1
                else:
                    counts["200"] += 1
                if js is not None:
                    rows.extend(_team_box_schedule_rows_v22(js, day, season))
            except Exception as exc:
                failures.append({"date": str(pd.Timestamp(day).date()), "error": repr(exc)})
            done += 1
            if done % ESPN_RAW_PROGRESS_EVERY_V22 == 0 or done == len(futs):
                print(
                    f"    ESPN raw archive {season}: {done:,}/{len(futs):,} dates; "
                    f"games_found={len(rows):,}; 200={counts['200']:,}; cache={counts['cache']:,}; "
                    f"404={counts['404']:,}; failures={len(failures):,}"
                )

    out = pd.DataFrame(rows)
    pd.DataFrame(failures).to_csv(RAW / "espn_raw_archive_failures_v22.csv", index=False)
    if out.empty:
        raise RuntimeError(f"{season}: static raw ESPN archive produced no NBA games")

    out["game_id"] = out["game_id"].astype(str)
    out["game_date"] = pd.to_datetime(out["game_date"], errors="coerce").dt.normalize()
    out = out.dropna(subset=["game_id", "game_date", "home_team", "away_team"])
    out = out.drop_duplicates("game_id").sort_values(["game_date", "game_id"]).reset_index(drop=True)

    # The independent NBA team log defines the regular-season date range; later
    # archived games are postseason/play-in. This label is for context only.
    out["game_type_espn"] = "regular_or_special"
    out.loc[out["game_date"] > reg_end, "game_type_espn"] = "postseason"
    for r in out[["game_id", "game_type_espn"]].itertuples(index=False):
        _ESPN_GAME_TYPE_MAP_V2[str(r.game_id)] = str(r.game_type_espn)

    found = int(out["game_id"].nunique())
    out.to_parquet(cache, index=False)
    print(
        f"  ESPN raw archive full schedule {season}: {found:,} games; "
        f"NBA regular-season benchmark={reg_expected:,}; failures={len(failures):,}; "
        f"dates={out['game_date'].min().date()} to {out['game_date'].max().date()}"
    )
    if found < int(0.95 * reg_expected):
        raise RuntimeError(
            f"{season}: raw ESPN archive only found {found:,} games vs "
            f"{reg_expected:,} NBA regular-season benchmark; refusing to continue"
        )
    return out


def self_test_espn_raw_archive_v22():
    fake = {
        "team_box": [
            {"gameId": "401585100", "tmName": "Boston Celtics", "homeTm": 1},
            {"gameId": "401585100", "tmName": "New York Knicks", "homeTm": 0},
        ]
    }
    rows = _team_box_schedule_rows_v22(fake, "2023-10-24", "2023-24")
    assert len(rows) == 1, rows
    assert rows[0]["home_team"] == "BOS" and rows[0]["away_team"] == "NYK", rows
    assert rows[0]["game_id"] == "401585100", rows
    assert _raw_url_v22(2024, "2023-10-24").endswith("/espn/2024/2023-10-24.json")
    print("v2.2 self-test passed: deterministic raw ESPN archive schedule parser works.")
    return True


self_test_espn_raw_archive_v22()
print("NBA RTP raw ESPN archive fix v2.2 loaded.")
print("  site.api.espn.com calls: OFF")
print("  api.github.com calls: OFF")
print("  historical schedule source: raw.githubusercontent.com static daily archive")
print("  existing injury-report PDF caches and shared ESPN player_box.parquet are reused")


## 6E. v2.3 historical official-injury CSV fix

For **2021–22 through 2023–24**, this override replaces the custom historical PDF parser with the public Stat Surge pre-parsed NBA injury-report archive (regular season + playoffs). The ESPN player-game layer and v2.2 raw-archive schedule remain unchanged; 2024–25 keeps the previously validated source.


In [ ]:
"""
NBA RTP historical official-injury CSV fix v2.3

Load AFTER the v2.2 historical source/archive layers and BEFORE the batch/run cell.

Purpose
-------
The custom PDF parser substantially under-extracts 2021-22 and 2022-23 official
injury reports. Replace historical PDF parsing for 2021-22 through 2023-24 with
Vaughn Hajra / Stat Surge's public pre-parsed historical CSV archive, while
leaving the validated 2024-25 Professor-Pete source unchanged.

The Stat Surge archive exposes separate regular-season and playoff CSVs for each
of 2021-22, 2022-23, and 2023-24. We normalize only the small set of fields the
existing RTP pipeline requires: Game Date, Team, Player Name, Current Status,
and Reason. Matchup is optional because v5 maps reports to games by team+date.
"""

from pathlib import Path
import io
import re
import time
import pandas as pd
import requests

# Published Google Sheet backing the historical CSV links on Stat Surge.
_STATSURGE_SHEET = (
    "https://docs.google.com/spreadsheets/d/e/"
    "2PACX-1vTI7Y9RhrP7BhS59Nsofqnb7QqOlqXotOAKQV8Tjyv0WHF2p2yoTsF6LIVBk9vOS2oWBs5Yt4Jxp2no/"
    "pub?gid={gid}&single=true&output=csv"
)

# GIDs obtained from the public historical download links.
_STATSURGE_GIDS = {
    "2021-22": {"regular": 183552761, "playoffs": 1351735005},
    "2022-23": {"regular": 2111042600, "playoffs": 482229493},
    "2023-24": {"regular": 0, "playoffs": 773871834},
}

_V23_HISTORICAL_SEASONS = set(_STATSURGE_GIDS)
_GET_OFFICIAL_IR_PRE_V23 = get_official_injury_designations_multiseason


def _v23_clean_key(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).strip().lower())


def _v23_pick_col(df, candidates):
    by_key = {_v23_clean_key(c): c for c in df.columns}
    for cand in candidates:
        hit = by_key.get(_v23_clean_key(cand))
        if hit is not None:
            return hit
    return None


def _v23_download_csv(url: str, path: Path, retries: int = 4) -> Path:
    if path.exists() and path.stat().st_size > 100:
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    last = None
    for attempt in range(retries):
        try:
            sess = _http_session() if "_http_session" in globals() else requests.Session()
            r = sess.get(url, timeout=(10, 90), allow_redirects=True)
            if r.status_code == 429:
                raise RuntimeError("HTTP 429 from published Google Sheet")
            r.raise_for_status()
            body = r.content
            # Protect against a login/HTML response being cached as CSV.
            head = body[:500].lower()
            if b"<html" in head or b"<!doctype" in head:
                raise RuntimeError("published Google Sheet returned HTML instead of CSV")
            tmp = path.with_suffix(path.suffix + ".tmp")
            tmp.write_bytes(body)
            tmp.replace(path)
            return path
        except Exception as e:
            last = e
            if attempt + 1 < retries:
                time.sleep(1.0 * (2 ** attempt))
    raise RuntimeError(f"historical injury CSV download failed: {url}: {last}")


def _v23_normalize_statsurge_frame(df: pd.DataFrame, season: str, phase: str) -> pd.DataFrame:
    """Normalize a Stat Surge historical CSV into the raw schema expected downstream."""
    if df is None or df.empty:
        return pd.DataFrame(columns=[
            "Game Date", "Game Time", "Matchup", "Team", "Player Name",
            "Current Status", "Reason", "source", "report_ts"
        ])

    d = df.copy()
    datec = _v23_pick_col(d, [
        "Game Date", "game_date", "Date", "GameDate", "Report Date", "report_date"
    ])
    teamc = _v23_pick_col(d, ["Team", "team", "Team Name", "team_name"])
    playerc = _v23_pick_col(d, [
        "Player Name", "player_name", "Player", "player", "Name"
    ])
    statusc = _v23_pick_col(d, [
        "Current Status", "current_status", "Status", "status", "Designation"
    ])
    reasonc = _v23_pick_col(d, [
        "Reason", "reason", "Injury/Condition", "Injury Condition",
        "Injury/Illness", "Injury", "Condition", "Description"
    ])
    matchupc = _v23_pick_col(d, ["Matchup", "matchup", "Game", "game"])
    timec = _v23_pick_col(d, ["Game Time", "game_time", "Time", "time"])

    required = {
        "Game Date": datec,
        "Team": teamc,
        "Player Name": playerc,
        "Reason": reasonc,
    }
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise RuntimeError(
            f"{season} {phase}: Stat Surge CSV schema missing {missing}; "
            f"columns={list(d.columns)}"
        )

    out = pd.DataFrame(index=d.index)
    out["Game Date"] = d[datec]
    out["Game Time"] = d[timec] if timec else ""
    out["Matchup"] = d[matchupc] if matchupc else ""
    out["Team"] = d[teamc]
    out["Player Name"] = d[playerc]
    out["Current Status"] = d[statusc] if statusc else ""
    out["Reason"] = d[reasonc]
    out["source"] = f"statsurge_{season}_{phase}_historical_csv"

    # This archive was built from the NBA's consolidated ~2pm daily report.
    # A synthetic 14:00 timestamp is sufficient only for deterministic ordering;
    # actual participation still comes from the player-game source, never status.
    gd = pd.to_datetime(out["Game Date"], errors="coerce")
    out["report_ts"] = gd.dt.normalize() + pd.Timedelta(hours=14)

    # Drop completely blank structural rows but otherwise preserve designations.
    p = out["Player Name"].fillna("").astype(str).str.strip()
    t = out["Team"].fillna("").astype(str).str.strip()
    out = out[(p != "") & (t != "") & gd.notna()].copy()
    return out.reset_index(drop=True)


def _v23_load_statsurge_season(season: str, force: bool = False) -> pd.DataFrame:
    if season not in _V23_HISTORICAL_SEASONS:
        raise ValueError(season)

    combined_cache = RAW / f"statsurge_official_injury_designations_{season_tag(season)}.csv"
    if combined_cache.exists() and combined_cache.stat().st_size > 100 and not force:
        out = pd.read_csv(combined_cache, low_memory=False)
        print(f"  Stat Surge injury archive {season}: cached {len(out):,} designation rows")
        return out

    frames = []
    for phase, gid in _STATSURGE_GIDS[season].items():
        url = _STATSURGE_SHEET.format(gid=gid)
        raw_path = RAW / f"statsurge_{season_tag(season)}_{phase}.csv"
        _v23_download_csv(url, raw_path)
        raw = pd.read_csv(raw_path, low_memory=False)
        norm = _v23_normalize_statsurge_frame(raw, season, phase)
        print(
            f"  Stat Surge {season} {phase}: raw_rows={len(raw):,}; "
            f"designation_rows={len(norm):,}; columns={list(raw.columns)}"
        )
        frames.append(norm)

    out = pd.concat(frames, ignore_index=True, sort=False)
    # Same player/team/date can legitimately occur once in each phase only at a
    # boundary; exact duplicate rows are not analytically useful.
    out = out.drop_duplicates().reset_index(drop=True)

    if len(out) < 5000:
        raise RuntimeError(
            f"{season}: Stat Surge historical archive produced only {len(out):,} rows; "
            "refusing to continue because historical injury coverage is implausibly sparse."
        )

    out.to_csv(combined_cache, index=False)
    print(f"  Stat Surge injury archive {season}: {len(out):,} total designation rows")
    return out


def get_official_injury_designations_multiseason(season: str):
    """v2.3: pre-parsed historical CSVs for 2021-24; validated source for 2024-25."""
    if season in _V23_HISTORICAL_SEASONS:
        return _v23_load_statsurge_season(season)
    return _GET_OFFICIAL_IR_PRE_V23(season)


# Small schema regression test independent of network access.
def self_test_v23():
    x = pd.DataFrame({
        "Date": ["2023-01-02"],
        "Team": ["Boston Celtics"],
        "Player": ["Test Player"],
        "Status": ["Out"],
        "Injury/Condition": ["Left Ankle; Sprain"],
    })
    y = _v23_normalize_statsurge_frame(x, "2022-23", "regular")
    assert list(y[["Game Date", "Team", "Player Name", "Current Status", "Reason"]].iloc[0]) == [
        "2023-01-02", "Boston Celtics", "Test Player", "Out", "Left Ankle; Sprain"
    ]
    assert pd.notna(y.loc[0, "report_ts"])
    print("v2.3 self-test passed: historical CSV schema normalization works.")


self_test_v23()
print("NBA RTP historical injury CSV fix v2.3 loaded.")
print("  2021-22 to 2023-24 injury source: Stat Surge pre-parsed historical CSV archive")
print("  2024-25 injury source: unchanged validated Professor-Pete pipeline")
print("  Existing ESPN player-game source and v2.2 schedule layer: unchanged")


## 6F. v2.5 schedule team-identity repair

Reconstructs historical schedule home/away team identity from the ESPN player-box source.
This also makes cached historical schedules robust to source-label inconsistencies.


In [ ]:

"""
NBA RTP schedule-team identity repair v2.5

Load AFTER:
  - v2.2 raw ESPN archive layer
  - v2.3 historical injury CSV layer
and BEFORE rerunning run_multiseason(...).

Purpose
-------
Historical raw ESPN daily JSONs are reliable for game ID + date, but some
team labels can differ from the canonical NBA abbreviations used by the
injury-report layer. The mismatch is strongly concentrated in SAS/NOP/PHX.

The shared ESPN player_box.parquet already contains, for every player-game:
  game_id, team, and home flag.
This patch uses that source to reconstruct canonical home_team/away_team for
each historical ESPN game, while preserving the raw archive's game_id/date.

No injury rows are shifted or altered.
"""

import pandas as pd
import numpy as np

_BUILD_ESPN_FULL_SCHEDULE_PRE_V25 = build_espn_full_schedule_v2


def _v25_bool_home(x):
    """Normalize ESPN home flags to True/False/NA."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"1", "1.0", "true", "t", "home", "h"}:
        return True
    if s in {"0", "0.0", "false", "f", "away", "a"}:
        return False
    try:
        return bool(int(float(x)))
    except Exception:
        return np.nan


def _v25_player_box_team_map(game_ids):
    """
    Build one canonical home/away row per game from ESPN player_box.
    Returns columns: game_id, home_team_pb, away_team_pb.
    """
    src, gidc = _read_espn_player_box_for_games_v2(list(game_ids))
    if src is None or src.empty:
        raise RuntimeError("v2.5: ESPN player_box returned no rows")

    teamc = _col_first_v2(src, "team", "team_name", "tmName")
    homec = _col_first_v2(src, "home", "homeTm", "is_home", "home_flag")
    if teamc is None or homec is None:
        raise RuntimeError(
            f"v2.5: player_box missing team/home fields; "
            f"team={teamc}, home={homec}, columns={list(src.columns)}"
        )

    x = src[[gidc, teamc, homec]].copy()
    x["game_id"] = x[gidc].astype(str)
    x["_team_raw"] = x[teamc]
    x["_team"] = x["_team_raw"].map(_espn_team_to_nba_v2)
    x["_home"] = x[homec].map(_v25_bool_home)
    x = x[x["_team"].notna() & x["_home"].notna()].copy()
    x["_team"] = x["_team"].astype(str).str.upper()

    # One record per game/team/home-away state; player rows are duplicates by design.
    x = x[["game_id", "_team", "_home"]].drop_duplicates()

    rows = []
    bad = []
    for gid, g in x.groupby("game_id", sort=False):
        homes = g.loc[g["_home"] == True, "_team"].drop_duplicates().tolist()
        aways = g.loc[g["_home"] == False, "_team"].drop_duplicates().tolist()
        if len(homes) == 1 and len(aways) == 1 and homes[0] != aways[0]:
            rows.append({
                "game_id": str(gid),
                "home_team_pb": homes[0],
                "away_team_pb": aways[0],
            })
        else:
            bad.append((str(gid), homes, aways))

    out = pd.DataFrame(rows)
    return out, bad


def _v25_repair_schedule_team_identity(sched: pd.DataFrame, season: str) -> pd.DataFrame:
    """
    Keep game_id/date from the raw archive, but take canonical home/away identity
    from player_box. Only games with an unambiguous 1-home/1-away reconstruction
    are overwritten.
    """
    if sched is None or sched.empty:
        return sched

    out = sched.copy()
    out["game_id"] = out["game_id"].astype(str)
    old_home = out["home_team"].astype(str).copy()
    old_away = out["away_team"].astype(str).copy()

    pb, bad = _v25_player_box_team_map(out["game_id"].unique())
    if pb.empty:
        raise RuntimeError(f"{season}: v2.5 could not reconstruct any player-box team identities")

    out = out.merge(pb, on="game_id", how="left", validate="one_to_one")

    usable = out["home_team_pb"].notna() & out["away_team_pb"].notna()
    out.loc[usable, "home_team"] = out.loc[usable, "home_team_pb"]
    out.loc[usable, "away_team"] = out.loc[usable, "away_team_pb"]

    changed_home = usable & (old_home.values != out["home_team"].astype(str).values)
    changed_away = usable & (old_away.values != out["away_team"].astype(str).values)
    changed_game = changed_home | changed_away

    # Summarize exactly what was repaired. This is useful for QC and should make
    # the SAS/NOP/PHX issue immediately visible.
    changes = pd.DataFrame({
        "game_id": out["game_id"],
        "old_home": old_home.values,
        "new_home": out["home_team"].astype(str).values,
        "old_away": old_away.values,
        "new_away": out["away_team"].astype(str).values,
        "changed": changed_game.values,
    })
    changed = changes[changes["changed"]].copy()

    out["schedule_team_identity_source"] = np.where(
        usable, "espn_player_box_home_flag", out.get("schedule_source", "raw_archive")
    )

    out = out.drop(columns=["home_team_pb", "away_team_pb"], errors="ignore")

    # Save an audit table and overwrite the v2.2 schedule cache with corrected labels.
    try:
        changed.to_csv(ROOT / "espn_schedule_team_identity_repairs_v25.csv", index=False)
        out.to_parquet(RAW / "espn_archive_full_schedule_v22.parquet", index=False)
    except Exception as e:
        print("  v2.5 schedule audit/cache write warning:", e)

    print(
        f"  v2.5 schedule team identity {season}: "
        f"player_box_resolved={int(usable.sum()):,}/{len(out):,}; "
        f"games_with_team_label_changes={int(changed_game.sum()):,}; "
        f"ambiguous_player_box_games={len(bad):,}"
    )

    if len(changed):
        # Show the most common old->new labels without flooding output.
        pairs = pd.concat([
            changed[["old_home", "new_home"]].rename(columns={"old_home":"old","new_home":"new"}),
            changed[["old_away", "new_away"]].rename(columns={"old_away":"old","new_away":"new"}),
        ], ignore_index=True)
        pairs = pairs[pairs["old"] != pairs["new"]]
        if len(pairs):
            print("  v2.5 most common team-label repairs:")
            print(pairs.value_counts(["old","new"]).head(12).to_string())

    return out


def build_espn_full_schedule_v2(season: str, force=False):
    """
    v2.5 wrapper:
      1) obtain v2.2 raw-archive schedule (cache-first)
      2) repair home/away identities from player_box
    """
    out = _BUILD_ESPN_FULL_SCHEDULE_PRE_V25(season, force=force)
    if season in HISTORICAL_ESPN_SEASONS:
        out = _v25_repair_schedule_team_identity(out, season)
        for r in out[["game_id", "game_type_espn"]].drop_duplicates().itertuples(index=False):
            _ESPN_GAME_TYPE_MAP_V2[str(r.game_id)] = str(r.game_type_espn)
    return out


def self_test_v25():
    # Pure helper test: home flag normalization.
    assert _v25_bool_home(1) is True
    assert _v25_bool_home(0) is False
    assert _v25_bool_home("home") is True
    assert _v25_bool_home("away") is False
    print("v2.5 self-test passed: player-box home/away normalization works.")


self_test_v25()
print("NBA RTP schedule-team identity repair v2.5 loaded.")
print("  Injury data: unchanged")
print("  Game dates/IDs: unchanged")
print("  Historical home/away team identity: reconstructed from ESPN player_box home flag")
print("  Expected target: remove SAS/NOP/PHX report↔game mapping mismatch")


## 6G. v2.6 canonical historical team aliases

Canonicalizes the three ESPN abbreviations that caused nearly all remaining historical
report-to-game mismatches:

- `NOR → NOP`
- `PHO → PHX`
- `SAN → SAS`

This patch operates on cached schedules and does not redownload data.


In [ ]:

"""
NBA RTP team-alias fix v2.6

Run AFTER the existing pipeline definitions and patches, BEFORE rerunning the
v2.4 unmatched diagnostic or the full season.

Fixes historical ESPN schedule abbreviations:
  NOR -> NOP
  PHO -> PHX
  SAN -> SAS
"""

import pandas as pd

_V26_ALIAS = {
    "NOR": "NOP",
    "PHO": "PHX",
    "SAN": "SAS",
}

# Patch the generic team normalizer if present.
if "normalize_team_abbr" in globals():
    _NORMALIZE_TEAM_ABBR_PRE_V26 = normalize_team_abbr

    def normalize_team_abbr(x):
        y = _NORMALIZE_TEAM_ABBR_PRE_V26(x)
        if pd.isna(y):
            s = str(x).strip().upper()
            return _V26_ALIAS.get(s, y)
        return _V26_ALIAS.get(str(y).strip().upper(), y)

# Patch ESPN team conversion as well.
if "_espn_team_to_nba_v2" in globals():
    _ESPN_TEAM_TO_NBA_PRE_V26 = _espn_team_to_nba_v2

    def _espn_team_to_nba_v2(team_obj):
        y = _ESPN_TEAM_TO_NBA_PRE_V26(team_obj)
        if y is None:
            return y
        return _V26_ALIAS.get(str(y).strip().upper(), y)

# Patch already-cached schedule outputs on read, so no redownload is required.
if "build_espn_full_schedule_v2" in globals():
    _BUILD_ESPN_FULL_SCHEDULE_PRE_V26 = build_espn_full_schedule_v2

    def build_espn_full_schedule_v2(season: str, force=False):
        out = _BUILD_ESPN_FULL_SCHEDULE_PRE_V26(season, force=force).copy()
        for c in ["home_team", "away_team"]:
            if c in out.columns:
                out[c] = out[c].astype(str).str.upper().replace(_V26_ALIAS)

        # Refresh the cache with canonical labels.
        try:
            if "RAW" in globals():
                out.to_parquet(RAW / "espn_archive_full_schedule_v22.parquet", index=False)
        except Exception as e:
            print("  v2.6 cache update warning:", e)

        return out

print("NBA RTP team-alias fix v2.6 loaded.")
print("  NOR -> NOP")
print("  PHO -> PHX")
print("  SAN -> SAS")
print("  No redownload required.")


## 6H. v2.8 corrected age merge

The historical injury-episode table contains `player_name` but not `player_key`. The prior
historical age function incorrectly required `player_key`, leaving age blank for 2021–24.

v2.8 derives the normalized key directly from `player_name`, uses the season-level NBA
`players_<year>.parquet` age field, and can also retrofit already-saved outputs without
rerunning extraction.


In [ ]:

"""
NBA RTP AGE RETROFIT v2.8
Run in the SAME Colab runtime after the final 2021-22 through 2024-25 season
outputs exist.

Purpose:
- Fix the historical age merge bug (injury_episodes has player_name, not player_key).
- Fill season_age from llimllib/NBA season-level players_<endyear>.parquet.
- Retrofit age into the saved season Parquets WITHOUT rerunning extraction.
- Regenerate the combined Parquets + Excel workbook.
- Write an audit of age coverage/unmatched names.

This uses the same season-level AGE definition already used for 2024-25.
"""

from pathlib import Path
import json
import numpy as np
import pandas as pd

AGE_RETROFIT_SEASONS = ["2021-22", "2022-23", "2023-24", "2024-25"]


# -------------------------------------------------------------------------
# 1) Correct the in-memory add_player_age() for any future rebuild.
# -------------------------------------------------------------------------
_ADD_PLAYER_AGE_PRE_V28 = add_player_age

def _v28_age_lookup(season: str):
    """
    Build normalized-name and NBA-player-ID age maps from players_<endyear>.parquet.
    Duplicate team rows are collapsed; age should be invariant within a season.
    """
    ps = get_public_player_season_stats(season).copy()

    namec = _first_col(ps, ["player_name", "PLAYER_NAME", "display_name"])
    agec = _first_col(ps, ["age", "AGE"])
    idc = _first_col(ps, ["player_id", "PLAYER_ID", "personId"])

    if not namec or not agec:
        raise RuntimeError(
            f"{season}: player season stats are missing player_name/age columns. "
            f"Columns={list(ps.columns)}"
        )

    x = ps[[namec, agec] + ([idc] if idc else [])].copy()
    x["_age_key"] = x[namec].map(norm_name)
    x["_age"] = pd.to_numeric(x[agec], errors="coerce")

    # Same player may appear on multiple team rows after a trade.
    name_map = (
        x.dropna(subset=["_age_key", "_age"])
         .groupby("_age_key", as_index=True)["_age"]
         .median()
    )

    id_map = pd.Series(dtype=float)
    if idc:
        x["_nba_id"] = pd.to_numeric(x[idc], errors="coerce").astype("Int64")
        id_map = (
            x.dropna(subset=["_nba_id", "_age"])
             .groupby("_nba_id", as_index=True)["_age"]
             .median()
        )

    return name_map, id_map


def add_player_age(episodes: pd.DataFrame, season: str) -> pd.DataFrame:
    """
    v2.8 corrected age merge.
    Works whether or not injury_episodes already contains player_key.
    """
    if episodes is None or episodes.empty:
        return episodes

    out = episodes.copy()
    try:
        name_map, id_map = _v28_age_lookup(season)
    except Exception as e:
        print(f"  v2.8 age merge skipped for {season}: {e}")
        return out

    if "season_age" not in out.columns:
        out["season_age"] = np.nan

    age = pd.to_numeric(out["season_age"], errors="coerce")

    # NBA ID is useful where IDs are truly NBA IDs (especially 2024-25).
    if "player_id" in out.columns and len(id_map):
        ids = pd.to_numeric(out["player_id"], errors="coerce").astype("Int64")
        age = age.fillna(ids.map(id_map))

    # Name matching is the reliable historical bridge because ESPN IDs != NBA IDs.
    if "player_key" in out.columns:
        key = out["player_key"].astype(str).map(norm_name)
    elif "player_name" in out.columns:
        key = out["player_name"].map(norm_name)
    else:
        key = pd.Series("", index=out.index)

    age = age.fillna(key.map(name_map))
    out["season_age"] = pd.to_numeric(age, errors="coerce")
    return out


# -------------------------------------------------------------------------
# 2) Helper to retrofit one saved table.
# -------------------------------------------------------------------------
def _v28_fill_age_table(df, season, name_map, id_map):
    out = df.copy()

    if "season_age" not in out.columns:
        out["season_age"] = np.nan

    age = pd.to_numeric(out["season_age"], errors="coerce")

    # ID fallback, when compatible.
    if "player_id" in out.columns and len(id_map):
        ids = pd.to_numeric(out["player_id"], errors="coerce").astype("Int64")
        age = age.fillna(ids.map(id_map))

    # Prefer an existing normalized player_key; otherwise derive from player_name.
    if "player_key" in out.columns:
        key = out["player_key"].astype(str).map(norm_name)
    elif "player_name" in out.columns:
        key = out["player_name"].map(norm_name)
    else:
        key = pd.Series("", index=out.index)

    age = age.fillna(key.map(name_map))
    out["season_age"] = pd.to_numeric(age, errors="coerce")
    return out


# -------------------------------------------------------------------------
# 3) Retrofit all four PASS seasons already saved on disk.
# -------------------------------------------------------------------------
def retrofit_age_existing_outputs_v28(
    seasons=AGE_RETROFIT_SEASONS,
    regenerate_combined=True,
):
    audit_rows = []
    unmatched_rows = []

    table_files = [
        "player_game_master.parquet",
        "injury_episodes.parquet",
        "all_absence_episodes.parquet",
        "rtp_episode_game_windows.parquet",
    ]

    for season in seasons:
        print("\n" + "=" * 78)
        print(f"[{season}] AGE RETROFIT")
        print("=" * 78)

        configure_season_paths(season)
        root = MULTI_ROOT / season

        name_map, id_map = _v28_age_lookup(season)
        print(f"  age source: {len(name_map):,} normalized players")

        for filename in table_files:
            path = root / filename
            if not path.exists():
                print(f"  {filename}: not found; skipped")
                continue

            df = pd.read_parquet(path)
            before = (
                pd.to_numeric(df["season_age"], errors="coerce").notna().sum()
                if "season_age" in df.columns else 0
            )

            out = _v28_fill_age_table(df, season, name_map, id_map)
            after = pd.to_numeric(out["season_age"], errors="coerce").notna().sum()

            out.to_parquet(path, index=False)

            audit_rows.append({
                "season": season,
                "table": filename,
                "rows": len(out),
                "age_nonmissing_before": int(before),
                "age_nonmissing_after": int(after),
                "age_coverage_after": float(after / len(out)) if len(out) else np.nan,
            })

            print(
                f"  {filename}: age {before:,}/{len(out):,} -> "
                f"{after:,}/{len(out):,} ({after/len(out):.1%})"
                if len(out) else f"  {filename}: empty"
            )

            if "player_name" in out.columns:
                miss = out[pd.to_numeric(out["season_age"], errors="coerce").isna()]
                if len(miss):
                    cols = [c for c in ["player_name", "player_id", "team"] if c in miss.columns]
                    u = miss[cols].drop_duplicates().copy()
                    u.insert(0, "season", season)
                    u.insert(1, "table", filename)
                    unmatched_rows.append(u)

        # Add age-coverage fields to existing QC summary without changing PASS/REVIEW.
        qpath = root / "qc_summary.csv"
        epath = root / "injury_episodes.parquet"
        if qpath.exists() and epath.exists():
            q = pd.read_csv(qpath)
            ep = pd.read_parquet(epath)
            age_ok = pd.to_numeric(ep.get("season_age"), errors="coerce").notna()

            q["injury_episode_age_coverage"] = float(age_ok.mean()) if len(ep) else np.nan

            if "primary_analysis_eligible" in ep.columns:
                elig = ep["primary_analysis_eligible"].fillna(False).astype(bool)
                q["primary_episode_age_coverage"] = (
                    float(age_ok[elig].mean()) if elig.any() else np.nan
                )

            q.to_csv(qpath, index=False)

    audit = pd.DataFrame(audit_rows)
    audit_path = MULTI_ROOT / "age_enrichment_audit.csv"
    audit.to_csv(audit_path, index=False)

    if unmatched_rows:
        unmatched = pd.concat(unmatched_rows, ignore_index=True, sort=False).drop_duplicates()
    else:
        unmatched = pd.DataFrame(columns=["season", "table", "player_name"])
    unmatched_path = MULTI_ROOT / "age_enrichment_unmatched.csv"
    unmatched.to_csv(unmatched_path, index=False)

    print("\nAge audit:", audit_path)
    print("Unmatched age names:", unmatched_path)

    # Regenerate all combined files and the master Excel from the corrected
    # season-level Parquets. No extraction/network-heavy work is repeated here.
    combined = None
    if regenerate_combined:
        print("\nRegenerating combined PASS-season datasets + Excel...")
        combined = combine_completed_seasons(list(seasons))

    # Final injury-episode coverage check.
    combined_ep_path = MULTI_ROOT / "combined_injury_episodes.parquet"
    if combined_ep_path.exists():
        ep = pd.read_parquet(combined_ep_path)
        age_ok = pd.to_numeric(ep.get("season_age"), errors="coerce").notna()
        elig = ep.get("primary_analysis_eligible", pd.Series(False, index=ep.index)).fillna(False).astype(bool)

        print("\n" + "=" * 78)
        print("FINAL AGE COVERAGE")
        print("=" * 78)
        print(f"All injury episodes:     {age_ok.sum():,}/{len(ep):,} ({age_ok.mean():.1%})")
        if elig.any():
            print(
                f"Primary eligible:        {age_ok[elig].sum():,}/{elig.sum():,} "
                f"({age_ok[elig].mean():.1%})"
            )

        if (~age_ok).any():
            cols = [c for c in ["season", "player_name", "player_id", "team"] if c in ep.columns]
            print("\nStill-unmatched injury episode names:")
            print(ep.loc[~age_ok, cols].drop_duplicates().head(50).to_string(index=False))

    return audit, unmatched, combined


print("NBA RTP age retrofit v2.8 loaded.")
print("Historical bug fixed: player_name is now normalized directly when player_key is absent.")
print("No injury reports or game logs need to be re-extracted.")


## 6I. v2.9 authoritative manual age fallback

After the reproducible season-level age merge, only two players account for the remaining
missing ages in the primary cohort. NBA.com birthdates are used as documented fallbacks:

- OG Anunoby, 2021–22: season age 24
- Jakob Poeltl, 2023–24: season age 28

The fallback is applied only when `season_age` remains missing and is documented in
`age_manual_resolution_audit.csv`.


In [ ]:

"""
NBA RTP MANUAL AGE FALLBACK v2.9
Targeted authoritative fallback for the only two players responsible for
missing age in the primary-analysis cohort after v2.8.

Authoritative birthdates verified from NBA.com:
- OG Anunoby: 1997-07-17 -> season_age 24 in 2021-22
- Jakob Poeltl: 1995-10-15 -> season_age 28 in 2023-24

Run AFTER v2.8 age retrofit in the same Colab runtime.

This:
- fills missing season_age in season-level saved Parquets,
- writes age_manual_resolution_audit.csv,
- regenerates the four-season combined Parquets + Excel workbook,
- confirms primary-eligible age coverage.
"""

from pathlib import Path
import numpy as np
import pandas as pd

V29_MANUAL_AGE = {
    ("2021-22", "og anunoby"): {
        "season_age": 24.0,
        "birthdate": "1997-07-17",
        "source": "NBA.com player profile",
        "source_url": "https://www.nba.com/player/1628384/og-anunoby/profile",
        "note": "Not matched in season-level player metadata; authoritative NBA.com fallback.",
    },
    ("2023-24", "jakob poeltl"): {
        "season_age": 28.0,
        "birthdate": "1995-10-15",
        "source": "NBA.com player profile",
        "source_url": "https://www.nba.com/player/1627751/jakob-p%C3%B6ltl/profile",
        "note": "Not matched in season-level player metadata; authoritative NBA.com fallback.",
    },
}

V29_PRIMARY_SEASONS = ["2021-22", "2022-23", "2023-24", "2024-25"]


def _v29_apply_table(df, season):
    out = df.copy()

    if "season_age" not in out.columns:
        out["season_age"] = np.nan

    if "player_key" in out.columns:
        key = out["player_key"].astype(str).map(norm_name)
    elif "player_name" in out.columns:
        key = out["player_name"].map(norm_name)
    else:
        return out, []

    changed = []
    for (s, player_key), meta in V29_MANUAL_AGE.items():
        if s != season:
            continue

        mask = key.eq(player_key) & pd.to_numeric(
            out["season_age"], errors="coerce"
        ).isna()

        n = int(mask.sum())
        if n:
            out.loc[mask, "season_age"] = float(meta["season_age"])
            changed.append({
                "season": season,
                "player_key": player_key,
                "rows_filled": n,
                **meta,
            })

    return out, changed


def apply_manual_age_fallback_v29(
    seasons=V29_PRIMARY_SEASONS,
    regenerate_combined=True,
):
    audit_rows = []

    table_files = [
        "player_game_master.parquet",
        "injury_episodes.parquet",
        "all_absence_episodes.parquet",
        "rtp_episode_game_windows.parquet",
    ]

    for season in seasons:
        configure_season_paths(season)
        root = MULTI_ROOT / season

        for filename in table_files:
            p = root / filename
            if not p.exists():
                continue

            df = pd.read_parquet(p)
            out, changes = _v29_apply_table(df, season)
            out.to_parquet(p, index=False)

            for rec in changes:
                rec = dict(rec)
                rec["table"] = filename
                audit_rows.append(rec)

    audit = pd.DataFrame(audit_rows)

    # Ensure one human-readable authoritative audit even if rerunning after values
    # are already filled.
    authoritative = []
    for (season, player_key), meta in V29_MANUAL_AGE.items():
        authoritative.append({
            "season": season,
            "player_key": player_key,
            **meta,
        })
    auth = pd.DataFrame(authoritative)

    if len(audit):
        summary = (
            audit.groupby(
                [
                    "season", "player_key", "season_age", "birthdate",
                    "source", "source_url", "note"
                ],
                as_index=False
            )["rows_filled"].sum()
        )
        final_audit = auth.merge(
            summary,
            on=[
                "season", "player_key", "season_age", "birthdate",
                "source", "source_url", "note"
            ],
            how="left"
        )
    else:
        final_audit = auth.copy()
        final_audit["rows_filled"] = 0

    audit_path = MULTI_ROOT / "age_manual_resolution_audit.csv"
    final_audit.to_csv(audit_path, index=False)
    print("Manual age audit:", audit_path)

    if regenerate_combined:
        print("\nRegenerating combined PASS-season datasets + Excel...")
        combine_completed_seasons(list(seasons))

    ep_path = MULTI_ROOT / "combined_injury_episodes.parquet"
    if ep_path.exists():
        ep = pd.read_parquet(ep_path)
        age_ok = pd.to_numeric(ep["season_age"], errors="coerce").notna()
        elig = ep["primary_analysis_eligible"].fillna(False).astype(bool)

        print("\n" + "=" * 78)
        print("FINAL PRIMARY AGE COVERAGE AFTER v2.9")
        print("=" * 78)
        print(
            f"All injury episodes: {age_ok.sum():,}/{len(ep):,} "
            f"({age_ok.mean():.1%})"
        )
        print(
            f"Primary eligible:    {age_ok[elig].sum():,}/{elig.sum():,} "
            f"({age_ok[elig].mean():.1%})"
        )

        still = ep.loc[
            elig & ~age_ok,
            [c for c in [
                "season", "player_name", "player_id", "team",
                "injury_body_part", "return_game_date"
            ] if c in ep.columns]
        ]
        if len(still):
            print("\nStill missing among primary eligible:")
            print(still.to_string(index=False))
        else:
            print("\nPrimary eligible age coverage is complete.")

    return final_audit


print("NBA RTP manual age fallback v2.9 loaded.")
print("  OG Anunoby 2021-22 -> season_age 24")
print("  Jakob Poeltl 2023-24 -> season_age 28")
print("  Sources documented in age_manual_resolution_audit.csv")


## 7. Run validation or full primary cohort

For a fresh validation, leave `RUN_MODE = "validation"`.
For a complete rebuild, set `RUN_MODE = "full"` in the configuration cell.

`SKIP_COMPLETED_PASS_SEASONS = True` prevents already-PASS seasons from being needlessly
re-extracted when the same project directory is reused.


In [ ]:
batch_summary = run_multiseason(SEASONS)


## 8. Combine all completed seasons

This cell concatenates every successfully built season, regardless of whether the batch was run in one sitting. Large player-game data stay in Parquet. A compact combined Excel workbook is created for episode-level analysis and QC rather than duplicating the entire 100+ column player-game archive into Excel.


In [ ]:
def combine_completed_seasons(seasons=None):
    seasons = list(seasons or FULL_SEASONS)
    pgs, eps, abses, wins, qc_summaries, qc_gates = [], [], [], [], [], []
    included = []
    for season in seasons:
        root = MULTI_ROOT / season
        manifest_path = root / "season_manifest.json"
        if not manifest_path.exists():
            continue
        try:
            mf = json.loads(manifest_path.read_text())
        except Exception:
            continue
        allowed = {"PASS", "REVIEW"} if INCLUDE_REVIEW_IN_COMBINED else {"PASS"}
        if mf.get("qc_status") not in allowed:
            continue
        files = {
            "pg": root / "player_game_master.parquet",
            "ep": root / "injury_episodes.parquet",
            "ab": root / "all_absence_episodes.parquet",
            "wi": root / "rtp_episode_game_windows.parquet",
        }
        if not all(p.exists() for p in files.values()):
            continue
        pg = pd.read_parquet(files["pg"]); pg["season"] = season
        ep = pd.read_parquet(files["ep"]); ep["season"] = season
        ab = pd.read_parquet(files["ab"]); ab["season"] = season
        wi = pd.read_parquet(files["wi"]); wi["season"] = season
        pgs.append(pg); eps.append(ep); abses.append(ab); wins.append(wi); included.append(season)
        q = root / "qc_summary.csv"
        g = root / "qc_gates.csv"
        if q.exists(): qc_summaries.append(pd.read_csv(q))
        if g.exists():
            gg = pd.read_csv(g); gg.insert(0, "season", season); qc_gates.append(gg)

    if not included:
        raise RuntimeError("No completed season outputs found to combine.")

    cpg = pd.concat(pgs, ignore_index=True, sort=False)
    cep = pd.concat(eps, ignore_index=True, sort=False)
    cab = pd.concat(abses, ignore_index=True, sort=False)
    cwi = pd.concat(wins, ignore_index=True, sort=False)
    cqc = pd.concat(qc_summaries, ignore_index=True, sort=False) if qc_summaries else pd.DataFrame()
    cgates = pd.concat(qc_gates, ignore_index=True, sort=False) if qc_gates else pd.DataFrame()

    cpg.to_parquet(MULTI_ROOT / "combined_player_game_master.parquet", index=False)
    cep.to_parquet(MULTI_ROOT / "combined_injury_episodes.parquet", index=False)
    cab.to_parquet(MULTI_ROOT / "combined_all_absence_episodes.parquet", index=False)
    cwi.to_parquet(MULTI_ROOT / "combined_rtp_episode_game_windows.parquet", index=False)
    cep.to_csv(MULTI_ROOT / "combined_injury_episodes.csv", index=False)
    cwi.to_csv(MULTI_ROOT / "combined_rtp_episode_game_windows.csv", index=False)
    if len(cqc): cqc.to_csv(MULTI_ROOT / "combined_qc_summary.csv", index=False)
    if len(cgates): cgates.to_csv(MULTI_ROOT / "combined_qc_gates.csv", index=False)

    if WRITE_COMBINED_EXCEL:
        out_xlsx = MULTI_ROOT / "NBA_Injury_RTP_MultiSeason_QC_and_Episodes.xlsx"
        engine = "xlsxwriter"
        with pd.ExcelWriter(out_xlsx, engine=engine) as writer:
            if len(cqc): cqc.to_excel(writer, sheet_name="season_qc_summary", index=False)
            if len(cgates): cgates.to_excel(writer, sheet_name="season_qc_gates", index=False)
            cep.to_excel(writer, sheet_name="injury_episodes", index=False)
            cab.to_excel(writer, sheet_name="all_absence_episodes", index=False)
            # Windows can be much larger but still fit comfortably for this four-season primary scope.
            cwi.to_excel(writer, sheet_name="rtp_game_windows", index=False)
        print("Combined Excel:", out_xlsx.resolve())

    print("Included seasons:", included)
    print("Combined player-game rows:", len(cpg))
    print("Combined injury episodes:", len(cep))
    print("Combined primary-analysis-eligible episodes:", int(cep.get("primary_analysis_eligible", pd.Series(dtype=int)).fillna(0).sum()))
    return cpg, cep, cab, cwi, cqc, cgates

# Run this after the desired seasons have completed:
# combined_pg, combined_ep, combined_abs, combined_windows, combined_qc, combined_gates = combine_completed_seasons()


### Existing-output age retrofit (use this for the dataset you already built)

If the four season folders already exist and passed QC, run the v2.8 retrofit and then v2.9 fallback **instead of
re-extracting them**. It updates age in the saved Parquets and regenerates the combined
Parquets + Excel workbook.

For a completely fresh run using this cleaned notebook, v2.8 is already active before
`run_multiseason`, so this retrofit is optional.


In [ ]:
# For already-built PASS seasons:
# age_audit, age_unmatched, _ = retrofit_age_existing_outputs_v28(
#     ["2021-22", "2022-23", "2023-24", "2024-25"]
# )
# manual_age_audit = apply_manual_age_fallback_v29(
#     ["2021-22", "2022-23", "2023-24", "2024-25"]
# )


### Combine the four PASS seasons

Run after the desired season outputs exist (or after the age retrofit above).
Only `PASS` seasons are included by default.


In [ ]:
# combined_pg, combined_ep, combined_abs, combined_windows, combined_qc, combined_gates = (
#     combine_completed_seasons(["2021-22", "2022-23", "2023-24", "2024-25"])
# )


## 9. Final archival / AI export helper

After age is fixed and the combined files are regenerated, this helper creates:

1. **Personal full archive ZIP** — entire project tree, including raw/cache material.
2. **AI analysis bundle ZIP** — combined analysis-ready Parquets, Excel, QC, age audit,
   schema inventory, README, and this cleaned notebook if it is present in `/content`.

The export step performs no extraction.


In [ ]:

"""
NBA RTP FINAL EXPORT HELPER v1.0

Run AFTER:
1) the four primary seasons are PASS,
2) NBA_RTP_AGE_RETROFIT_v2_8 has been loaded, and
3) retrofit_age_existing_outputs_v28(...) has completed successfully.

Creates:
A) NBA_RTP_PERSONAL_FULL_ARCHIVE.zip
   - entire nba_injury_rtp_multiseason project tree (including raw/cache files)
   - cleaned final notebook if available
   - archive README + file manifest

B) NBA_RTP_AI_ANALYSIS_BUNDLE.zip
   - analysis-ready combined Parquets
   - master Excel workbook
   - QC files
   - age audit files
   - cleaned final notebook if available
   - AI README + automatically generated schema inventory

No extraction is performed.
"""

from pathlib import Path
import json
import os
import shutil
import zipfile
import pandas as pd


PRIMARY_SEASONS = ["2021-22", "2022-23", "2023-24", "2024-25"]

# Put the cleaned notebook downloaded from ChatGPT in /content with this name
# if you want it embedded inside both ZIPs.
CLEAN_NOTEBOOK_NAME = "NBA_Injury_RTP_MultiSeason_FINAL_2021_25.ipynb"

PERSONAL_ZIP_NAME = "NBA_RTP_PERSONAL_FULL_ARCHIVE.zip"
AI_ZIP_NAME = "NBA_RTP_AI_ANALYSIS_BUNDLE.zip"


def _find_clean_notebook():
    candidates = [
        Path(CLEAN_NOTEBOOK_NAME),
        Path("/content") / CLEAN_NOTEBOOK_NAME,
        Path("/mnt/data") / CLEAN_NOTEBOOK_NAME,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def _zip_tree(zip_path, root_dir, arc_prefix=None):
    root_dir = Path(root_dir)
    prefix = arc_prefix or root_dir.name
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as z:
        for p in sorted(root_dir.rglob("*")):
            if p.is_file():
                arc = Path(prefix) / p.relative_to(root_dir)
                z.write(p, arcname=str(arc))


def _write_manifest(root, out_path):
    rows = []
    root = Path(root)
    for p in sorted(root.rglob("*")):
        if p.is_file():
            rows.append({
                "relative_path": str(p.relative_to(root)),
                "bytes": p.stat().st_size,
            })
    pd.DataFrame(rows).to_csv(out_path, index=False)
    return len(rows)


def _make_schema_inventory(ai_dir):
    records = []

    for p in sorted(ai_dir.iterdir()):
        try:
            if p.suffix.lower() == ".parquet":
                df = pd.read_parquet(p)
            elif p.suffix.lower() == ".csv" and p.name != "DATA_SCHEMA.csv":
                df = pd.read_csv(p, low_memory=False)
            else:
                continue
        except Exception:
            continue

        for c in df.columns:
            s = df[c]
            records.append({
                "file": p.name,
                "column": c,
                "dtype": str(s.dtype),
                "nonmissing_n": int(s.notna().sum()),
                "rows_n": int(len(df)),
                "nonmissing_fraction": float(s.notna().mean()) if len(df) else None,
            })

    out = pd.DataFrame(records)
    out.to_csv(ai_dir / "DATA_SCHEMA.csv", index=False)
    return out


def _ai_readme():
    return """# NBA Injury / Return-to-Play AI Analysis Bundle

## Primary cohort
NBA seasons 2021-22, 2022-23, 2023-24, and 2024-25.

Only seasons that passed the pipeline QC gates are intended for the primary
analysis. Review `combined_qc_summary.csv` and `combined_qc_gates.csv` before
modeling.

## Main files

### combined_injury_episodes.parquet
Primary episode-level modeling table. One row per injury/RTP episode.
Use `primary_analysis_eligible == True` for the main cohort unless the analysis
plan specifies otherwise.

Important exposure families include:
- return_minutes_ratio
- post3_mean_minutes_ratio
- post5_mean_minutes_ratio
- ramp_slope_minutes_per_game
- max_positive_minute_step_first5
- ramp_deficit_auc_first5
- games_to_90pct_baseline
- return_team_is_b2b
- return_team_calendar_rest_days
- return_travel_km
- b2b_count_first5_team_games_post
- travel_km_first5_team_games_post
- season_age

Important outcome families include:
- subsequent_any_injury_absence_within_14d / 30d / 60d
- subsequent_same_region_absence_within_*
- subsequent_same_body_part_absence_within_*
- subsequent_same_body_laterality_dx_absence_within_*
- early_same_body_continuation_proxy_30d_lt3_played
- separated_same_body_event_proxy_30d_ge3_played
- separated_same_body_laterality_dx_proxy_60d_ge3_played

Performance fields include pre/post true shooting, points, plus/minus, and some
advanced measures. Check DATA_SCHEMA.csv for coverage before using a field.

### combined_rtp_episode_game_windows.parquet
Game-level rows around each RTP episode. Use this to reconstruct alternative
pre/post windows, minute-ramp definitions, performance trajectories, schedule
burden, or time-varying exposures.

### combined_player_game_master.parquet
Full player-game master. This is the most flexible source for constructing new
game-level variables without re-extracting raw NBA/ESPN data.

### combined_all_absence_episodes.parquet
All classified absence episodes, useful for alternative recurrence/continuation
definitions and sensitivity analyses.

### NBA_Injury_RTP_MultiSeason_QC_and_Episodes.xlsx
Human-readable workbook containing QC, injury episodes, absence episodes, and
RTP windows.

### combined_qc_summary.csv / combined_qc_gates.csv
Season-level quality-control evidence.

### age_enrichment_audit.csv / age_enrichment_unmatched.csv
Audit trail for the season_age retrofit.

### NBA_Injury_RTP_MultiSeason_FINAL_2021_25.ipynb
Cleaned reproducible extraction/processing notebook, if included.

## Statistical cautions
- Injury episodes are clustered within players; do not assume episode-level
  independence. Use player-clustered SEs, mixed effects, or an appropriate
  recurrent-event framework.
- Confounding by indication is substantial: injury severity/type, games missed,
  surgery/recovery wording, age, baseline role/minutes, team, season, and
  schedule may affect both RTP ramp and subsequent injury.
- Do not equate every subsequent injury-report absence with a true reinjury.
  The separated-event proxies are intended to help distinguish continuation
  from a later event.
- Handle end-of-season censoring explicitly for 14/30/60-day outcomes.
- Exact rest-hour fields are incomplete historically; calendar-rest/B2B fields
  are more consistently available.
"""


def _personal_readme():
    return """# NBA RTP Personal Full Archive

This archive is the preservation copy intended to prevent future re-extraction.

It contains the ENTIRE `nba_injury_rtp_multiseason` project directory available
in the Colab runtime at export time, including season-level processed outputs,
raw/cache material, injury-report downloads, diagnostics, combined datasets, and
QC artifacts.

Keep this ZIP in at least two durable locations.

For routine statistical analysis, use the smaller AI analysis bundle rather
than unpacking the full raw archive.

Important:
- Primary validated cohort: 2021-22 through 2024-25.
- Experimental/failed extension season folders may also be present if they were
  created in the same runtime. Their presence does NOT mean they passed QC.
- `season_manifest.json` and QC files determine whether a season is validated.
"""


def create_final_archives(
    project_root=None,
    download=True,
):
    project_root = Path(project_root or MULTI_ROOT)
    if not project_root.exists():
        raise FileNotFoundError(f"Project root not found: {project_root}")

    # Verify the core analysis files exist before archiving.
    required = [
        "combined_player_game_master.parquet",
        "combined_injury_episodes.parquet",
        "combined_all_absence_episodes.parquet",
        "combined_rtp_episode_game_windows.parquet",
        "combined_qc_summary.csv",
        "combined_qc_gates.csv",
        "NBA_Injury_RTP_MultiSeason_QC_and_Episodes.xlsx",
    ]
    missing = [f for f in required if not (project_root / f).exists()]
    if missing:
        raise RuntimeError(
            "Do not archive yet. Missing combined files:\n  - " + "\n  - ".join(missing)
        )

    clean_nb = _find_clean_notebook()

    # ------------------------------------------------------------------
    # PERSONAL archive staging
    # ------------------------------------------------------------------
    personal_stage = Path("NBA_RTP_PERSONAL_FULL_ARCHIVE")
    if personal_stage.exists():
        shutil.rmtree(personal_stage)
    personal_stage.mkdir()

    # Keep the project tree intact.
    shutil.copytree(project_root, personal_stage / project_root.name)

    (personal_stage / "README_PERSONAL_ARCHIVE.md").write_text(
        _personal_readme(), encoding="utf-8"
    )

    if clean_nb:
        shutil.copy2(clean_nb, personal_stage / clean_nb.name)

    n_manifest = _write_manifest(
        personal_stage,
        personal_stage / "FILE_MANIFEST.csv"
    )

    personal_zip = Path(PERSONAL_ZIP_NAME)
    if personal_zip.exists():
        personal_zip.unlink()
    _zip_tree(personal_zip, personal_stage, arc_prefix=personal_stage.name)

    # ------------------------------------------------------------------
    # AI bundle staging
    # ------------------------------------------------------------------
    ai_stage = Path("NBA_RTP_AI_ANALYSIS_BUNDLE")
    if ai_stage.exists():
        shutil.rmtree(ai_stage)
    ai_stage.mkdir()

    ai_files = required + [
        "age_enrichment_audit.csv",
        "age_enrichment_unmatched.csv",
    ]
    copied = []
    for name in ai_files:
        src = project_root / name
        if src.exists():
            shutil.copy2(src, ai_stage / name)
            copied.append(name)

    if clean_nb:
        shutil.copy2(clean_nb, ai_stage / clean_nb.name)
        copied.append(clean_nb.name)

    (ai_stage / "README_FOR_AI.md").write_text(
        _ai_readme(), encoding="utf-8"
    )
    _make_schema_inventory(ai_stage)

    ai_zip = Path(AI_ZIP_NAME)
    if ai_zip.exists():
        ai_zip.unlink()
    _zip_tree(ai_zip, ai_stage, arc_prefix=ai_stage.name)

    print("\n" + "=" * 78)
    print("FINAL ARCHIVES CREATED")
    print("=" * 78)
    print(f"Personal archive: {personal_zip.resolve()}")
    print(f"  files indexed:  {n_manifest:,}")
    print(f"  size:           {personal_zip.stat().st_size / (1024**2):,.1f} MB")
    print()
    print(f"AI bundle:        {ai_zip.resolve()}")
    print(f"  size:           {ai_zip.stat().st_size / (1024**2):,.1f} MB")
    print(f"  cleaned notebook included: {'YES' if clean_nb else 'NO'}")

    if clean_nb is None:
        print(
            "\nNOTE: The cleaned notebook was not found in /content. "
            "Upload NBA_Injury_RTP_MultiSeason_FINAL_2021_25.ipynb to Colab "
            "and rerun this function if you want it embedded in both ZIPs."
        )

    if download:
        try:
            from google.colab import files
            print("\nStarting browser downloads...")
            files.download(str(personal_zip))
            files.download(str(ai_zip))
        except Exception as e:
            print("Automatic Colab download skipped:", e)

    return personal_zip, ai_zip


print("NBA RTP final export helper v1.0 loaded.")
print("Call create_final_archives() only AFTER the age retrofit has completed.")


In [ ]:
# After placing this cleaned notebook in /content, run:
# personal_zip, ai_zip = create_final_archives()


## 9. QC interpretation

A season receives `PASS` only if the automated hard gates are satisfied: all observed games are dated, all 30 teams are represented, no team has zero injury absences, official report game mapping is at least 97%, unmatched collapsed report keys are at most 2%, orthopedic-looking illness misclassification is essentially absent, and the season yields at least 50 primary RTP episodes.

`REVIEW` does **not** delete the season. It preserves all outputs and flags that season for inspection. `FAILED` similarly preserves the error/traceback and lets the batch continue to the next year.

Special environments such as 2019–20 (COVID shutdown/bubble) and 2020–21 (72-game shortened season) are intentionally retained as season labels rather than forced to match an 82-game expectation.


## 10. Recommended final workflow

1. For the already-built dataset, load v2.8 and run `retrofit_age_existing_outputs_v28(...)`.
2. Confirm primary-eligible age coverage is near-complete; inspect `age_enrichment_unmatched.csv`
   if any names remain.
3. Regenerate/confirm the four-season combined files.
4. Preserve **PASS seasons only** for the primary analysis.
5. Keep the full player-game Parquet even if the Excel workbook is your main human-readable file.
6. Run `create_final_archives()` to create both the personal preservation ZIP and AI analysis ZIP.
7. Treat experimental 2019–21 / 2025–26 outputs separately unless they independently pass the
   same QC framework.
